# Reranker

## Описание
Расширение pipeline новым candidate pool `merged_with_dl_all_repeat_history` (512 шардов, исправлена user-universe утечка).
Добавлены группы repeat-history признаков A–D; candidate recall@100: **0.100 → 0.183** (+83%).

## Результаты (val_eval, full run)
| Метрика | Значение |
|---------|----------|
| NDCG@100 | **0.087751** |
| NDCG@50 | 0.083814 |
| recall@100 | 0.145960 |
| hit_rate@100 | 0.310854 |
| user_coverage | 1.0 |

## Ablation (val_fit NDCG@100, фиксированные параметры кроме D)
| Set | Признаков | val_fit NDCG@100 | Δ vs A |
|-----|-----------|-----------------|--------|
| A | 199 (без repeat) | 0.191055 | — |
| B | 227 (+raw repeat) | 0.207544 | +8.6% |
| C | 233 (+rank-gap) | 0.207724 | +8.7% |
| D | 240 (полный, Optuna 6 проб) | **0.208407** | +9.1% |

## Leakage Checks
- `repeat_*` признаки выведены только из context window ✓
- Derived features — преобразования pool columns без target данных ✓
- Ground truth из `agg/orders/user_item` target window ✓
- Optuna настраивается только на `ranker_train` ✓
- val используется только для финального сравнения с baseline ✓


In [31]:
from __future__ import annotations

import gc
import json
import os
import re
import shutil
import time
import warnings
from collections import defaultdict
from dataclasses import asdict, dataclass
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from typing import Iterable, Sequence

# Ограничиваем внутренние BLAS-пулы до импорта numpy/lightgbm.
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import joblib
import lightgbm as lgbm
import numpy as np
import optuna
import pandas as pd
import polars as pl
from optuna.samplers import TPESampler
from tqdm.auto import tqdm, trange

try:
    from catboost import CatBoostRanker, Pool
    HAS_CATBOOST = True
except ImportError:
    CatBoostRanker = None
    Pool = None
    HAS_CATBOOST = False

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 200)
pl.Config.set_tbl_cols(200)
pl.Config.set_tbl_rows(30)

polars.config.Config

In [32]:
def find_project_root(start: Path = Path.cwd()) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "raw").exists() or (p / "data" / "datamart").exists() or (p / "data").exists():
            return p
    raise FileNotFoundError("Не найден project root с папкой data")


@dataclass(frozen=True)
class FinalRankerConfig:
    # Даты должны совпадать с datamart/candidate/DL ноутбуками.
    train_context_start: date = date(2025, 4, 18)
    train_context_end: date = date(2025, 6, 3)
    train_target_start: date = date(2025, 6, 4)
    train_target_end: date = date(2025, 6, 17)

    val_context_start: date = date(2025, 4, 18)
    val_context_end: date = date(2025, 6, 17)
    val_target_start: date = date(2025, 6, 18)
    val_target_end: date = date(2025, 7, 1)

    full_context_start: date = date(2025, 5, 2)
    full_context_end: date = date(2025, 7, 1)
    test_target_start: date = date(2025, 7, 2)
    test_target_end: date = date(2025, 7, 15)

    # В этом проекте test не blind split: покупки test target window доступны в agg/orders/user_item.
    # Поэтому test собирается с label и используется как финальный holdout, но не для подбора модели.
    test_has_labels: bool = True

    # Основной вход финального ранкера.
    candidate_input_name: str = "merged_with_dl_all_repeat_history"
    fallback_candidate_input_names: tuple[str, ...] = (
        "merged_with_dl_all",
        "merged_with_bert4rec_small",
        "merged_with_sasrec_v3",
        "merged",
    )

    # Имена выходов. run_name не должен совпадать со старым 5_reranker, чтобы не перетереть кеши.
    run_name: str = "final_reranker_v12_repeat"
    feature_store_version: str = "reranker_v1"

    n_shards: int = 512
    final_k: int = 100
    metric_ks: tuple[int, ...] = (20, 50, 100, 200)
    optimize_metric: str = "ndcg@100"

    # Feature windows are strictly before the target window.
    feature_windows: tuple[int, ...] = (7, 14, 30, 60)
    feature_levels: tuple[str, ...] = ("user", "item", "category", "user_category", "user_item")
    candidate_scoped_feature_levels: tuple[str, ...] = ("user_item",)
    candidate_scoped_shard_batch_size: int = 8

    # Build-time fit dataset downsampling. Existing train_fit/val_fit can be reused; the memory-safe reader below
    # additionally samples rows before converting to pandas.
    always_keep_top_n_candidates: int = 50
    max_random_negatives_per_user: int = 120
    filter_fit_groups_without_positive: bool = True

    # Memory-safe model matrix controls.
    # Previous failure: 19.3M rows * 566 float32 features ~= 43.7GB just for one numeric block.
    model_max_numeric_features: int | None = 240
    model_keep_feature_windows: tuple[int, ...] = (7, 14, 30, 60)
    model_train_max_rows: int | None = 6_000_000
    model_val_max_rows: int | None = 2_000_000
    model_fit_hash_mod: int = 1_000_003
    model_force_all_positives: bool = True
    refit_production_on_train_val: bool = True  # legacy flag; production_train_mode is the active control.

    # Debug switches: set to an integer for a quick local run.
    sample_train_users: int | None = None
    sample_val_users: int | None = None
    sample_test_users: int | None = None

    # Model selection. CatBoost fixed configs in v9 degraded quality; v10 returns to stable LGBM branch.
    run_lightgbm: bool = True
    run_catboost: bool = False
    optuna_trials: int = 6
    # Optuna still uses val_fit for cheap pre-selection; these top trials are then compared on full val_eval.
    lgbm_valeval_top_trials: int = 2
    lgbm_min_trees: int = 250
    catboost_optuna_trials: int = 12
    optuna_timeout_sec: int | None = None
    early_stopping_rounds: int = 100
    random_state: int = 42
    catboost_use_id_features: bool = False

    # Rebuild controls.
    rebuild_features: bool = False
    rebuild_datasets: bool = False  # datasets already built
    smoke_shards: tuple[int, ...] | None = None
    rebuild_models: bool = False
    enable_test: bool = True  # v12: repeat_history/test not built
    positive_weight: float = 8.0  # sample_weight for positives in fit

    # Submission/test output controls.
    fill_missing_test_users_with_top_popular: bool = True
    fallback_pool_size: int = 5000

    # Final scoring-policy selection.
    # Policy selection is done on val_eval using the selection model.
    # Test remains a final labeled holdout and is not used for choosing alpha/policy.
    enable_policy_search: bool = True
    select_final_policy_on_test: bool = False
    policy_anchor_grid: tuple[float, ...] = (0.0, 1.0, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 7.5, 10.0)
    policy_search_k: int = 200
    # Robust policy selection: if NDCG ties are tiny, prefer higher recall / deeper quality.
    policy_selection_ndcg_epsilon: float = 1e-4
    policy_selection_tie_breakers: tuple[str, ...] = ("recall@100", "ndcg@200", "recall@200", "hit_rate@100", "map@100")
    # Hand-picked raw-score mixed policies. Format: (old_alpha, extra_alpha, dl_alpha).
    # extra = BM25/CLIP RR anchor; dl is normally 0 or small negative because DL acts mostly as recall booster.
    policy_raw_mixed_specs: tuple[tuple[float, float, float], ...] = (
        # Dense grid around the best v7 area: old 4.0, extra 0.5.
        (3.0, 0.25, 0.00),
        (3.5, 0.25, 0.00),
        (4.0, 0.25, 0.00),
        (4.5, 0.25, 0.00),
        (5.0, 0.25, 0.00),
        (5.5, 0.25, 0.00),
        (6.0, 0.25, 0.00),

        (3.0, 0.50, 0.00),
        (3.5, 0.50, 0.00),
        (4.0, 0.50, 0.00),
        (4.5, 0.50, 0.00),
        (5.0, 0.50, 0.00),
        (5.5, 0.50, 0.00),
        (6.0, 0.50, 0.00),

        (3.5, 0.75, 0.00),
        (4.0, 0.75, 0.00),
        (4.5, 0.75, 0.00),
        (5.0, 0.75, 0.00),
        (5.5, 0.75, 0.00),
        (6.0, 0.75, 0.00),

        (4.0, 1.00, 0.00),
        (4.5, 1.00, 0.00),
        (5.0, 1.00, 0.00),
        (5.5, 1.00, 0.00),

        # Small DL penalty variants; previous runs did not prove positive DL-anchor value.
        (4.0, 0.50, -0.05),
        (4.5, 0.50, -0.05),
        (5.0, 0.50, -0.05),
        (5.5, 0.50, -0.05),
        (4.0, 0.75, -0.05),
        (4.5, 0.75, -0.05),
        (5.0, 0.75, -0.05),
    )
    # Extra candidate-rank correction. Keep weights small, otherwise we regress to raw candidate order.
    # Format: (old_alpha, extra_alpha, dl_alpha, candidate_alpha).
    policy_candidate_mixed_specs: tuple[tuple[float, float, float, float], ...] = (
        (3.5, 0.50, 0.00, 0.10),
        (4.0, 0.50, 0.00, 0.10),
        (4.5, 0.50, 0.00, 0.10),
        (5.0, 0.50, 0.00, 0.10),
        (5.5, 0.50, 0.00, 0.10),
        (4.0, 0.75, 0.00, 0.10),
        (4.5, 0.75, 0.00, 0.10),
        (5.0, 0.75, 0.00, 0.10),

        (3.5, 0.50, 0.00, 0.25),
        (4.0, 0.50, 0.00, 0.25),
        (4.5, 0.50, 0.00, 0.25),
        (5.0, 0.50, 0.00, 0.25),
        (5.5, 0.50, 0.00, 0.25),
        (4.0, 0.75, 0.00, 0.25),
        (4.5, 0.75, 0.00, 0.25),
        (5.0, 0.75, 0.00, 0.25),

        (4.0, 0.50, 0.00, 0.50),
        (4.5, 0.50, 0.00, 0.50),
        (5.0, 0.50, 0.00, 0.50),
        (4.0, 0.75, 0.00, 0.50),
        (4.5, 0.75, 0.00, 0.50),
        (5.0, 0.75, 0.00, 0.50),
    )
    # v10: optional old champion anchor. If old 5_reranker outputs exist, use their rank as
    # a strong safety prior. This should keep strong old CatBoost ordering while allowing the
    # new merged_with_dl_all pool to insert additional high-confidence candidates.
    old_champion_anchor_enable: bool = True
    old_champion_val_scored_paths: tuple[str, ...] = (
        "data/interim/rerank_val_scored.parquet",
        "../data/interim/rerank_val_scored.parquet",
    )
    old_champion_test_scored_paths: tuple[str, ...] = (
        "data/interim/rerank_test_scored.parquet",
        "../data/interim/rerank_test_scored.parquet",
    )
    # Format: (old_alpha, extra_alpha, dl_alpha, candidate_alpha, champion_alpha).
    # Champion alpha is intentionally large: old champion rank_rr is only ~1/rank and should
    # dominate only where the old CatBoost is highly confident.
    policy_champion_mixed_specs: tuple[tuple[float, float, float, float, float], ...] = (
        (4.0, 0.75, 0.00, 0.50, 2.0),
        (4.5, 0.75, 0.00, 0.50, 2.0),
        (5.0, 0.75, 0.00, 0.50, 2.0),
        (4.0, 0.75, 0.00, 0.50, 5.0),
        (4.5, 0.75, 0.00, 0.50, 5.0),
        (5.0, 0.75, 0.00, 0.50, 5.0),
        (4.0, 0.75, 0.00, 0.50, 10.0),
        (4.5, 0.75, 0.00, 0.50, 10.0),
        (5.0, 0.75, 0.00, 0.50, 10.0),
        (4.0, 0.75, 0.00, 0.50, 20.0),
        (5.0, 0.75, 0.00, 0.50, 20.0),
        (4.0, 1.00, 0.00, 0.50, 5.0),
        (4.5, 1.00, 0.00, 0.50, 5.0),
        (5.0, 1.00, 0.00, 0.50, 5.0),
        (4.0, 1.00, 0.00, 0.50, 10.0),
        (5.0, 1.00, 0.00, 0.50, 10.0),
    )
    cleanup_nonselected_policy_outputs: bool = True
    old_anchor_rank_cols: tuple[str, ...] = ("als_rank", "top_popular_rank")
    extra_anchor_rank_cols: tuple[str, ...] = ("i2i_bm25_rank", "clip_hnsw_rank", "repeat_history_rank")
    candidate_anchor_rank_cols: tuple[str, ...] = ("candidate_rank",)
    dl_anchor_rank_cols: tuple[str, ...] = ("dl_dssm_final_rank", "dl_bert4rec_small_rank", "dl_sasrec_v3_rank")

    # NDCG-boost policy search.
    # raw_plus_rr is the only enabled family: raw model score + alpha * RR(old-source anchor).
    # rr_blend was tested and consistently degraded NDCG, so it is disabled.
    policy_enable_rr_blend: bool = False
    policy_model_rr_weight: float = 1.0
    policy_search_record_roles: tuple[str, ...] = ("selection",)
    policy_search_top_models: int = 3

    production_train_mode: str = "val_fit"  # "selection", "val_fit", "train_val"
    model_train_user_sample: int | None = None
    model_val_user_sample: int | None = 120_000

RCFG = FinalRankerConfig()

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
DATAMART_DIR = DATA_DIR / "datamart"
AGG_DIR = DATAMART_DIR / "agg"
CANDIDATES_DIR = DATAMART_DIR / "candidates"
RERANKED_ROOT = CANDIDATES_DIR / "reranked"
PROCESSED_ROOT = DATA_DIR / "processed" / "reranker"
FEATURES_DIR = DATAMART_DIR / "features" / RCFG.feature_store_version
MODELS_ROOT = PROJECT_ROOT / "models" / "reranker"
REPORTS_ROOT = PROJECT_ROOT / "artifacts" / "reports"

for path in [CANDIDATES_DIR, RERANKED_ROOT, PROCESSED_ROOT, FEATURES_DIR, MODELS_ROOT, REPORTS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print(json.dumps({k: str(v) if isinstance(v, date) else v for k, v in asdict(RCFG).items()}, ensure_ascii=False, indent=2))


PROJECT_ROOT: c:\Users\denis\Documents\Repo\ozon
{
  "train_context_start": "2025-04-18",
  "train_context_end": "2025-06-03",
  "train_target_start": "2025-06-04",
  "train_target_end": "2025-06-17",
  "val_context_start": "2025-04-18",
  "val_context_end": "2025-06-17",
  "val_target_start": "2025-06-18",
  "val_target_end": "2025-07-01",
  "full_context_start": "2025-05-02",
  "full_context_end": "2025-07-01",
  "test_target_start": "2025-07-02",
  "test_target_end": "2025-07-15",
  "test_has_labels": true,
  "candidate_input_name": "merged_with_dl_all_repeat_history",
  "fallback_candidate_input_names": [
    "merged_with_dl_all",
    "merged_with_bert4rec_small",
    "merged_with_sasrec_v3",
    "merged"
  ],
  "run_name": "final_reranker_v12_repeat",
  "feature_store_version": "reranker_v1",
  "n_shards": 512,
  "final_k": 100,
  "metric_ks": [
    20,
    50,
    100,
    200
  ],
  "optimize_metric": "ndcg@100",
  "feature_windows": [
    7,
    14,
    30,
    60
  ],
  "featu

In [33]:
EVENT_METRIC_COLS = (
    "cnt_events",
    "cnt_page_view",
    "cnt_view_description",
    "cnt_review_view",
    "cnt_to_cart",
    "cnt_favorite",
    "cnt_remove",
    "cnt_unfavorite",
)

ORDER_METRIC_COLS = (
    "cnt_orders",
    "cnt_delivered_orders",
    "cnt_canceled_orders",
    "cnt_proccesed_orders",
)

LEVEL_KEYS = {
    "user": ["user_id"],
    "item": ["item_id"],
    "user_item": ["user_id", "item_id"],
    "user_category": ["user_id", "catalogid"],
    "category": ["catalogid"],
}

_NUMERIC_DTYPES = {
    pl.Int8, pl.Int16, pl.Int32, pl.Int64,
    pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
    pl.Float32, pl.Float64,
    pl.Boolean,
}


def sanitize_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_") or "default"


def path_has_parquet(path: Path) -> bool:
    return path.exists() and any(path.rglob("*.parquet"))


def scan_parquet(path: Path | str, *, hive_partitioning: bool = True) -> pl.LazyFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.is_dir():
        if not path_has_parquet(path):
            raise FileNotFoundError(f"No parquet files found in {path}")
        return pl.scan_parquet(str(path / "**" / "*.parquet"), hive_partitioning=hive_partitioning)
    return pl.scan_parquet(str(path), hive_partitioning=hive_partitioning)


def collect_streaming(lf: pl.LazyFrame) -> pl.DataFrame:
    try:
        return lf.collect(engine="streaming")
    except TypeError:
        return lf.collect(streaming=True)


def lazy_columns(lf: pl.LazyFrame) -> list[str]:
    try:
        return list(lf.collect_schema().names())
    except Exception:
        return list(lf.schema.keys())


def ensure_numeric_cols(lf: pl.LazyFrame, cols: Iterable[str], dtype=pl.Float32) -> pl.LazyFrame:
    existing = set(lazy_columns(lf))
    missing_exprs = [pl.lit(0).cast(dtype).alias(c) for c in cols if c not in existing]
    return lf.with_columns(missing_exprs) if missing_exprs else lf


def clean_output_dir(path: Path) -> None:
    if path.exists():
        if path.is_dir():
            shutil.rmtree(path)
        else:
            path.unlink()
    path.mkdir(parents=True, exist_ok=True)


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def events_path(level: str) -> Path:
    return AGG_DIR / "events" / level


def orders_path(level: str) -> Path:
    return AGG_DIR / "orders" / level


def _date_filter(lf: pl.LazyFrame, date_col: str, start: date, end: date) -> pl.LazyFrame:
    return (
        lf.with_columns(pl.col(date_col).cast(pl.Date, strict=False).alias(date_col))
        .filter(
            (pl.col(date_col) >= pl.lit(start).cast(pl.Date))
            & (pl.col(date_col) <= pl.lit(end).cast(pl.Date))
        )
    )


def read_events_lf(level: str, start: date, end: date) -> pl.LazyFrame:
    lf = scan_parquet(events_path(level), hive_partitioning=True)
    return _date_filter(lf, "event_date", start, end)


def read_orders_lf(level: str, start: date, end: date) -> pl.LazyFrame:
    lf = scan_parquet(orders_path(level), hive_partitioning=True)
    return _date_filter(lf, "created_date", start, end)


def candidate_files(path: Path) -> list[Path]:
    if not path.exists():
        return []
    return sorted(path.rglob("*.parquet"))


def scan_candidate_shard_or_none(candidates_dir: Path, shard_id: int) -> pl.LazyFrame | None:
    shard_dir = candidates_dir / f"user_shard={shard_id}"
    if not path_has_parquet(shard_dir):
        return None
    return scan_parquet(shard_dir, hive_partitioning=True)


def write_single_shard(df: pl.DataFrame, out_dir: Path, shard_id: int, filename: str = "part.parquet") -> int:
    shard_dir = out_dir / f"user_shard={int(shard_id)}"
    ensure_dir(shard_dir)
    path = shard_dir / filename
    df.write_parquet(path, compression="zstd")
    return int(df.height)


def rel(path: Path | str) -> str:
    path = Path(path)
    try:
        return str(path.resolve().relative_to(PROJECT_ROOT.resolve()))
    except Exception:
        return str(path)


def candidate_set_key(split_name: str) -> str:
    # Candidate-scoped features depend on candidate pool. Keep them separated from old base-reranker caches.
    return f"{sanitize_name(ACTIVE_CANDIDATE_INPUT_NAME)}_{sanitize_name(split_name)}"

In [34]:
def resolve_candidate_dir(split: str) -> tuple[Path, str]:
    tried: list[Path] = []
    for name in (RCFG.candidate_input_name, *RCFG.fallback_candidate_input_names):
        path = CANDIDATES_DIR / name / split
        tried.append(path)
        if path_has_parquet(path):
            return path, name
    tried_msg = "\n".join(f"- {rel(p)}" for p in tried)
    raise FileNotFoundError(
        f"Не найден candidate pool для split={split}. Проверенные пути:\n{tried_msg}\n"
        "Для финального ранкера сначала запусти DL_Merge_Experiments_4sources.ipynb "
        "с materialized_output_name='merged_with_dl_all'."
    )


train_candidates_dir, train_candidate_input_name = resolve_candidate_dir("ranker_train")
val_candidates_dir, val_candidate_input_name = resolve_candidate_dir("val")

if RCFG.enable_test:
    test_candidates_dir, test_candidate_input_name = resolve_candidate_dir("test")
else:
    test_candidates_dir = None
    test_candidate_input_name = None
    print("NOTE: test disabled (RCFG.enable_test=False) — repeat_history/test not built")
if train_candidate_input_name != val_candidate_input_name:
    raise RuntimeError(
        f"train and val must use the same candidate pool: "
        f"{train_candidate_input_name!r} vs {val_candidate_input_name!r}"
    )
if test_candidate_input_name and test_candidate_input_name != train_candidate_input_name:
    print(
        f"NOTE: test uses fallback pool {test_candidate_input_name!r} "
        f"(train/val use {train_candidate_input_name!r})."
    )

ACTIVE_CANDIDATE_INPUT_NAME = train_candidate_input_name
RUN_NAME = f"{sanitize_name(RCFG.run_name)}__{sanitize_name(ACTIVE_CANDIDATE_INPUT_NAME)}"
PROCESSED_DIR = PROCESSED_ROOT / RUN_NAME
DATASET_DIR = PROCESSED_DIR / "dataset"
MODELS_DIR = MODELS_ROOT / RUN_NAME
RERANKED_DIR = RERANKED_ROOT / RUN_NAME
REPORTS_DIR = REPORTS_ROOT / RUN_NAME

for path in [PROCESSED_DIR, DATASET_DIR, MODELS_DIR, RERANKED_DIR, REPORTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("ACTIVE_CANDIDATE_INPUT_NAME:", ACTIVE_CANDIDATE_INPUT_NAME)
print("train_candidates_dir:", rel(train_candidates_dir))
print("val_candidates_dir:  ", rel(val_candidates_dir))
print("test_candidates_dir: ", rel(test_candidates_dir) if test_candidates_dir is not None else "DISABLED")
print("RUN_NAME:", RUN_NAME)
print("DATASET_DIR:", rel(DATASET_DIR))
print("MODELS_DIR:", rel(MODELS_DIR))
print("REPORTS_DIR:", rel(REPORTS_DIR))

if ACTIVE_CANDIDATE_INPUT_NAME != RCFG.candidate_input_name:
    print(
        f"WARNING: основной финальный pool {RCFG.candidate_input_name!r} не найден; "
        f"используется fallback {ACTIVE_CANDIDATE_INPUT_NAME!r}."
    )

ACTIVE_CANDIDATE_INPUT_NAME: merged_with_dl_all_repeat_history
train_candidates_dir: data\datamart\candidates\merged_with_dl_all_repeat_history\ranker_train
val_candidates_dir:   data\datamart\candidates\merged_with_dl_all_repeat_history\val
test_candidates_dir:  data\datamart\candidates\merged_with_dl_all_repeat_history\test
RUN_NAME: final_reranker_v12_repeat__merged_with_dl_all_repeat_history
DATASET_DIR: data\processed\reranker\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\dataset
MODELS_DIR: models\reranker\final_reranker_v12_repeat__merged_with_dl_all_repeat_history
REPORTS_DIR: artifacts\reports\final_reranker_v12_repeat__merged_with_dl_all_repeat_history


In [35]:
def build_ground_truth(start: date, end: date) -> dict[int, list[int]]:
    gt_df = collect_streaming(
        read_orders_lf("user_item", start, end)
        .select(
            pl.col("user_id").cast(pl.Int64),
            pl.col("item_id").cast(pl.Int64),
        )
        .drop_nulls()
        .unique()
        .group_by("user_id")
        .agg(pl.col("item_id").sort().alias("gt_items"))
        .sort("user_id")
    )
    return {int(u): [int(x) for x in items] for u, items in zip(gt_df["user_id"], gt_df["gt_items"])}


def sample_user_ids(user_ids: list[int], n: int | None, seed: int = RCFG.random_state) -> list[int]:
    user_ids = sorted(int(x) for x in user_ids)
    if n is None or n >= len(user_ids):
        return user_ids
    rng = np.random.default_rng(seed)
    sampled = rng.choice(np.asarray(user_ids, dtype=np.int64), size=n, replace=False)
    return sorted(int(x) for x in sampled)


def find_test_users_path() -> Path | None:
    candidates = [
        DATA_DIR / "ml_ozon_recsys_test.snappy.parquet",
        DATA_DIR / "ml_ozon_recsys_test.parquet",
        DATA_DIR / "test.parquet",
        RAW_DIR / "ml_ozon_recsys_test.snappy.parquet",
        RAW_DIR / "ml_ozon_recsys_test.parquet",
        RAW_DIR / "ml_ozon_recsys_test_final_apparel_orders_data",
        RAW_DIR / "ml_ozon_recsys_test_final_apparel_orders_data.parquet",
    ]
    for path in candidates:
        if path.exists():
            return path

    for root in [DATA_DIR, RAW_DIR]:
        if root.exists():
            matched = sorted(
                p for p in root.iterdir()
                if "test" in p.name.lower() and p.suffix.lower() in {".parquet", ".pq"}
            )
            if matched:
                return matched[0]
    return None


def load_test_users() -> list[int]:
    test_path = find_test_users_path()
    if test_path is not None:
        print("test users path:", test_path)
        df = collect_streaming(
            scan_parquet(test_path, hive_partitioning=True)
            .select(pl.col("user_id").cast(pl.Int64))
            .drop_nulls()
            .unique()
            .sort("user_id")
        )
        return [int(x) for x in df["user_id"].to_numpy()]

    print("Test users file not found. Fallback: users from datamart orders in test target window.")
    return sorted(build_ground_truth(RCFG.test_target_start, RCFG.test_target_end).keys())


train_gt = build_ground_truth(RCFG.train_target_start, RCFG.train_target_end)
val_gt = build_ground_truth(RCFG.val_target_start, RCFG.val_target_end)
if RCFG.enable_test:
    test_gt = build_ground_truth(RCFG.test_target_start, RCFG.test_target_end) if RCFG.test_has_labels else {}
    test_users_all = load_test_users()
else:
    test_gt = {}
    test_users_all = []

train_users = sample_user_ids(sorted(train_gt.keys()), RCFG.sample_train_users)
val_users = sample_user_ids(sorted(val_gt.keys()), RCFG.sample_val_users)
test_users = sample_user_ids(test_users_all, RCFG.sample_test_users) if RCFG.enable_test else []

# Для финальных test-метрик оцениваем только тех test users, у кого есть ground truth в test target window
# и кто попал в текущий test dataset/user sample. Остальные users все равно попадут в recommendations/fallback.
test_user_set = set(map(int, test_users))
test_eval_gt = {int(u): items for u, items in test_gt.items() if int(u) in test_user_set}

print(f"train users with GT: {len(train_users):,}")
print(f"val users with GT:   {len(val_users):,}")
print(f"test users total:    {len(test_users):,}")
print(f"test users with GT:  {len(test_eval_gt):,}")
print(f"avg train GT size:   {np.mean([len(train_gt[u]) for u in train_users]):.3f}")
print(f"avg val GT size:     {np.mean([len(val_gt[u]) for u in val_users]):.3f}")
if test_eval_gt:
    print(f"avg test GT size:    {np.mean([len(v) for v in test_eval_gt.values()]):.3f}")
elif RCFG.test_has_labels:
    warnings.warn("RCFG.test_has_labels=True, но test_eval_gt пустой. Проверь test users и окно test target.")


test users path: c:\Users\denis\Documents\Repo\ozon\data\raw\ml_ozon_recsys_test.snappy.parquet
train users with GT: 376,016
val users with GT:   359,928
test users total:    470,347
test users with GT:  309,565
avg train GT size:   4.626
avg val GT size:     4.381
avg test GT size:    4.395


## Dataset Build

Строит `train_fit`, `val_fit`, `val_eval` из нового пула 512 шардов.
`add_repeat_derived_features` вызывается сразу после `add_candidate_features` внутри `build_ranker_dataset`,
поэтому все repeat-признаки записываются в датасет за один проход.

- `*_rank` / `*_score` колонки пула → автопрефикс `f_num__candidate__*` через `add_candidate_features`
- Bool-флаги и производные признаки → добавляются `add_repeat_derived_features`
- test отключён (`enable_test = False`): пул `repeat_history/test` не построен


## Метрики

Метрики считаются по пользователям с ground truth в target-окне.
`val_eval` используется для выбора модели и гиперпараметров.
`test` в этом проекте — labeled holdout: на нём считаются финальные метрики один раз, без подбора модели по test.


In [36]:
def _unique_int_list(values) -> list[int]:
    if values is None:
        return []
    return list(dict.fromkeys(int(x) for x in values))


def user_metrics_at_k(gt_items: Sequence[int], pred_items: Sequence[int], k: int) -> dict[str, float]:
    gt = set(_unique_int_list(gt_items))
    if not gt:
        return {
            "hit_rate": np.nan,
            "recall": np.nan,
            "precision": np.nan,
            "ndcg": np.nan,
            "map": np.nan,
            "mrr": np.nan,
        }

    pred = _unique_int_list(pred_items)[:k]
    hits = [1 if item in gt else 0 for item in pred]

    hit_rate = float(any(hits))
    recall = float(sum(hits) / len(gt))
    precision = float(sum(hits) / k) if k > 0 else 0.0

    dcg = sum(h / np.log2(rank + 2.0) for rank, h in enumerate(hits))
    idcg = sum(1.0 / np.log2(rank + 2.0) for rank in range(min(len(gt), k)))
    ndcg = float(dcg / idcg) if idcg > 0 else 0.0

    hit_count = 0
    ap_sum = 0.0
    rr = 0.0
    for rank, h in enumerate(hits, start=1):
        if h:
            hit_count += 1
            ap_sum += hit_count / rank
            if rr == 0.0:
                rr = 1.0 / rank
    ap = float(ap_sum / len(gt))

    return {
        "hit_rate": hit_rate,
        "recall": recall,
        "precision": precision,
        "ndcg": ndcg,
        "map": ap,
        "mrr": float(rr),
    }


def evaluate_predictions_by_user(
    preds_by_user: dict[int, list[int]],
    ground_truth: dict[int, list[int]],
    *,
    ks: Iterable[int] = RCFG.metric_ks,
    label: str = "model",
) -> pd.DataFrame:
    rows = []
    gt_users = set(int(u) for u in ground_truth)

    for k in ks:
        sums = defaultdict(float)
        n_eval = 0

        for user_id, gt_items in ground_truth.items():
            user_id = int(user_id)
            pred_items = preds_by_user.get(user_id, [])
            m = user_metrics_at_k(gt_items, pred_items, int(k))
            if np.isnan(m["ndcg"]):
                continue
            n_eval += 1
            for metric, value in m.items():
                sums[metric] += float(value)

        row = {
            "model": label,
            "k": int(k),
            "users_total": len(gt_users),
            "users_with_predictions": len(set(preds_by_user) & gt_users),
            "user_coverage": len(set(preds_by_user) & gt_users) / len(gt_users) if gt_users else np.nan,
        }
        for metric in ["hit_rate", "recall", "precision", "ndcg", "map", "mrr"]:
            row[metric] = sums[metric] / n_eval if n_eval else np.nan
        rows.append(row)

    return pd.DataFrame(rows)


def evaluate_scored_df(
    df: pl.DataFrame | pd.DataFrame,
    ground_truth: dict[int, list[int]],
    *,
    score_col: str = "score",
    label: str = "model",
    ks: Iterable[int] = RCFG.metric_ks,
    higher_is_better: bool = True,
) -> pd.DataFrame:
    if isinstance(df, pl.DataFrame):
        pdf = df.select(["user_id", "item_id", score_col]).to_pandas()
    else:
        pdf = df[["user_id", "item_id", score_col]].copy()

    pdf = pdf.sort_values(["user_id", score_col, "item_id"], ascending=[True, not higher_is_better, True])
    preds_by_user = {
        int(user_id): group["item_id"].astype(int).tolist()
        for user_id, group in pdf.groupby("user_id", sort=False)
    }
    return evaluate_predictions_by_user(preds_by_user, ground_truth, ks=ks, label=label)


def evaluate_dataset_score_col(
    dataset_dir: Path,
    ground_truth: dict[int, list[int]],
    *,
    score_col: str,
    label: str,
    higher_is_better: bool = True,
    ks: Iterable[int] = RCFG.metric_ks,
) -> pd.DataFrame:
    preds_by_user: dict[int, list[int]] = {}
    files = candidate_files(dataset_dir)
    for file in tqdm(files, desc=f"Evaluate {label}"):
        part = pl.read_parquet(file, columns=["user_id", "item_id", score_col])
        part = part.sort(["user_id", score_col, "item_id"], descending=[False, higher_is_better, False])
        for row in part.group_by("user_id", maintain_order=True).agg(pl.col("item_id").alias("items")).iter_rows(named=True):
            preds_by_user[int(row["user_id"])] = [int(x) for x in row["items"]]
    return evaluate_predictions_by_user(preds_by_user, ground_truth, ks=ks, label=label)


def evaluate_scored_dir(
    scored_dir: Path,
    ground_truth: dict[int, list[int]],
    *,
    rank_col: str = "rank",
    label: str = "model",
    ks: Iterable[int] = RCFG.metric_ks,
) -> pd.DataFrame:
    preds_by_user: dict[int, list[int]] = {}
    files = candidate_files(scored_dir)
    for file in tqdm(files, desc=f"Evaluate {label}"):
        part = pl.read_parquet(file, columns=["user_id", "item_id", rank_col])
        part = part.sort(["user_id", rank_col, "item_id"], descending=[False, False, False])
        for row in part.group_by("user_id", maintain_order=True).agg(pl.col("item_id").alias("items")).iter_rows(named=True):
            preds_by_user[int(row["user_id"])] = [int(x) for x in row["items"]]
    return evaluate_predictions_by_user(preds_by_user, ground_truth, ks=ks, label=label)

## Point-in-time Feature Store

Уровни `user / item / category / user_category` переиспользуются по `as_of_date`.
Высококардинальный уровень `user_item` строится только для candidate pairs текущего пула —
полная матрица user-item не материализуется.


In [37]:
def safe_ratio_expr(num_col: str, den_col: str, out_col: str) -> pl.Expr:
    num = pl.col(num_col).cast(pl.Float32).fill_null(0.0)
    den = pl.col(den_col).cast(pl.Float32).fill_null(0.0)
    return (
        pl.when(den > 0.0)
        .then(num / den)
        .otherwise(pl.lit(None))
        .cast(pl.Float32)
        .alias(out_col)
    )


def item_catalog_lf() -> pl.LazyFrame:
    """
    item_id -> catalogid mapping. It is needed both for global category features and for
    candidate-scoped user_category features.
    """
    path = DATAMART_DIR / "mappings" / "item_catalog.parquet"
    if path.exists():
        return scan_parquet(path, hive_partitioning=False).select(
            pl.col("item_id").cast(pl.Int64),
            pl.col("catalogid").cast(pl.Int64),
        ).unique()

    # Fallback: infer mapping from available user_item aggregates.
    for candidate in [events_path("user_item"), orders_path("user_item")]:
        if path_has_parquet(candidate):
            lf = scan_parquet(candidate, hive_partitioning=True)
            cols = lazy_columns(lf)
            if "catalogid" in cols:
                return lf.select(
                    pl.col("item_id").cast(pl.Int64),
                    pl.col("catalogid").cast(pl.Int64),
                ).drop_nulls().unique()

    raise FileNotFoundError(
        "Не найден item->catalogid mapping. Ожидался data/datamart/mappings/item_catalog.parquet "
        "или catalogid в agg/events/user_item | agg/orders/user_item."
    )


def source_lf_for_level(domain: str, level: str, start: date, end: date) -> pl.LazyFrame | None:
    if domain == "events":
        path_fn = events_path
        read_fn = read_events_lf
    elif domain == "orders":
        path_fn = orders_path
        read_fn = read_orders_lf
    else:
        raise ValueError(domain)

    if path_has_parquet(path_fn(level)):
        return read_fn(level, start, end)

    # Fallback from user_item if derived aggs were not built.
    if level != "user_item" and path_has_parquet(path_fn("user_item")):
        return read_fn("user_item", start, end)

    return None


def prepare_source_keys_for_level(lf: pl.LazyFrame, level: str) -> pl.LazyFrame:
    """Ensure keys required by `level` exist and have stable integer dtypes."""
    cols = set(lazy_columns(lf))

    if "catalogid" in LEVEL_KEYS[level] and "catalogid" not in cols:
        if "item_id" not in cols:
            raise ValueError(f"Cannot build {level}: source has neither catalogid nor item_id")
        lf = lf.join(item_catalog_lf(), on="item_id", how="left")
        cols.add("catalogid")

    casts = []
    for key in LEVEL_KEYS[level]:
        if key not in cols and key not in lazy_columns(lf):
            raise ValueError(f"Cannot build {level}: missing key column {key}")
        casts.append(pl.col(key).cast(pl.Int64, strict=False).alias(key))
    return lf.with_columns(casts)


def aggregate_events_level(level: str, start: date, end: date, window: int) -> pl.LazyFrame | None:
    keys = LEVEL_KEYS[level]
    lf = source_lf_for_level("events", level, start, end)
    if lf is None:
        return None

    lf = prepare_source_keys_for_level(lf, level)
    lf = ensure_numeric_cols(lf, EVENT_METRIC_COLS)

    prefix = f"f_num__{level}__"
    agg_exprs = [
        pl.sum(c).cast(pl.Float32).alias(f"{prefix}{c}_{window}d")
        for c in EVENT_METRIC_COLS
    ]
    agg_exprs.append(pl.max("event_date").alias("__last_event_date"))

    out = lf.group_by(keys).agg(agg_exprs)

    cnt_events = f"{prefix}cnt_events_{window}d"
    cnt_page = f"{prefix}cnt_page_view_{window}d"
    cnt_desc = f"{prefix}cnt_view_description_{window}d"
    cnt_review = f"{prefix}cnt_review_view_{window}d"
    cnt_cart = f"{prefix}cnt_to_cart_{window}d"
    cnt_fav = f"{prefix}cnt_favorite_{window}d"

    out = out.with_columns(
        (pl.lit(end).cast(pl.Date) - pl.col("__last_event_date").cast(pl.Date))
        .dt.total_days()
        .cast(pl.Float32)
        .alias(f"{prefix}days_since_last_event_{window}d"),
        (pl.col(cnt_events).fill_null(0.0) > 0).cast(pl.Int8).alias(f"{prefix}has_events_{window}d"),
        safe_ratio_expr(cnt_cart, cnt_page, f"{prefix}to_cart_per_page_view_{window}d"),
        safe_ratio_expr(cnt_fav, cnt_page, f"{prefix}favorite_per_page_view_{window}d"),
        safe_ratio_expr(cnt_desc, cnt_page, f"{prefix}description_per_page_view_{window}d"),
        safe_ratio_expr(cnt_review, cnt_page, f"{prefix}review_per_page_view_{window}d"),
    ).drop("__last_event_date")

    return out


def aggregate_orders_level(level: str, start: date, end: date, window: int) -> pl.LazyFrame | None:
    keys = LEVEL_KEYS[level]
    lf = source_lf_for_level("orders", level, start, end)
    if lf is None:
        return None

    lf = prepare_source_keys_for_level(lf, level)
    lf = ensure_numeric_cols(lf, ORDER_METRIC_COLS)

    prefix = f"f_num__{level}__"
    agg_exprs = [
        pl.sum(c).cast(pl.Float32).alias(f"{prefix}{c}_{window}d")
        for c in ORDER_METRIC_COLS
    ]
    agg_exprs.append(pl.max("created_date").alias("__last_order_date"))

    out = lf.group_by(keys).agg(agg_exprs)

    cnt_orders = f"{prefix}cnt_orders_{window}d"
    cnt_delivered = f"{prefix}cnt_delivered_orders_{window}d"
    cnt_cancel = f"{prefix}cnt_canceled_orders_{window}d"
    cnt_processed = f"{prefix}cnt_proccesed_orders_{window}d"

    out = out.with_columns(
        (pl.lit(end).cast(pl.Date) - pl.col("__last_order_date").cast(pl.Date))
        .dt.total_days()
        .cast(pl.Float32)
        .alias(f"{prefix}days_since_last_order_{window}d"),
        (pl.col(cnt_orders).fill_null(0.0) > 0).cast(pl.Int8).alias(f"{prefix}has_orders_{window}d"),
        safe_ratio_expr(cnt_delivered, cnt_orders, f"{prefix}delivered_share_{window}d"),
        safe_ratio_expr(cnt_cancel, cnt_orders, f"{prefix}cancel_share_{window}d"),
        safe_ratio_expr(cnt_processed, cnt_orders, f"{prefix}processed_share_{window}d"),
    ).drop("__last_order_date")

    return out


def feature_date_dir(as_of_date: date) -> Path:
    return FEATURES_DIR / f"as_of_date={as_of_date.isoformat()}"


def build_level_features_for_split(split_name: str, context_start: date, context_end: date, level: str) -> Path | None:
    # split_name is used only for logs. The physical feature partition is date-based.
    out_dir = feature_date_dir(context_end)
    ensure_dir(out_dir)
    out_path = out_dir / f"{level}.parquet"

    if out_path.exists() and not RCFG.rebuild_features:
        return out_path

    parts: list[pl.LazyFrame] = []

    for window in RCFG.feature_windows:
        win_start = max(context_start, context_end - timedelta(days=int(window) - 1))

        ev = aggregate_events_level(level, win_start, context_end, int(window))
        if ev is not None:
            parts.append(ev)

        orders = aggregate_orders_level(level, win_start, context_end, int(window))
        if orders is not None:
            parts.append(orders)

    if not parts:
        print(f"{split_name}/{level}: no feature parts")
        return None

    keys = LEVEL_KEYS[level]
    base_keys = pl.concat([p.select(keys) for p in parts], how="vertical_relaxed").unique()

    features_lf = base_keys
    for part in parts:
        features_lf = features_lf.join(part, on=keys, how="left")

    df = collect_streaming(features_lf)
    if df.is_empty():
        print(f"{split_name}/{level}: empty")
        return None

    df.write_parquet(out_path, compression="zstd")
    print(f"{split_name}/{level}: rows={df.height:,}, cols={df.width:,} -> {rel(out_path)}")
    del df
    gc.collect()
    return out_path


def scan_candidate_keys_for_shards(candidates_dir: Path, level: str, shard_ids: Sequence[int]) -> pl.LazyFrame | None:
    """
    Read only keys that may be joined to candidates. This is the core optimization:
    high-cardinality features are computed for candidate pairs/categories, not for all historical pairs.
    """
    keys = LEVEL_KEYS[level]
    parts: list[pl.LazyFrame] = []

    for shard_id in shard_ids:
        cand_lf = scan_candidate_shard_or_none(candidates_dir, int(shard_id))
        if cand_lf is None:
            continue

        cand_lf = cand_lf.select(
            pl.col("user_id").cast(pl.Int64),
            pl.col("item_id").cast(pl.Int64),
        )

        if level == "user_item":
            key_lf = cand_lf.select(keys)
        elif level == "user_category":
            key_lf = (
                cand_lf.join(item_catalog_lf(), on="item_id", how="left")
                .with_columns(pl.col("catalogid").fill_null(-1).cast(pl.Int64).alias("catalogid"))
                .select(keys)
            )
        else:
            raise ValueError(f"Candidate-scoped build is intended only for high-cardinality levels, got {level}")

        parts.append(key_lf)

    if not parts:
        return None

    return pl.concat(parts, how="vertical_relaxed").unique()


def restrict_to_candidate_keys(lf: pl.LazyFrame, level: str, candidate_keys_lf: pl.LazyFrame) -> pl.LazyFrame:
    keys = LEVEL_KEYS[level]
    lf = prepare_source_keys_for_level(lf, level)
    candidate_keys_lf = candidate_keys_lf.select(keys).unique()
    return lf.join(candidate_keys_lf, on=keys, how="inner")


def aggregate_events_level_candidate_scoped(
    level: str,
    candidate_keys_lf: pl.LazyFrame,
    start: date,
    end: date,
    window: int,
) -> pl.LazyFrame | None:
    lf = source_lf_for_level("events", level, start, end)
    if lf is None:
        return None
    lf = restrict_to_candidate_keys(lf, level, candidate_keys_lf)
    lf = ensure_numeric_cols(lf, EVENT_METRIC_COLS)

    keys = LEVEL_KEYS[level]
    prefix = f"f_num__{level}__"
    agg_exprs = [pl.sum(c).cast(pl.Float32).alias(f"{prefix}{c}_{window}d") for c in EVENT_METRIC_COLS]
    agg_exprs.append(pl.max("event_date").alias("__last_event_date"))

    out = lf.group_by(keys).agg(agg_exprs)

    cnt_events = f"{prefix}cnt_events_{window}d"
    cnt_page = f"{prefix}cnt_page_view_{window}d"
    cnt_desc = f"{prefix}cnt_view_description_{window}d"
    cnt_review = f"{prefix}cnt_review_view_{window}d"
    cnt_cart = f"{prefix}cnt_to_cart_{window}d"
    cnt_fav = f"{prefix}cnt_favorite_{window}d"

    return out.with_columns(
        (pl.lit(end).cast(pl.Date) - pl.col("__last_event_date").cast(pl.Date))
        .dt.total_days()
        .cast(pl.Float32)
        .alias(f"{prefix}days_since_last_event_{window}d"),
        (pl.col(cnt_events).fill_null(0.0) > 0).cast(pl.Int8).alias(f"{prefix}has_events_{window}d"),
        safe_ratio_expr(cnt_cart, cnt_page, f"{prefix}to_cart_per_page_view_{window}d"),
        safe_ratio_expr(cnt_fav, cnt_page, f"{prefix}favorite_per_page_view_{window}d"),
        safe_ratio_expr(cnt_desc, cnt_page, f"{prefix}description_per_page_view_{window}d"),
        safe_ratio_expr(cnt_review, cnt_page, f"{prefix}review_per_page_view_{window}d"),
    ).drop("__last_event_date")


def aggregate_orders_level_candidate_scoped(
    level: str,
    candidate_keys_lf: pl.LazyFrame,
    start: date,
    end: date,
    window: int,
) -> pl.LazyFrame | None:
    lf = source_lf_for_level("orders", level, start, end)
    if lf is None:
        return None
    lf = restrict_to_candidate_keys(lf, level, candidate_keys_lf)
    lf = ensure_numeric_cols(lf, ORDER_METRIC_COLS)

    keys = LEVEL_KEYS[level]
    prefix = f"f_num__{level}__"
    agg_exprs = [pl.sum(c).cast(pl.Float32).alias(f"{prefix}{c}_{window}d") for c in ORDER_METRIC_COLS]
    agg_exprs.append(pl.max("created_date").alias("__last_order_date"))

    out = lf.group_by(keys).agg(agg_exprs)

    cnt_orders = f"{prefix}cnt_orders_{window}d"
    cnt_delivered = f"{prefix}cnt_delivered_orders_{window}d"
    cnt_cancel = f"{prefix}cnt_canceled_orders_{window}d"
    cnt_processed = f"{prefix}cnt_proccesed_orders_{window}d"

    return out.with_columns(
        (pl.lit(end).cast(pl.Date) - pl.col("__last_order_date").cast(pl.Date))
        .dt.total_days()
        .cast(pl.Float32)
        .alias(f"{prefix}days_since_last_order_{window}d"),
        (pl.col(cnt_orders).fill_null(0.0) > 0).cast(pl.Int8).alias(f"{prefix}has_orders_{window}d"),
        safe_ratio_expr(cnt_delivered, cnt_orders, f"{prefix}delivered_share_{window}d"),
        safe_ratio_expr(cnt_cancel, cnt_orders, f"{prefix}cancel_share_{window}d"),
        safe_ratio_expr(cnt_processed, cnt_orders, f"{prefix}processed_share_{window}d"),
    ).drop("__last_order_date")


def write_feature_batch_by_user_shard(df: pl.DataFrame, out_dir: Path, shard_ids: Sequence[int]) -> int:
    if df.is_empty():
        return 0

    if "user_id" not in df.columns:
        raise ValueError("Candidate-scoped features must contain user_id for sharded write")

    df = df.with_columns((pl.col("user_id").cast(pl.Int64) % int(RCFG.n_shards)).cast(pl.Int32).alias("user_shard"))
    rows_written = 0
    for shard_id in shard_ids:
        part = df.filter(pl.col("user_shard") == int(shard_id)).drop("user_shard")
        if part.is_empty():
            continue
        rows_written += write_single_shard(part, out_dir, int(shard_id), "features.parquet")
    return rows_written


def build_candidate_scoped_level_features_for_split(
    split_name: str,
    candidates_dir: Path,
    context_start: date,
    context_end: date,
    level: str,
) -> Path | None:
    """
    Optimized high-cardinality feature builder.

    Instead of materializing all historical user_item/user_category combinations, it:
    1. reads candidate keys for a small batch of user shards;
    2. filters historical events/orders to these keys via inner join;
    3. aggregates only the surviving rows;
    4. writes features back partitioned by user_shard.
    """
    # Candidate-scoped interaction features are date-based, but also depend on the candidate set.
    # candidate_set is kept in the path because the keys are limited to this particular pool.
    out_dir = FEATURES_DIR / "candidate_scoped" / f"as_of_date={context_end.isoformat()}" / f"candidate_set={candidate_set_key(split_name)}" / level

    if path_has_parquet(out_dir) and not RCFG.rebuild_features:
        return out_dir

    clean_output_dir(out_dir)
    total_rows = 0
    batch_size = max(1, int(RCFG.candidate_scoped_shard_batch_size))
    all_shards = list(RCFG.smoke_shards) if RCFG.smoke_shards is not None else list(range(int(RCFG.n_shards)))

    for pos in trange(0, len(all_shards), batch_size, desc=f"{split_name}/{level} candidate-scoped"):
        shard_ids = all_shards[pos:pos + batch_size]
        candidate_keys_lf = scan_candidate_keys_for_shards(candidates_dir, level, shard_ids)
        if candidate_keys_lf is None:
            continue

        parts: list[pl.LazyFrame] = []
        for window in RCFG.feature_windows:
            win_start = max(context_start, context_end - timedelta(days=int(window) - 1))

            ev = aggregate_events_level_candidate_scoped(level, candidate_keys_lf, win_start, context_end, int(window))
            if ev is not None:
                parts.append(ev)

            orders = aggregate_orders_level_candidate_scoped(level, candidate_keys_lf, win_start, context_end, int(window))
            if orders is not None:
                parts.append(orders)

        if not parts:
            continue

        keys = LEVEL_KEYS[level]
        base_keys = pl.concat([p.select(keys) for p in parts], how="vertical_relaxed").unique()
        features_lf = base_keys
        for part in parts:
            features_lf = features_lf.join(part, on=keys, how="left")

        df = collect_streaming(features_lf)
        if df.is_empty():
            del df
            gc.collect()
            continue

        rows_written = write_feature_batch_by_user_shard(df, out_dir, shard_ids)
        total_rows += rows_written
        print(f"{split_name}/{level}: shards={shard_ids[0]}..{shard_ids[-1]}, rows={rows_written:,}, cols={df.width:,}")
        del df
        gc.collect()

    if total_rows == 0:
        print(f"{split_name}/{level}: no candidate-scoped features")
        return None

    print(f"{split_name}/{level}: total_rows={total_rows:,} -> {rel(out_dir)}")
    return out_dir


def build_feature_store_for_split(
    split_name: str,
    context_start: date,
    context_end: date,
    candidates_dir: Path,
) -> dict[str, Path]:
    paths: dict[str, Path] = {}
    candidate_scoped_levels = set(RCFG.candidate_scoped_feature_levels)

    for level in RCFG.feature_levels:
        if level not in LEVEL_KEYS:
            raise ValueError(f"Unknown feature level: {level}. Available: {sorted(LEVEL_KEYS)}")

        if level in candidate_scoped_levels:
            path = build_candidate_scoped_level_features_for_split(
                split_name,
                candidates_dir,
                context_start,
                context_end,
                level,
            )
        else:
            path = build_level_features_for_split(split_name, context_start, context_end, level)

        if path is not None:
            paths[level] = path
    return paths

## Candidate Schema и Ranker Dataset

Pipeline устойчив к разным candidate pool'ам (`merged`, `merged_with_dl_all`, `merged_with_dl_all_repeat_history` и др.).
Source-specific колонки `*_rank`, `*_score`, `in_*` автоматически преобразуются
в числовые признаки `f_num__candidate__...` через `add_candidate_features`.


In [38]:
import math as _math


def add_repeat_derived_features(lf: pl.LazyFrame) -> pl.LazyFrame:
    """Add repeat-history features not auto-handled by add_candidate_features.

    Called right after add_candidate_features(lf) inside build_ranker_dataset.
    At that point f_num__candidate__*_rank / *_score columns already exist
    (fill_null(1_000_000) for ranks, fill_null(0.0) for scores).
    Raw pool columns (repeat_in_*, is_repeat_history_candidate, etc.) also exist.

    Groups added:
      A  non-auto bool flags  (repeat_in_*, is_repeat_history_candidate)
      B  rank-gap interaction (repeat_vs_*_rank_gap, repeat_rank_delta_to_best,
                               repeat_rank_is_top_source)
      C  position priors      (num_repeat_signals, repeat_rank_norm, repeat_rank_log,
                               is_top10_repeat, is_top50_repeat)
      D  source interaction   (n_source_signals, is_repeat_only_candidate)
    """
    cols = set(lf.collect_schema().names())
    FC = "f_num__candidate__"
    TOPK = 200.0
    exprs = []

    # ── Group A: non-auto bool flags ─────────────────────────────────────────
    for _raw in [
        "repeat_in_recency", "repeat_in_frequency", "repeat_in_cart_orders",
        "is_repeat_history_candidate",
    ]:
        if _raw in cols:
            exprs.append(
                pl.col(_raw).fill_null(False).cast(pl.Float32).alias(f"{FC}{_raw}")
            )

    # ── Group C: position priors (use raw repeat_history_rank, null = not present) ──
    if "repeat_history_rank" in cols:
        _rhr = pl.col("repeat_history_rank").cast(pl.Float32)
        _signal_cols = [
            c for c in ["repeat_in_recency", "repeat_in_frequency", "repeat_in_cart_orders"]
            if c in cols
        ]
        if _signal_cols:
            exprs.append(
                pl.sum_horizontal(*[
                    pl.col(c).fill_null(False).cast(pl.Float32) for c in _signal_cols
                ]).alias(f"{FC}num_repeat_signals")
            )
        exprs.extend([
            (_rhr.fill_null(9999.0) / TOPK).clip(upper_bound=5.0).alias(f"{FC}repeat_rank_norm"),
            (_rhr.fill_null(9999.0) + 1.0).log(base=_math.e).alias(f"{FC}repeat_rank_log"),
            (_rhr <= 10.0).fill_null(False).cast(pl.Float32).alias(f"{FC}is_top10_repeat"),
            (_rhr <= 50.0).fill_null(False).cast(pl.Float32).alias(f"{FC}is_top50_repeat"),
        ])

    # ── Group B: rank-gap features (use f_num__candidate__ prefixed ranks) ───
    _rhr_fc = f"{FC}repeat_history_rank"
    if _rhr_fc in cols:
        _rhr_v = pl.col(_rhr_fc)
        _gap_srcs = [
            ("dssm",   f"{FC}dl_dssm_final_rank"),
            ("sasrec", f"{FC}dl_sasrec_v3_rank"),
            ("als",    f"{FC}als_rank"),
            ("bm25",   f"{FC}i2i_bm25_rank"),
        ]
        _present_srcs = [(lbl, fc) for lbl, fc in _gap_srcs if fc in cols]
        _non_repeat_fcs = [fc for _, fc in _present_srcs]
        for _lbl, _fc in _present_srcs:
            exprs.append(
                (_rhr_v - pl.col(_fc)).cast(pl.Float32)
                .alias(f"{FC}repeat_vs_{_lbl}_rank_gap")
            )
        if _non_repeat_fcs:
            _best_non_repeat = pl.min_horizontal(*[pl.col(c) for c in _non_repeat_fcs])
            exprs.append(
                (_rhr_v - _best_non_repeat).cast(pl.Float32)
                .alias(f"{FC}repeat_rank_delta_to_best")
            )
            _best_all = pl.min_horizontal(*([_rhr_v] + [pl.col(c) for c in _non_repeat_fcs]))
            exprs.append(
                ((_rhr_v < 1_000_000) & (_rhr_v == _best_all)).cast(pl.Float32)
                .alias(f"{FC}repeat_rank_is_top_source")
            )

    # ── Group D: source interaction ───────────────────────────────────────────
    _in_dl = [
        c for c in [
            f"{FC}in_dl_dssm_final", f"{FC}in_dl_sasrec_v3", f"{FC}in_dl_bert4rec_small",
        ] if c in cols
    ]
    if _in_dl:
        _has_dl = pl.max_horizontal(*[pl.col(c).fill_null(0) for c in _in_dl])
        _signals = [_has_dl]
        for _fc in [f"{FC}in_als", f"{FC}in_i2i_bm25", f"{FC}in_top_popular"]:
            if _fc in cols:
                _signals.append(pl.col(_fc).fill_null(0))
        if "is_repeat_history_candidate" in cols:
            _repeat_sig = (
                pl.col("is_repeat_history_candidate").fill_null(False).cast(pl.Float32)
            )
            _signals.append(_repeat_sig)
            _non_repeat_max = pl.max_horizontal(*_signals[:-1])
            exprs.append(
                (_repeat_sig * (1 - _non_repeat_max)).cast(pl.Float32)
                .alias(f"{FC}is_repeat_only_candidate")
            )
        exprs.append(
            pl.sum_horizontal(*_signals).cast(pl.Float32).alias(f"{FC}n_source_signals")
        )

    if exprs:
        lf = lf.with_columns(exprs)
    return lf


def label_file_stats(path: Path) -> dict[str, int]:
    if not path.exists():
        return {"rows": 0, "positives": 0}
    df = collect_streaming(
        scan_parquet(path, hive_partitioning=False)
        .select(
            pl.len().alias("rows"),
            pl.col("label").fill_null(0).sum().alias("positives"),
        )
    )
    row = df.row(0, named=True)
    return {"rows": int(row["rows"] or 0), "positives": int(row["positives"] or 0)}


def build_labels_for_target(split_name: str, start: date, end: date, *, has_labels: bool) -> Path:
    out_dir = PROCESSED_DIR / "labels"
    ensure_dir(out_dir)
    out_path = out_dir / f"labels_{split_name}_{start.isoformat()}_{end.isoformat()}.parquet"

    schema = {
        "user_id": pl.Int64,
        "item_id": pl.Int64,
        "label": pl.UInt8,
        "target_order_cnt": pl.UInt16,
        "target_delivered_order_cnt": pl.UInt16,
        "target_canceled_order_cnt": pl.UInt16,
    }

    if out_path.exists() and not RCFG.rebuild_datasets:
        if not has_labels:
            return out_path

        stats = label_file_stats(out_path)
        if stats["rows"] > 0 and stats["positives"] > 0:
            return out_path

        print(
            f"{split_name}: existing labels are empty/unlabeled "
            f"(rows={stats['rows']:,}, positives={stats['positives']:,}) -> rebuild labels"
        )
        out_path.unlink(missing_ok=True)

    if not has_labels:
        pl.DataFrame(schema=schema).write_parquet(out_path, compression="zstd")
        return out_path

    lf = read_orders_lf("user_item", start, end)
    lf = ensure_numeric_cols(lf, ORDER_METRIC_COLS, dtype=pl.Float32)
    labels = collect_streaming(
        lf.select(
            pl.col("user_id").cast(pl.Int64),
            pl.col("item_id").cast(pl.Int64),
            pl.col("cnt_orders").fill_null(0).cast(pl.Float32),
            pl.col("cnt_delivered_orders").fill_null(0).cast(pl.Float32),
            pl.col("cnt_canceled_orders").fill_null(0).cast(pl.Float32),
        )
        .drop_nulls(["user_id", "item_id"])
        .group_by(["user_id", "item_id"])
        .agg(
            pl.sum("cnt_orders").alias("target_order_cnt_f"),
            pl.sum("cnt_delivered_orders").alias("target_delivered_order_cnt_f"),
            pl.sum("cnt_canceled_orders").alias("target_canceled_order_cnt_f"),
        )
        .with_columns(
            (pl.col("target_order_cnt_f") > 0).cast(pl.UInt8).alias("label"),
            pl.col("target_order_cnt_f").clip(0, 65535).cast(pl.UInt16).alias("target_order_cnt"),
            pl.col("target_delivered_order_cnt_f").clip(0, 65535).cast(pl.UInt16).alias("target_delivered_order_cnt"),
            pl.col("target_canceled_order_cnt_f").clip(0, 65535).cast(pl.UInt16).alias("target_canceled_order_cnt"),
        )
        .select(list(schema))
    )
    labels.write_parquet(out_path, compression="zstd")
    print(f"{split_name}: labels rows={labels.height:,}, positives={int(labels['label'].sum()):,} -> {rel(out_path)}")
    return out_path


def reciprocal_rank_expr(col: str, out_col: str) -> pl.Expr:
    x = pl.col(col).cast(pl.Float32, strict=False)
    return (
        pl.when(x.is_not_null() & (x > 0.0))
        .then(1.0 / x)
        .otherwise(0.0)
        .cast(pl.Float32)
        .alias(out_col)
    )


def log1p_nonnegative_expr(col: str, out_col: str) -> pl.Expr:
    x = pl.col(col).cast(pl.Float32, strict=False).fill_null(0.0)
    x = pl.when(x > 0.0).then(x).otherwise(0.0)
    return (x + 1.0).log().cast(pl.Float32).alias(out_col)


def ensure_candidate_schema(lf: pl.LazyFrame) -> pl.LazyFrame:
    cols = set(lazy_columns(lf))
    exprs: list[pl.Expr] = []

    if "candidate_rank" not in cols:
        if "rank" in cols:
            exprs.append(pl.col("rank").cast(pl.Int32, strict=False).alias("candidate_rank"))
        else:
            raise ValueError("Candidate shard must contain candidate_rank or rank")

    # Base non-DL source columns.
    defaults = {
        "top_popular_score": pl.Float32,
        "top_popular_rank": pl.Int32,
        "global_pop_rank": pl.Int32,
        "als_score": pl.Float32,
        "als_rank": pl.Int32,
        "in_top_popular": pl.Int8,
        "in_als": pl.Int8,
    }
    for col, dtype in defaults.items():
        if col not in cols:
            fill = 0 if col.startswith("in_") else None
            exprs.append(pl.lit(fill, dtype=dtype).alias(col))

    lf = lf.with_columns(exprs) if exprs else lf
    cols = set(lazy_columns(lf))

    in_cols = [c for c in cols if c.startswith("in_") and not c.startswith("infinite")]
    rank_cols = [c for c in cols if c.endswith("_rank") or c in {"rank", "candidate_rank"}]
    score_cols = [c for c in cols if c.endswith("_score") or c in {"score", "max_source_score", "best_score"}]

    exprs = []
    if "sources_count" not in cols:
        if in_cols:
            exprs.append(pl.sum_horizontal(*[pl.col(c).fill_null(0).cast(pl.Int16) for c in in_cols]).cast(pl.Int16).alias("sources_count"))
        else:
            exprs.append(pl.lit(1, dtype=pl.Int16).alias("sources_count"))

    if "min_source_rank" not in cols:
        usable = [c for c in rank_cols if c != "candidate_rank"] or ["candidate_rank"]
        exprs.append(pl.min_horizontal(*[pl.col(c).fill_null(1_000_000_000).cast(pl.Int32, strict=False) for c in usable]).cast(pl.Int32).alias("min_source_rank"))

    if "max_source_score" not in cols:
        usable = [c for c in score_cols if c != "max_source_score"]
        if usable:
            exprs.append(pl.max_horizontal(*[pl.col(c).fill_null(-1e9).cast(pl.Float32, strict=False) for c in usable]).cast(pl.Float32).alias("max_source_score"))
        else:
            exprs.append(pl.lit(0.0, dtype=pl.Float32).alias("max_source_score"))

    return lf.with_columns(exprs) if exprs else lf


def add_candidate_features(lf: pl.LazyFrame) -> pl.LazyFrame:
    cols = set(lazy_columns(lf))
    exprs: list[pl.Expr] = []
    produced = set(cols)

    def add(expr: pl.Expr, name: str):
        if name not in produced:
            exprs.append(expr.alias(name))
            produced.add(name)

    # Generic source flags.
    for col in sorted(c for c in cols if c.startswith("in_") and not c.startswith("f_")):
        add(pl.col(col).fill_null(0).cast(pl.Float32), f"f_num__candidate__{col}")

    # Generic ranks and reciprocal ranks.
    for col in sorted(c for c in cols if (c.endswith("_rank") or c in {"candidate_rank", "rank", "min_source_rank", "global_pop_rank"}) and not c.startswith("f_")):
        safe = col.replace("rank", "rank")
        add(pl.col(col).fill_null(1_000_000).cast(pl.Float32, strict=False), f"f_num__candidate__{safe}")
        add(reciprocal_rank_expr(col, f"f_num__candidate__{safe}_rr"), f"f_num__candidate__{safe}_rr")
        add(pl.col(col).is_null().cast(pl.Float32), f"f_num__candidate__{safe}_missing")

    # Generic scores. Keep raw scores because DL cosine/logit scores may be negative.
    for col in sorted(c for c in cols if (c.endswith("_score") or c in {"score", "max_source_score", "best_score"}) and not c.startswith("f_")):
        add(pl.col(col).fill_null(0.0).cast(pl.Float32, strict=False), f"f_num__candidate__{col}")
        add(log1p_nonnegative_expr(col, f"f_num__candidate__{col}_log1p_nonnegative"), f"f_num__candidate__{col}_log1p_nonnegative")
        add(pl.col(col).is_null().cast(pl.Float32), f"f_num__candidate__{col}_missing")

    # Stable aliases used by earlier notebooks/reports.
    aliases = {
        "sources_count": "f_num__candidate__sources_count",
        "candidate_rank": "f_num__candidate__rank",
        "min_source_rank": "f_num__candidate__min_source_rank",
        "max_source_score": "f_num__candidate__max_source_score",
    }
    for src, dst in aliases.items():
        if src in cols:
            add(pl.col(src).fill_null(0).cast(pl.Float32, strict=False), dst)

    if exprs:
        lf = lf.with_columns(exprs)

    # Ensure all f_num candidate columns are numeric float32.
    f_cols = [c for c in lazy_columns(lf) if c.startswith("f_num__candidate__")]
    return lf.with_columns([pl.col(c).cast(pl.Float32, strict=False).fill_null(0.0).alias(c) for c in f_cols])


def maybe_filter_users(lf: pl.LazyFrame, user_ids: Sequence[int] | None) -> pl.LazyFrame:
    if user_ids is None:
        return lf
    ids = list(map(int, user_ids))
    return lf.filter(pl.col("user_id").cast(pl.Int64).is_in(ids))


def feature_lf_for_join(level: str, path: Path, shard_id: int | None) -> pl.LazyFrame | None:
    if level in set(RCFG.candidate_scoped_feature_levels) and path.is_dir() and shard_id is not None:
        shard_dir = path / f"user_shard={int(shard_id)}"
        if not path_has_parquet(shard_dir):
            return None
        return scan_parquet(shard_dir, hive_partitioning=True)

    if not path_has_parquet(path) and not path.exists():
        return None
    return scan_parquet(path, hive_partitioning=path.is_dir())


def join_feature_paths(lf: pl.LazyFrame, feature_paths: dict[str, Path], shard_id: int | None = None) -> pl.LazyFrame:
    catalog = item_catalog_lf()
    lf = lf.join(catalog, on="item_id", how="left")
    lf = lf.with_columns(pl.col("catalogid").fill_null(-1).cast(pl.Int64).alias("catalogid"))

    for level in RCFG.feature_levels:
        keys = LEVEL_KEYS[level]
        path = feature_paths.get(level)
        if path is None:
            continue

        feat = feature_lf_for_join(level, path, shard_id)
        if feat is None:
            continue

        casts = []
        for key in keys:
            if key in {"user_id", "item_id", "catalogid"}:
                casts.append(pl.col(key).cast(pl.Int64, strict=False).alias(key))
        if casts:
            feat = feat.with_columns(casts)

        feat_cols = [c for c in lazy_columns(feat) if c.startswith("f_")]
        if feat_cols:
            lf = lf.join(feat.select(keys + feat_cols), on=keys, how="left")

    return lf


def maybe_downsample_fit_rows(lf: pl.LazyFrame, *, downsample: bool) -> pl.LazyFrame:
    if not downsample:
        return lf

    lf = lf.with_columns(
        pl.struct(["user_id", "item_id"]).hash(seed=RCFG.random_state).alias("__rand")
    ).with_columns(
        pl.col("__rand").rank(method="ordinal").over("user_id").alias("__rand_rank")
    )

    keep_expr = (
        (pl.col("label") > 0)
        | (pl.col("candidate_rank").fill_null(1_000_000) <= int(RCFG.always_keep_top_n_candidates))
        | (pl.col("__rand_rank") <= int(RCFG.max_random_negatives_per_user))
        | (pl.col("repeat_history_rank").fill_null(9999).cast(pl.Float32) <= 200)
    )
    lf = lf.filter(keep_expr).drop(["__rand", "__rand_rank"])

    if RCFG.filter_fit_groups_without_positive:
        lf = (
            lf.with_columns(pl.max("label").over("user_id").alias("__group_max_label"))
            .filter(pl.col("__group_max_label") > 0)
            .drop("__group_max_label")
        )
    return lf


def shard_ids_with_parquet(root: Path) -> set[int]:
    if not root.exists():
        return set()
    ids: set[int] = set()
    for p in root.glob("user_shard=*"):
        if not p.is_dir() or not path_has_parquet(p):
            continue
        try:
            ids.add(int(p.name.split("=", 1)[1]))
        except ValueError:
            continue
    return ids


def dataset_label_stats(dataset_dir: Path) -> dict[str, int]:
    if not path_has_parquet(dataset_dir):
        return {"rows": 0, "positives": 0}
    df = collect_streaming(
        scan_parquet(dataset_dir, hive_partitioning=True)
        .select(
            pl.len().alias("rows"),
            pl.col("label").fill_null(0).sum().alias("positives"),
        )
    )
    row = df.row(0, named=True)
    return {"rows": int(row["rows"] or 0), "positives": int(row["positives"] or 0)}


def dataset_is_reusable(split_name: str, candidates_dir: Path, out_dir: Path, *, has_labels: bool) -> bool:
    if RCFG.rebuild_datasets or not path_has_parquet(out_dir):
        return False

    expected_shards = shard_ids_with_parquet(candidates_dir)
    actual_shards = shard_ids_with_parquet(out_dir)
    missing = sorted(expected_shards - actual_shards)

    if missing:
        preview = ", ".join(map(str, missing[:12]))
        suffix = "..." if len(missing) > 12 else ""
        print(
            f"{split_name}: existing dataset is partial "
            f"({len(actual_shards)}/{len(expected_shards)} shards, missing: {preview}{suffix}) -> rebuild"
        )
        return False

    if has_labels:
        stats = dataset_label_stats(out_dir)
        if stats["positives"] <= 0:
            print(
                f"{split_name}: existing dataset has no positives "
                f"(rows={stats['rows']:,}, positives={stats['positives']:,}) -> rebuild"
            )
            return False

    print(f"{split_name}: reusable dataset exists -> {rel(out_dir)}")
    return True


def build_ranker_dataset(
    split_name: str,
    *,
    candidates_dir: Path,
    feature_paths: dict[str, Path],
    target_start: date,
    target_end: date,
    out_dir: Path,
    has_labels: bool,
    downsample_for_fit: bool,
    user_ids: Sequence[int] | None = None,
) -> Path:
    if dataset_is_reusable(split_name, candidates_dir, out_dir, has_labels=has_labels):
        return out_dir

    clean_output_dir(out_dir)
    labels_path = build_labels_for_target(split_name, target_start, target_end, has_labels=has_labels)
    labels_lf = scan_parquet(labels_path, hive_partitioning=False)

    total_rows = 0
    total_pos = 0
    total_users = 0

    _shard_ids = list(RCFG.smoke_shards) if RCFG.smoke_shards is not None else list(range(int(RCFG.n_shards)))
    for shard_id in tqdm(_shard_ids, desc=f"Build dataset {split_name}"):
        shard_id = int(shard_id)
        cand_lf = scan_candidate_shard_or_none(candidates_dir, shard_id)
        if cand_lf is None:
            continue

        lf = ensure_candidate_schema(cand_lf)
        lf = maybe_filter_users(lf, user_ids)
        lf = add_candidate_features(lf)
        lf = add_repeat_derived_features(lf)

        lf = lf.join(labels_lf, on=["user_id", "item_id"], how="left")
        lf = lf.with_columns(
            pl.col("label").fill_null(0).cast(pl.UInt8),
            pl.col("target_order_cnt").fill_null(0).cast(pl.UInt16),
            pl.col("target_delivered_order_cnt").fill_null(0).cast(pl.UInt16),
            pl.col("target_canceled_order_cnt").fill_null(0).cast(pl.UInt16),
        )

        lf = join_feature_paths(lf, feature_paths, shard_id=shard_id)
        lf = maybe_downsample_fit_rows(lf, downsample=downsample_for_fit)

        df = collect_streaming(lf)
        if df.is_empty():
            continue

        total_rows += df.height
        total_pos += int(df["label"].sum()) if "label" in df.columns else 0
        total_users += df["user_id"].n_unique()

        write_single_shard(df, out_dir, shard_id, "dataset.parquet")
        del df
        gc.collect()

    print(
        f"{split_name}: rows={total_rows:,}, positives={total_pos:,}, "
        f"users_by_shard_sum={total_users:,} -> {rel(out_dir)}"
    )
    return out_dir

## Построение Feature Paths и Датасетов

Строит feature store для каждого split, затем собирает ranker dataset
(candidates + features + labels) по шардам.
При `rebuild_datasets = False` использует уже построенные датасеты.


In [39]:
train_feature_paths = build_feature_store_for_split(
    "train",
    RCFG.train_context_start,
    RCFG.train_context_end,
    train_candidates_dir,
)
val_feature_paths = build_feature_store_for_split(
    "val",
    RCFG.val_context_start,
    RCFG.val_context_end,
    val_candidates_dir,
)
if RCFG.enable_test and test_candidates_dir is not None:
    test_feature_paths = build_feature_store_for_split(
        "test",
        RCFG.full_context_start,
        RCFG.full_context_end,
        test_candidates_dir,
    )
else:
    test_feature_paths = {}

print("train feature levels:", sorted(train_feature_paths))
print("val feature levels:  ", sorted(val_feature_paths))
print("test feature levels: ", sorted(test_feature_paths))

train_fit_dir = build_ranker_dataset(
    "train_fit",
    candidates_dir=train_candidates_dir,
    feature_paths=train_feature_paths,
    target_start=RCFG.train_target_start,
    target_end=RCFG.train_target_end,
    out_dir=DATASET_DIR / "train_fit",
    has_labels=True,
    downsample_for_fit=True,
    user_ids=train_users if RCFG.sample_train_users is not None else None,
)

val_fit_dir = build_ranker_dataset(
    "val_fit",
    candidates_dir=val_candidates_dir,
    feature_paths=val_feature_paths,
    target_start=RCFG.val_target_start,
    target_end=RCFG.val_target_end,
    out_dir=DATASET_DIR / "val_fit",
    has_labels=True,
    downsample_for_fit=True,
    user_ids=val_users if RCFG.sample_val_users is not None else None,
)

val_eval_dir = build_ranker_dataset(
    "val_eval",
    candidates_dir=val_candidates_dir,
    feature_paths=val_feature_paths,
    target_start=RCFG.val_target_start,
    target_end=RCFG.val_target_end,
    out_dir=DATASET_DIR / "val_eval",
    has_labels=True,
    downsample_for_fit=False,
    user_ids=val_users if RCFG.sample_val_users is not None else None,
)

if RCFG.enable_test and test_candidates_dir is not None:
    test_dataset_dir = build_ranker_dataset(
        "test",
        candidates_dir=test_candidates_dir,
        feature_paths=test_feature_paths,
        target_start=RCFG.test_target_start,
        target_end=RCFG.test_target_end,
        out_dir=DATASET_DIR / "test",
        has_labels=RCFG.test_has_labels,
        downsample_for_fit=False,
        user_ids=test_users if RCFG.sample_test_users is not None else None,
    )
else:
    test_dataset_dir = None

train feature levels: ['category', 'item', 'user', 'user_category', 'user_item']
val feature levels:   ['category', 'item', 'user', 'user_category', 'user_item']
test feature levels:  ['category', 'item', 'user', 'user_category', 'user_item']
train_fit: reusable dataset exists -> data\processed\reranker\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\dataset\train_fit
val_fit: reusable dataset exists -> data\processed\reranker\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\dataset\val_fit
val_eval: reusable dataset exists -> data\processed\reranker\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\dataset\val_eval
test: reusable dataset exists -> data\processed\reranker\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\dataset\test


In [40]:
def dataset_overview(dataset_dir: Path) -> dict:
    lf = scan_parquet(dataset_dir, hive_partitioning=True)
    df = collect_streaming(
        lf.select(
            pl.len().alias("rows"),
            pl.col("user_id").n_unique().alias("users"),
            pl.col("item_id").n_unique().alias("items"),
            pl.col("label").sum().alias("positives"),
        )
    )
    row = df.row(0, named=True)
    row["positive_rate"] = float(row["positives"] / row["rows"]) if row["rows"] else 0.0
    return row


_overview_rows = [
    {"split": "train_fit", **dataset_overview(train_fit_dir)},
    {"split": "val_fit", **dataset_overview(val_fit_dir)},
    {"split": "val_eval", **dataset_overview(val_eval_dir)},
]
if RCFG.enable_test and test_dataset_dir is not None:
    _overview_rows.append({"split": "test", **dataset_overview(test_dataset_dir)})
overview = pd.DataFrame(_overview_rows)
overview_path = REPORTS_DIR / "final_reranker_dataset_overview.parquet"
overview.to_parquet(overview_path, index=False)
display(overview)
print("saved:", rel(overview_path))

,split,rows,users,items,positives,positive_rate
0,train_fit,51362434,165313,3489684,317103,0.006174
1,val_fit,48778601,153240,3490931,288578,0.005916
2,val_eval,484823527,359928,3966854,288578,0.000595
3,test,469913720,346759,3880277,272467,0.000580


saved: artifacts\reports\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\final_reranker_dataset_overview.parquet


In [41]:
def available_score_cols(dataset_dir: Path) -> list[str]:
    cols = set(lazy_columns(scan_parquet(dataset_dir, hive_partitioning=True)))
    candidates = [
        ("candidate_rank", False),
        ("top_popular_rank", False),
        ("als_rank", False),
        ("min_source_rank", False),
        ("max_source_score", True),
        ("dl_dssm_final_rank", False),
        ("dl_bert4rec_small_rank", False),
        ("dl_sasrec_v3_rank", False),
        ("dl_dssm_final_score", True),
        ("dl_bert4rec_small_score", True),
        ("dl_sasrec_v3_score", True),
        ("repeat_history_rank",      False),
        ("repeat_recency_rank",       False),
        ("repeat_frequency_rank",     False),
        ("repeat_cart_orders_rank",   False),
        ("repeat_history_score",      True),
        ("repeat_recency_score",      True),
        ("repeat_frequency_score",    True),
        ("repeat_cart_orders_score",  True),
    ]
    return [(c, higher) for c, higher in candidates if c in cols]


baseline_rows = []
for score_col, higher_is_better in available_score_cols(val_eval_dir):
    baseline_rows.append(
        evaluate_dataset_score_col(
            val_eval_dir,
            val_gt,
            score_col=score_col,
            label=score_col,
            higher_is_better=higher_is_better,
        )
    )

baseline_metrics = pd.concat(baseline_rows, ignore_index=True) if baseline_rows else pd.DataFrame()
baseline_metrics_path = REPORTS_DIR / "final_reranker_baseline_val_metrics.parquet"
baseline_metrics.to_parquet(baseline_metrics_path, index=False)
display(baseline_metrics.sort_values(["model", "k"]))
print("saved:", rel(baseline_metrics_path))

Evaluate candidate_rank:   0%|          | 0/512 [00:00<?, ?it/s]

Evaluate top_popular_rank:   0%|          | 0/512 [00:00<?, ?it/s]

Evaluate als_rank:   0%|          | 0/512 [00:00<?, ?it/s]

Evaluate min_source_rank:   0%|          | 0/512 [00:00<?, ?it/s]

Evaluate max_source_score:   0%|          | 0/512 [00:00<?, ?it/s]

Evaluate dl_dssm_final_rank:   0%|          | 0/512 [00:00<?, ?it/s]

... (11 outputs left out)

Evaluate repeat_frequency_score:   0%|          | 0/512 [00:00<?, ?it/s]

Evaluate repeat_cart_orders_score:   0%|          | 0/512 [00:00<?, ?it/s]

,model,k,users_total,users_with_predictions,user_coverage,hit_rate,recall,precision,ndcg,map,mrr
8,als_rank,20,359928,359928,1.0,0.011372,0.003084,0.000580,0.001543,0.000571,0.002043
9,als_rank,50,359928,359928,1.0,0.026675,0.007427,0.000558,0.002711,0.000705,0.002510
10,als_rank,100,359928,359928,1.0,0.052674,0.015342,0.000575,0.004486,0.000820,0.002869
11,als_rank,200,359928,359928,1.0,0.097942,0.030584,0.000573,0.007434,0.000935,0.003182
0,candidate_rank,20,359928,359928,1.0,0.069908,0.024861,0.003886,0.011793,0.004723,0.013568
...,...,...,...,...,...,...,...,...,...,...,...
67,repeat_recency_score,200,359928,359928,1.0,0.038772,0.010968,0.000209,0.002648,0.000310,0.001168
4,top_popular_rank,20,359928,359928,1.0,0.012408,0.003455,0.000631,0.001703,0.000634,0.002206
5,top_popular_rank,50,359928,359928,1.0,0.030503,0.008690,0.000639,0.003105,0.000794,0.002751
6,top_popular_rank,100,359928,359928,1.0,0.058481,0.017450,0.000640,0.005056,0.000921,0.003137


saved: artifacts\reports\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\final_reranker_baseline_val_metrics.parquet


## Feature Selection

Из 607 доступных числовых признаков выбирается `model_max_numeric_features = 240`.
Все repeat-признаки форсированно включаются через `_REPEAT_FORCE_TOKENS` до применения cap.
Признаки `user / item / category` сортируются по приоритету метрики и временного окна.


In [42]:
def list_feature_cols(dataset_dir: Path) -> list[str]:
    lf = scan_parquet(dataset_dir, hive_partitioning=True)
    schema = lf.collect_schema()
    return sorted([
        col for col, dtype in zip(schema.names(), schema.dtypes())
        if col.startswith("f_") and dtype in _NUMERIC_DTYPES
    ])


def list_existing_cols(dataset_dir: Path) -> list[str]:
    lf = scan_parquet(dataset_dir, hive_partitioning=True)
    return list(lf.collect_schema().names())


def clean_feature_expr(col: str) -> pl.Expr:
    x = pl.col(col).cast(pl.Float32, strict=False)
    return (
        pl.when(x.is_null() | x.is_nan() | x.is_infinite())
        .then(pl.lit(0.0))
        .otherwise(x)
        .cast(pl.Float32)
        .alias(col)
    )


def clean_categorical_expr(col: str) -> pl.Expr:
    return pl.col(col).fill_null(-1).cast(pl.Int64, strict=False).fill_null(-1).alias(col)


def _feature_window(col: str) -> int:
    m = re.search(r"_(\d+)d(?:$|_)", col)
    return int(m.group(1)) if m else 0


def _feature_metric_priority(col: str) -> int:
    # Lower is better. Keep source/rank/click/order/candidate signals first.
    tokens = [
        ("candidate__", 0),
        ("sources_count", 1),
        ("_rank", 2),
        ("_score", 3),
        ("has_orders", 4),
        ("cnt_orders", 5),
        ("cnt_delivered_orders", 6),
        ("has_events", 7),
        ("cnt_to_cart", 8),
        ("cnt_favorite", 9),
        ("cnt_page_view", 10),
        ("days_since_last_order", 11),
        ("days_since_last_event", 12),
        ("delivered_share", 13),
        ("to_cart_per_page_view", 14),
        ("favorite_per_page_view", 15),
        ("description_per_page_view", 16),
        ("review_per_page_view", 17),
        ("cnt_events", 18),
    ]
    for token, pr in tokens:
        if token in col:
            return pr
    return 50


def _feature_level_priority(col: str) -> int:
    # Lower is better. Candidate/source features are mandatory-like; user_item is useful but can explode feature count.
    if "__candidate__" in col:
        return 0
    if "__user__" in col:
        return 10
    if "__item__" in col:
        return 20
    if "__user_item__" in col:
        return 30
    if "__user_category__" in col:
        return 40
    if "__category__" in col:
        return 50
    return 60


def feature_priority_key(col: str) -> tuple[int, int, int, str]:
    window = _feature_window(col)
    # prefer configured/recent windows; unknown window goes after known windows
    if window in RCFG.model_keep_feature_windows:
        try:
            window_pr = list(RCFG.model_keep_feature_windows).index(window)
        except ValueError:
            window_pr = 99
    else:
        window_pr = 99
    return (_feature_level_priority(col), _feature_metric_priority(col), window_pr, col)


def select_memory_safe_numeric_features(all_numeric_features: list[str]) -> list[str]:
    cols = [c for c in all_numeric_features if (_feature_window(c) == 0 or _feature_window(c) in RCFG.model_keep_feature_windows)]

    # Candidate features encode source ranks/scores and should be kept even under aggressive caps.
    candidate_cols = sorted([c for c in cols if "__candidate__" in c], key=feature_priority_key)
    other_cols = sorted([c for c in cols if "__candidate__" not in c], key=feature_priority_key)

    _REPEAT_FORCE_TOKENS = (
        "repeat", "rank_gap", "rank_delta", "rank_is_top",
        "n_source_signals", "is_repeat_only",
    )
    forced_cols = [c for c in candidate_cols
                   if any(t in c for t in _REPEAT_FORCE_TOKENS)]
    _forced_set = set(forced_cols)
    rest_cols = [c for c in candidate_cols if c not in _forced_set] + other_cols
    if RCFG.model_max_numeric_features is None:
        selected = forced_cols + rest_cols
    else:
        max_n = int(RCFG.model_max_numeric_features)
        selected = forced_cols + rest_cols[:max(0, max_n - len(forced_cols))]

    missing_candidate = [c for c in candidate_cols if c not in selected]
    if missing_candidate:
        # If candidate feature count alone exceeds cap, we still respect cap but warn.
        print(f"WARNING: candidate feature count exceeds cap; dropped {len(missing_candidate)} candidate features")

    return selected


all_numeric_feature_cols = sorted(set(list_feature_cols(train_fit_dir)) & set(list_feature_cols(val_fit_dir)))
numeric_feature_cols = select_memory_safe_numeric_features(all_numeric_feature_cols)
all_train_cols = set(list_existing_cols(train_fit_dir))

base_catboost_cat_cols = ["catalogid"]
if RCFG.catboost_use_id_features:
    base_catboost_cat_cols = ["user_id", "item_id", "catalogid"]
catboost_cat_cols = [c for c in base_catboost_cat_cols if c in all_train_cols or c in {"user_id", "item_id"}]
catboost_feature_cols = catboost_cat_cols + numeric_feature_cols

print(f"numeric features available: {len(all_numeric_feature_cols):,}")
print(f"numeric features selected for model: {len(numeric_feature_cols):,}")
print(f"CatBoost categorical features: {catboost_cat_cols}")
print("Selected feature preview:")
for c in numeric_feature_cols[:80]:
    print(" ", c)

feature_list_path = MODELS_DIR / "feature_cols.json"
feature_manifest = {
    "all_numeric_feature_cols_count": len(all_numeric_feature_cols),
    "numeric_feature_cols": numeric_feature_cols,
    "catboost_cat_cols": catboost_cat_cols,
    "catboost_feature_cols": catboost_feature_cols,
    "selection_config": {
        "model_max_numeric_features": RCFG.model_max_numeric_features,
        "model_keep_feature_windows": list(RCFG.model_keep_feature_windows),
        "model_train_max_rows": RCFG.model_train_max_rows,
        "model_val_max_rows": RCFG.model_val_max_rows,
    },
}
feature_list_path.write_text(json.dumps(feature_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", rel(feature_list_path))


numeric features available: 607
numeric features selected for model: 240
CatBoost categorical features: ['catalogid']
Selected feature preview:
  f_num__candidate__is_repeat_history_candidate
  f_num__candidate__is_repeat_only_candidate
  f_num__candidate__is_top10_repeat
  f_num__candidate__is_top50_repeat
  f_num__candidate__n_source_signals
  f_num__candidate__num_repeat_signals
  f_num__candidate__repeat_cart_orders_rank
  f_num__candidate__repeat_cart_orders_rank_missing
  f_num__candidate__repeat_cart_orders_rank_rr
  f_num__candidate__repeat_cart_orders_score
  f_num__candidate__repeat_cart_orders_score_log1p_nonnegative
  f_num__candidate__repeat_cart_orders_score_missing
  f_num__candidate__repeat_frequency_rank
  f_num__candidate__repeat_frequency_rank_missing
  f_num__candidate__repeat_frequency_rank_rr
  f_num__candidate__repeat_frequency_score
  f_num__candidate__repeat_frequency_score_log1p_nonnegative
  f_num__candidate__repeat_frequency_score_missing
  f_num__candidate_

## Ablation Feature Sets

Определяет наборы A / B / C / D для анализа вклада repeat-признаков.
A / B / C обучаются с фиксированными параметрами (n_estimators=250), D — через Optuna (6 проб).


In [43]:
def ablation_feature_sets(all_feature_cols: list[str]) -> dict[str, list[str]]:
    """Partition features into ablation sets A / B / C / D.

    A  no repeat features (baseline)
    B  A + repeat raw (auto-prefixed *_rank/*_score + bool flags)
    C  B + rank-gap interaction features
    D  full v12 (C + position priors + source interaction) = all features
    """
    _REPEAT_RAW = {
        "repeat_history", "repeat_recency", "repeat_frequency", "repeat_cart_orders",
        "repeat_in_recency", "repeat_in_frequency", "repeat_in_cart_orders",
        "is_repeat_history_candidate",
    }
    _RANK_GAP   = {"rank_gap", "rank_delta_to_best", "rank_is_top_source"}
    _DERIVED    = {
        "num_repeat_signals", "repeat_rank_norm", "repeat_rank_log",
        "is_top10_repeat", "is_top50_repeat",
    }
    _INTERACT   = {"n_source_signals", "is_repeat_only_candidate"}

    def _match(col: str, markers: set) -> bool:
        return any(m in col for m in markers)

    repeat_raw  = [c for c in all_feature_cols if _match(c, _REPEAT_RAW)
                   and not _match(c, _RANK_GAP | _DERIVED | _INTERACT)]
    rank_gap    = [c for c in all_feature_cols if _match(c, _RANK_GAP)]
    derived     = [c for c in all_feature_cols if _match(c, _DERIVED)]
    interaction = [c for c in all_feature_cols if _match(c, _INTERACT)]

    _all_repeat = set(repeat_raw + rank_gap + derived + interaction)
    baseline    = [c for c in all_feature_cols if c not in _all_repeat]

    sets = {
        "A": baseline,
        "B": sorted(set(baseline + repeat_raw)),
        "C": sorted(set(baseline + repeat_raw + rank_gap)),
        "D": all_feature_cols,
    }
    print("Ablation feature sets:")
    for _k, _v in sets.items():
        _extra = len(_v) - len(baseline)
        print(f"  {_k}: {len(_v):3d} features  (+{_extra:3d} repeat)")
    return sets


ablation_sets = ablation_feature_sets(numeric_feature_cols)


Ablation feature sets:
  A: 199 features  (+  0 repeat)
  B: 227 features  (+ 28 repeat)
  C: 233 features  (+ 34 repeat)
  D: 240 features  (+ 41 repeat)


In [44]:
def dataset_basic_stats(dataset_dir: Path) -> dict[str, int]:
    lf = scan_parquet(dataset_dir, hive_partitioning=True)
    df = collect_streaming(
        lf.select(
            pl.len().alias("rows"),
            pl.col("label").fill_null(0).sum().alias("positives"),
            pl.col("user_id").n_unique().alias("users"),
        )
    )
    row = df.row(0, named=True)
    return {k: int(row[k] or 0) for k in ["rows", "positives", "users"]}


def add_fit_row_sampling(
    lf: pl.LazyFrame,
    *,
    total_rows: int,
    total_positives: int,
    max_rows: int | None,
    split_name: str,
) -> pl.LazyFrame:
    """Sample negatives before pandas conversion while always keeping positives by default."""
    if max_rows is None or total_rows <= int(max_rows):
        print(
            f"{split_name}: no model-row sampling | "
            f"total_rows={total_rows:,}, positives={total_positives:,}, "
            f"max_rows={max_rows}, expected≈{total_rows:,}"
        )
        return lf

    max_rows = int(max_rows)
    pos = int(total_positives)
    neg = max(total_rows - pos, 0)

    if RCFG.model_force_all_positives:
        neg_budget = max(max_rows - pos, 1)
        if neg <= 0:
            print(f"{split_name}: only positives; no negative sampling")
            return lf.filter(pl.col("label") > 0)
        threshold = max(1, min(RCFG.model_fit_hash_mod, int(RCFG.model_fit_hash_mod * (neg_budget / neg))))
        keep_expr = (
            (pl.col("label") > 0)
            | ((pl.struct(["user_id", "item_id"]).hash(seed=RCFG.random_state) % int(RCFG.model_fit_hash_mod)) < threshold)
        )
        expected = pos + int(neg * threshold / RCFG.model_fit_hash_mod)
        print(
            f"{split_name}: model-row sampling rows={total_rows:,}, positives={pos:,}, "
            f"max_rows={max_rows:,}, neg_threshold={threshold}/{RCFG.model_fit_hash_mod}, expected≈{expected:,}"
        )
        return lf.filter(keep_expr)

    threshold = max(1, min(RCFG.model_fit_hash_mod, int(RCFG.model_fit_hash_mod * (max_rows / total_rows))))
    print(f"{split_name}: row sampling threshold={threshold}/{RCFG.model_fit_hash_mod}")
    return lf.filter((pl.struct(["user_id", "item_id"]).hash(seed=RCFG.random_state) % int(RCFG.model_fit_hash_mod)) < threshold)


def collect_model_pdf_and_group(
    dataset_dir: Path,
    numeric_features: list[str],
    categorical_features: list[str] | None = None,
    *,
    split_name: str,
    max_rows: int | None,
) -> tuple[pd.DataFrame, np.ndarray]:
    """Collect fit data in a memory-safer way.

    Key points:
    - filtering groups without positives is done in Polars, before pandas;
    - sorting by user_id/item_id is done in Polars, before pandas;
    - group sizes are computed in Polars;
    - pandas no longer does loc[mask].copy() over a 100+ feature block.
    """
    categorical_features = categorical_features or []
    lf = scan_parquet(dataset_dir, hive_partitioning=True)
    available = set(lazy_columns(lf))

    numeric_features = [c for c in numeric_features if c in available]
    select_cols = ["user_id", "item_id", "label"]

    for col in categorical_features:
        if col not in select_cols:
            if col in available or col in {"user_id", "item_id"}:
                select_cols.append(col)

    for col in numeric_features:
        if col not in select_cols:
            select_cols.append(col)

    stats = dataset_basic_stats(dataset_dir)
    lf = add_fit_row_sampling(
        lf,
        total_rows=stats["rows"],
        total_positives=stats["positives"],
        max_rows=max_rows,
        split_name=split_name,
    )

    missing_exprs = []
    if "catalogid" in categorical_features and "catalogid" not in available:
        missing_exprs.append(pl.lit(-1, dtype=pl.Int64).alias("catalogid"))
    if missing_exprs:
        lf = lf.with_columns(missing_exprs)

    clean_exprs = [clean_feature_expr(c) for c in numeric_features]
    clean_exprs += [clean_categorical_expr(c) for c in categorical_features if c in select_cols]

    lf = lf.with_columns(clean_exprs)

    if RCFG.filter_fit_groups_without_positive:
        before = stats["rows"]
        lf = (
            lf.with_columns(
                pl.col("label").fill_null(0).max().over("user_id").alias("_user_has_positive")
            )
            .filter(pl.col("_user_has_positive") > 0)
            .drop("_user_has_positive")
        )
        print(f"{split_name}: filter groups without positives is pushed down to Polars")

    # LGBM group arrays require contiguous user groups.
    lf = lf.select(select_cols).sort(["user_id", "item_id"])

    df = collect_streaming(lf)
    actual_rows = df.height
    approx_numeric_gb = actual_rows * max(len(numeric_features), 1) * 4 / (1024 ** 3)

    group = (
        df.group_by("user_id", maintain_order=True)
        .len()
        .get_column("len")
        .to_numpy()
        .astype(np.int32, copy=False)
    )

    print(
        f"{split_name}: collected sorted polars rows={actual_rows:,}, users={len(group):,}, "
        f"cols={len(select_cols):,}, numeric_block≈{approx_numeric_gb:.2f} GB before pandas overhead"
    )

    pdf = df.to_pandas()
    del df
    gc.collect()

    pdf["user_id"] = pdf["user_id"].astype(np.int64, copy=False)
    pdf["item_id"] = pdf["item_id"].astype(np.int64, copy=False)
    pdf["label"] = pdf["label"].astype(np.int32, copy=False)

    for col in numeric_features:
        if col in pdf.columns:
            pdf[col] = pdf[col].astype(np.float32, copy=False)

    for col in categorical_features:
        if col in pdf.columns:
            pdf[col] = pdf[col].fillna(-1).astype(np.int64, copy=False)

    print(
        f"{split_name}: pandas ready shape={pdf.shape}, groups={len(group):,}, "
        f"positives={int(pdf['label'].sum()):,}"
    )
    return pdf, group


def prepare_grouped_pdf(pdf: pd.DataFrame, *, split_name: str) -> tuple[pd.DataFrame, np.ndarray]:
    """Fallback only. The main path now uses collect_model_pdf_and_group()."""
    before_rows = len(pdf)
    before_users = pdf["user_id"].nunique()

    if RCFG.filter_fit_groups_without_positive:
        keep = pdf.groupby("user_id")["label"].transform("max") > 0
        # Avoid deep copy here; if this fallback is used, sorting below will still allocate.
        pdf = pdf.loc[keep]

    pdf = pdf.sort_values(["user_id", "item_id"], kind="mergesort", ignore_index=True)
    group = pdf.groupby("user_id", sort=False).size().to_numpy(dtype=np.int32)

    print(
        f"{split_name}: grouped rows {before_rows:,}->{len(pdf):,}, "
        f"users {before_users:,}->{len(group):,}, positives={int(pdf['label'].sum()):,}"
    )
    return pdf, group


train_pdf, train_group = collect_model_pdf_and_group(
    train_fit_dir,
    numeric_feature_cols,
    catboost_cat_cols,
    split_name="train_fit",
    max_rows=RCFG.model_train_max_rows,
)
val_pdf, val_group = collect_model_pdf_and_group(
    val_fit_dir,
    numeric_feature_cols,
    catboost_cat_cols,
    split_name="val_fit",
    max_rows=RCFG.model_val_max_rows,
)

print("train_pdf:", train_pdf.shape, "groups:", len(train_group), "positives:", int(train_pdf["label"].sum()))
print("val_pdf:  ", val_pdf.shape, "groups:", len(val_group), "positives:", int(val_pdf["label"].sum()))


train_fit: model-row sampling rows=51,362,434, positives=317,103, max_rows=6,000,000, neg_threshold=111330/1000003, expected≈5,999,962
train_fit: filter groups without positives is pushed down to Polars
train_fit: collected sorted polars rows=5,994,984, users=165,313, cols=244, numeric_block≈5.36 GB before pandas overhead


C:\Users\denis\AppData\Local\Temp\ipykernel_26452\1227134254.py:144: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  pdf["user_id"] = pdf["user_id"].astype(np.int64, copy=False)
C:\Users\denis\AppData\Local\Temp\ipykernel_26452\1227134254.py:145: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  pdf["item_id"] = pdf["item_id"].astype(np.int64, copy=False)
C:\Users\denis\AppData\Local\Temp\ipykernel_26452\1227134254.py:146: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy me

train_fit: pandas ready shape=(5994984, 244), groups=165,313, positives=317,103
val_fit: model-row sampling rows=48,778,601, positives=288,578, max_rows=2,000,000, neg_threshold=35294/1000003, expected≈1,999,979
val_fit: filter groups without positives is pushed down to Polars
val_fit: collected sorted polars rows=2,000,860, users=153,240, cols=244, numeric_block≈1.79 GB before pandas overhead
val_fit: pandas ready shape=(2000860, 244), groups=153,240, positives=288,578
train_pdf: (5994984, 244) groups: 165313 positives: 317103
val_pdf:   (2000860, 244) groups: 153240 positives: 288578


C:\Users\denis\AppData\Local\Temp\ipykernel_26452\1227134254.py:144: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  pdf["user_id"] = pdf["user_id"].astype(np.int64, copy=False)
C:\Users\denis\AppData\Local\Temp\ipykernel_26452\1227134254.py:145: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  pdf["item_id"] = pdf["item_id"].astype(np.int64, copy=False)
C:\Users\denis\AppData\Local\Temp\ipykernel_26452\1227134254.py:146: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy me

## Обучение Модели

Ablation A / B / C: одиночный fit с фиксированными параметрами (без Optuna).
Ablation D: Optuna, 6 проб, полный набор признаков `numeric_feature_cols`.
Финальные кандидаты из top-`lgbm_valeval_top_trials` проб Optuna дополнительно сравниваются на `val_eval`.


In [45]:
def parse_metric_name(metric_name: str) -> tuple[str, int]:
    name, k = metric_name.lower().split("@")
    return name, int(k)


def metric_from_metrics_df(metrics: pd.DataFrame, metric_name: str) -> float:
    name, k = parse_metric_name(metric_name)
    return float(metrics.loc[metrics["k"] == k, name].iloc[0])


def evaluate_scored_pandas(
    pdf: pd.DataFrame,
    ground_truth: dict[int, list[int]],
    *,
    score_col: str = "score",
    label: str = "model",
) -> pd.DataFrame:
    return evaluate_scored_df(pdf[["user_id", "item_id", score_col]], ground_truth, score_col=score_col, label=label)


def make_model_record(
    *,
    name: str,
    model_type: str,
    model,
    features: list[str],
    categorical_features: list[str],
    params: dict,
    best_iteration: int | None,
    tuning_time_sec: float,
    selection_fit_time_sec: float,
    study: optuna.Study | None,
) -> dict:
    return {
        "name": name,
        "model_type": model_type,
        "model": model,
        "features": features,
        "categorical_features": categorical_features,
        "params": params,
        "best_iteration": best_iteration,
        "tuning_time_sec": float(tuning_time_sec),
        "selection_fit_time_sec": float(selection_fit_time_sec),
        "study": study,
    }


def predict_model(record: dict, pdf: pd.DataFrame) -> np.ndarray:
    features = record["features"]
    model = record["model"]

    if record["model_type"] == "lgbm":
        num_iteration = record.get("best_iteration")
        return model.predict(pdf[features], num_iteration=num_iteration)

    if record["model_type"] == "catboost":
        cat_features = [c for c in record["categorical_features"] if c in features]
        pool = Pool(pdf[features], cat_features=cat_features)
        return model.predict(pool)

    raise ValueError(record["model_type"])

def base_lgbm_params() -> dict:
    return {
        "objective": "lambdarank",
        "metric": "ndcg",
        "boosting_type": "gbdt",
        "random_state": RCFG.random_state,
        "n_jobs": -1,
        "verbosity": -1,
        "label_gain": [0, 1],
        "force_col_wise": True,
    }


def _make_sample_weight(labels: "pd.Series") -> "np.ndarray | None":
    pw = getattr(RCFG, "positive_weight", 1.0)
    if pw == 1.0:
        return None
    return np.where(labels.values > 0, float(pw), 1.0).astype(np.float32)


def fit_lgbm_ranker(
    params: dict,
    train_data: pd.DataFrame,
    train_groups: np.ndarray,
    *,
    valid_data: pd.DataFrame | None = None,
    valid_groups: np.ndarray | None = None,
):
    features = numeric_feature_cols
    return fit_lgbm_ranker_with_features(
        features, params, train_data, train_groups,
        valid_data=valid_data, valid_groups=valid_groups,
    )


def fit_lgbm_ranker_with_features(
    features: list,
    params: dict,
    train_data: pd.DataFrame,
    train_groups: np.ndarray,
    *,
    valid_data: pd.DataFrame | None = None,
    valid_groups: np.ndarray | None = None,
):
    model = lgbm.LGBMRanker(**base_lgbm_params(), **params)
    sample_weight = _make_sample_weight(train_data["label"])

    callbacks = []
    eval_set = None
    eval_group = None

    if valid_data is not None and valid_groups is not None:
        eval_set = [(valid_data[features], valid_data["label"])]
        eval_group = [valid_groups]
        callbacks = [
            lgbm.early_stopping(RCFG.early_stopping_rounds, verbose=False),
            lgbm.log_evaluation(period=0),
        ]

    model.fit(
        train_data[features],
        train_data["label"],
        group=train_groups,
        sample_weight=sample_weight,
        eval_set=eval_set,
        eval_group=eval_group,
        eval_at=list(RCFG.metric_ks),
        callbacks=callbacks,
    )
    return model


def _lgbm_params_from_trial(trial: optuna.trial.FrozenTrial, *, use_min_trees: bool = True) -> tuple[dict, int | None]:
    """Build final LGBM params from an Optuna trial.

    The trial objective uses val_fit only as a cheap pre-selection signal.
    Candidate models for the real comparison are refit without val_fit early stopping
    and then compared on full val_eval in the model-comparison cell.
    """
    params = trial.params.copy()
    best_iteration = trial.user_attrs.get("best_iteration")
    if best_iteration is not None and int(best_iteration) > 0:
        n_estimators = int(best_iteration)
        if use_min_trees:
            n_estimators = max(n_estimators, int(getattr(RCFG, "lgbm_min_trees", 200)))
        params["n_estimators"] = n_estimators

    return params, int(best_iteration) if best_iteration is not None else None


def tune_lgbm_ranker() -> list[dict]:
    """Tune LightGBM and return several candidate records.

    Important: Optuna's objective still evaluates on val_fit because it is cheap.
    We do NOT select the final LGBM by that metric. Instead, the top Optuna trials
    are refit as separate candidate records. The later model-comparison cell scores
    all of them on full val_eval and selects the true best by RCFG.optimize_metric.
    """
    if not RCFG.run_lightgbm:
        print("LightGBM disabled by config")
        return []

    opt_metric_name, opt_k = parse_metric_name(RCFG.optimize_metric)

    def objective_lgbm(trial: optuna.Trial) -> float:
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 300, 1200),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.12, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 31, 191),
            "max_depth": trial.suggest_int("max_depth", 3, 14),
            "min_child_samples": trial.suggest_int("min_child_samples", 20, 350),
            "subsample": trial.suggest_float("subsample", 0.65, 1.0),
            "subsample_freq": 1,
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-5, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-5, 10.0, log=True),
            "lambdarank_truncation_level": trial.suggest_int("lambdarank_truncation_level", 50, 200),
        }

        # val_fit is used only for early stopping and cheap Optuna pre-selection.
        model = fit_lgbm_ranker(params, train_pdf, train_group, valid_data=val_pdf, valid_groups=val_group)
        best_iter = getattr(model, "best_iteration_", None)
        if best_iter is None or best_iter <= 0:
            best_iter = params["n_estimators"]

        val_scores = model.predict(val_pdf[numeric_feature_cols], num_iteration=best_iter)
        scored = val_pdf[["user_id", "item_id"]].copy()
        scored["score"] = val_scores
        metrics = evaluate_scored_pandas(scored, val_gt, score_col="score", label="lgbm_trial_val_fit")
        metric_value = float(metrics.loc[metrics["k"] == opt_k, opt_metric_name].iloc[0])

        trial.set_user_attr("best_iteration", int(best_iter))
        trial.set_user_attr("val_fit_metric", metric_value)
        trial.set_user_attr("val_fit_metrics", metrics.to_dict(orient="records"))
        return metric_value

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=RCFG.random_state),
        study_name="final_lgbm_ranker",
    )

    t0 = time.perf_counter()
    study.optimize(
        objective_lgbm,
        n_trials=RCFG.optuna_trials,
        timeout=RCFG.optuna_timeout_sec,
        show_progress_bar=True,
    )
    tuning_time_sec = time.perf_counter() - t0

    trials_path = REPORTS_DIR / "final_reranker_lgbm_optuna_trials.parquet"
    study.trials_dataframe().to_parquet(trials_path, index=False)

    completed_trials = [
        t for t in study.trials
        if t.state == optuna.trial.TrialState.COMPLETE and t.value is not None
    ]
    completed_trials = sorted(completed_trials, key=lambda t: float(t.value), reverse=True)

    top_n = max(1, int(getattr(RCFG, "lgbm_valeval_top_trials", 3)))
    top_trials = completed_trials[:top_n]

    print(f"LightGBM Optuna best val_fit {RCFG.optimize_metric}: {study.best_value:.6f}")
    print(f"Refitting top {len(top_trials)} LGBM trial(s) for full val_eval comparison")
    print("saved:", rel(trials_path))

    records: list[dict] = []
    for rank, trial in enumerate(top_trials, start=1):
        params, trial_best_iteration = _lgbm_params_from_trial(trial, use_min_trees=True)

        # Refit without val_fit eval_set: this avoids selecting a model merely because it overfits the fit-like val sample.
        # The next cell will score this record on full val_eval and select the true best.
        t1 = time.perf_counter()
        model = fit_lgbm_ranker(params, train_pdf, train_group, valid_data=None, valid_groups=None)
        selection_fit_time_sec = time.perf_counter() - t1

        record = make_model_record(
            name=f"lgbm_ranker_t{rank:02d}",
            model_type="lgbm",
            model=model,
            features=numeric_feature_cols,
            categorical_features=[],
            params=params,
            # No early-stopping set was used for this refit; predict with all trees.
            best_iteration=None,
            tuning_time_sec=tuning_time_sec,
            selection_fit_time_sec=selection_fit_time_sec,
            study=study,
        )
        record["source_trial_number"] = int(trial.number)
        record["source_trial_val_fit_metric"] = float(trial.value)
        record["source_trial_best_iteration"] = int(trial_best_iteration) if trial_best_iteration is not None else None

        print(
            f"{record['name']}: source_trial={trial.number}, "
            f"val_fit_metric={float(trial.value):.6f}, "
            f"trial_best_iter={trial_best_iteration}, "
            f"refit_n_estimators={params.get('n_estimators')}"
        )
        records.append(record)

    return records


lgbm_records = tune_lgbm_ranker()


catboost_record = None  # CatBoost disabled in v12

  0%|          | 0/6 [00:00<?, ?it/s]

LightGBM Optuna best val_fit ndcg@100: 0.208407
Refitting top 2 LGBM trial(s) for full val_eval comparison
saved: artifacts\reports\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\final_reranker_lgbm_optuna_trials.parquet
lgbm_ranker_t01: source_trial=5, val_fit_metric=0.208407, trial_best_iter=82, refit_n_estimators=250
lgbm_ranker_t02: source_trial=2, val_fit_metric=0.208216, trial_best_iter=839, refit_n_estimators=839


In [46]:
# ── Ablation A / B / C: single fixed-param fit (no Optuna) ───────────────────
# fit_lgbm_ranker reads `numeric_feature_cols` from outer scope.
# We temporarily rebind it so each ablation uses its own feature set.
import copy as _copy
import time as _time

_abl_fixed_params = {"n_estimators": 250}  # base_lgbm_params() already merged inside fit_lgbm_ranker
_opt_metric_name, _opt_k = parse_metric_name(RCFG.optimize_metric)

ablation_records: dict[str, dict] = {}
for _abl_name in ["A", "B", "C"]:
    _feats = ablation_sets[_abl_name]
    print(f"\n{'='*55}")
    print(f"Ablation {_abl_name}  ({len(_feats)} features)")
    print(f"{'='*55}")

    _t0 = _time.time()
    _model = fit_lgbm_ranker_with_features(
        _feats,
        _copy.deepcopy(_abl_fixed_params),
        train_pdf, train_group,
        valid_data=val_pdf, valid_groups=val_group,
    )

    _best_iter = getattr(_model, "best_iteration_", None) or _abl_fixed_params["n_estimators"]
    _val_scores = _model.predict(val_pdf[_feats], num_iteration=_best_iter)
    _scored = val_pdf[["user_id", "item_id"]].copy()
    _scored["score"] = _val_scores
    _metrics = evaluate_scored_pandas(_scored, val_gt, score_col="score", label=f"abl_{_abl_name}_val_fit")
    _metric_value = float(_metrics.loc[_metrics["k"] == _opt_k, _opt_metric_name].iloc[0])

    _rec = make_model_record(
        name=f"abl_{_abl_name}_lgbm",
        model_type="lgbm",
        model=_model,
        features=_feats,
        categorical_features=catboost_cat_cols,
        params=_copy.deepcopy(_abl_fixed_params),
        best_iteration=_best_iter,
        tuning_time_sec=_time.time() - _t0,
        selection_fit_time_sec=0.0,
        study=None,
    )
    _rec["val_fit_metric"] = _metric_value
    ablation_records[_abl_name] = _rec
    print(f"  val_fit_metric ({RCFG.optimize_metric}) = {_metric_value:.6f}")
    del _model, _scored, _val_scores, _metrics

print("\nAblation A/B/C training complete.")
del _copy, _time



Ablation A  (199 features)
  val_fit_metric (ndcg@100) = 0.191055

Ablation B  (227 features)
  val_fit_metric (ndcg@100) = 0.207544

Ablation C  (233 features)
  val_fit_metric (ndcg@100) = 0.207724

Ablation A/B/C training complete.


## Ablation: Сравнение

Сравнивает наборы A / B / C / D по `val_fit NDCG@100` (proxy-метрика).
При best(A/B/C) > D выводится предупреждение.


In [47]:
# ── Ablation comparison (val_fit NDCG proxy) ─────────────────────────────────
# Best D result from Optuna lgbm_records
# tune_lgbm_ranker stores the metric under "source_trial_val_fit_metric"; normalize to "val_fit_metric"
if lgbm_records:
    _d_rec = dict(lgbm_records[0])
    _lgbm_d_metric = _d_rec.get("source_trial_val_fit_metric", _d_rec.get("val_fit_metric", float("nan")))
    _d_rec["val_fit_metric"] = _lgbm_d_metric
    ablation_records["D"] = _d_rec
else:
    _lgbm_d_metric = float("nan")
    ablation_records["D"] = {}

print("\n" + "=" * 60)
print("ABLATION SUMMARY  (val_fit NDCG@100 proxy)")
print("=" * 60)
print(f"{'Set':>4}  {'Features':>8}  {'val_fit_NDCG@100':>18}  {'vs D':>8}")
print("-" * 45)
_d_metric = ablation_records.get("D", {}).get("val_fit_metric", float("nan"))
for _k in ["A", "B", "C", "D"]:
    _rec = ablation_records.get(_k, {})
    _m   = _rec.get("val_fit_metric", float("nan"))
    _n   = len(ablation_sets.get(_k, []))
    _delta = _m - _d_metric if _k != "D" else 0.0
    _sign  = "=" if _k == "D" else f"{_delta:+.6f}"
    print(f"  {_k:>2}  {_n:>8}  {_m:>18.6f}  {_sign:>8}")
print("=" * 60)
print()
print("Repeat feature uplift (val_fit NDCG@100 delta vs A):")
_a_m = ablation_records.get("A", {}).get("val_fit_metric", float("nan"))
for _k in ["B", "C", "D"]:
    _rec = ablation_records.get(_k, {})
    _m   = _rec.get("val_fit_metric", float("nan"))
    print(f"  +{_k.replace('A','')}  Δ = {_m - _a_m:+.6f}")

import pandas as _pd
_abl_rows = []
for _k in ["A", "B", "C", "D"]:
    _rec = ablation_records.get(_k, {})
    _abl_rows.append({
        "set": _k,
        "n_features": len(ablation_sets.get(_k, [])),
        "val_fit_ndcg100": _rec.get("val_fit_metric", float("nan")),
    })
_abl_df = _pd.DataFrame(_abl_rows)
_abl_path = REPORTS_DIR / "v12_ablation_results.parquet"
_abl_df.to_parquet(_abl_path, index=False)
print(f"\nsaved: {rel(_abl_path)}")
display(_abl_df)

_best_abc_k = max(["A", "B", "C"], key=lambda _k: ablation_records.get(_k, {}).get("val_fit_metric", float("-inf")))
_best_abc_m = ablation_records.get(_best_abc_k, {}).get("val_fit_metric", float("nan"))
_d_m        = ablation_records.get("D", {}).get("val_fit_metric", float("nan"))
if _best_abc_m > _d_m:
    print(f"\nWARNING: best ablation set {_best_abc_k} ({_best_abc_m:.6f}) beats D ({_d_m:.6f}) "
          f"on val_fit; consider using {_best_abc_k} feature set.")
del _pd



ABLATION SUMMARY  (val_fit NDCG@100 proxy)
 Set  Features    val_fit_NDCG@100      vs D
---------------------------------------------
   A       199            0.191055  -0.017352
   B       227            0.207544  -0.000863
   C       233            0.207724  -0.000683
   D       240            0.208407         =

Repeat feature uplift (val_fit NDCG@100 delta vs A):
  +B  Δ = +0.016489
  +C  Δ = +0.016670
  +D  Δ = +0.017352

saved: artifacts\reports\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\v12_ablation_results.parquet


,set,n_features,val_fit_ndcg100
0,A,199,0.191055
1,B,227,0.207544
2,C,233,0.207724
3,D,240,0.208407


In [48]:
def save_model_record(record: dict, suffix: str = "selection") -> dict[str, str]:
    name = record["name"]
    outputs: dict[str, str] = {}

    if record["model_type"] == "lgbm":
        joblib_path = MODELS_DIR / f"{name}_{suffix}.joblib"
        txt_path = MODELS_DIR / f"{name}_{suffix}.txt"
        joblib.dump(record["model"], joblib_path)
        record["model"].booster_.save_model(str(txt_path))
        outputs["joblib"] = rel(joblib_path)
        outputs["txt"] = rel(txt_path)

    elif record["model_type"] == "catboost":
        cbm_path = MODELS_DIR / f"{name}_{suffix}.cbm"
        joblib_path = MODELS_DIR / f"{name}_{suffix}.joblib"
        record["model"].save_model(str(cbm_path))
        joblib.dump(record["model"], joblib_path)
        outputs["cbm"] = rel(cbm_path)
        outputs["joblib"] = rel(joblib_path)
    else:
        raise ValueError(record["model_type"])
    return outputs


def score_dataset_dir_to_dir(record: dict, dataset_dir: Path, out_dir: Path, *, k: int = RCFG.final_k) -> tuple[Path, float]:
    clean_output_dir(out_dir)

    features = record["features"]
    categorical_features = set(record["categorical_features"])
    required_cols = [
        "user_id", "item_id", "label", "candidate_rank",
        "top_popular_rank", "als_rank", "min_source_rank", "max_source_score",
    ]
    select_cols = list(dict.fromkeys(required_cols + features))

    rows_written = 0
    t0 = time.perf_counter()
    old_champion_lf = old_champion_rank_lazy_for_dataset(dataset_dir)
    if old_champion_lf is None:
        print(f"old champion anchor: not available for {rel(dataset_dir)}; old_champion_anchor_score=0")
    else:
        print(f"old champion anchor: enabled for {rel(dataset_dir)}")

    files_by_shard = {}
    for shard_id in range(RCFG.n_shards):
        shard_dir = dataset_dir / f"user_shard={shard_id}"
        if path_has_parquet(shard_dir):
            files_by_shard[shard_id] = candidate_files(shard_dir)

    for shard_id in tqdm(files_by_shard, desc=f"Score {record['name']} on {rel(dataset_dir)}"):
        part_lf = scan_parquet(dataset_dir / f"user_shard={shard_id}", hive_partitioning=True)
        part_lf = maybe_join_old_champion_anchor(part_lf, dataset_dir, old_champion_lf)
        available = set(lazy_columns(part_lf))

        missing_exprs = []
        for col in select_cols:
            if col in available:
                continue
            if col in categorical_features or col in {"user_id", "item_id", "catalogid"}:
                missing_exprs.append(pl.lit(-1, dtype=pl.Int64).alias(col))
            elif col == "label":
                missing_exprs.append(pl.lit(0, dtype=pl.UInt8).alias(col))
            elif col.endswith("_rank") or col == "candidate_rank":
                missing_exprs.append(pl.lit(1_000_000, dtype=pl.Int32).alias(col))
            else:
                missing_exprs.append(pl.lit(0.0, dtype=pl.Float32).alias(col))
        if missing_exprs:
            part_lf = part_lf.with_columns(missing_exprs)

        clean_exprs = []
        for col in features:
            if col in categorical_features or col in {"user_id", "item_id", "catalogid"}:
                clean_exprs.append(clean_categorical_expr(col))
            else:
                clean_exprs.append(clean_feature_expr(col))

        part = collect_streaming(part_lf.with_columns(clean_exprs).select(select_cols))
        if part.is_empty():
            continue

        pdf = part.select(features).to_pandas()
        for col in categorical_features:
            if col in pdf.columns:
                pdf[col] = pdf[col].fillna(-1).astype(np.int64)

        scores = predict_model(record, pdf)

        scored = (
            part
            .with_columns(pl.Series("ranker_score", scores).cast(pl.Float32))
            .sort(["user_id", "ranker_score", "item_id"], descending=[False, True, False])
            .with_columns(pl.col("item_id").cum_count().over("user_id").cast(pl.Int32).alias("rank"))
            .filter(pl.col("rank") <= int(k))
            .select(
                "user_id", "item_id", "ranker_score", "rank", "label",
                "candidate_rank", "top_popular_rank", "als_rank", "min_source_rank", "max_source_score",
            )
        )

        rows_written += write_single_shard(scored, out_dir, shard_id, "reranked.parquet")
        del part, pdf, scored
        gc.collect()

    elapsed = time.perf_counter() - t0
    print(f"{record['name']}: scored rows={rows_written:,}, time={elapsed:.1f}s -> {rel(out_dir)}")
    return out_dir, elapsed


# -----------------------------
# Scoring policies
# -----------------------------

def _safe_rank_rr_expr(col: str) -> pl.Expr:
    """Reciprocal rank from raw rank column. Missing/unavailable columns are added before select."""
    x = pl.col(col).cast(pl.Float32, strict=False)
    return (
        pl.when(x.is_null() | x.is_nan() | x.is_infinite() | (x <= 0) | (x >= 1_000_000))
        .then(pl.lit(0.0, dtype=pl.Float32))
        .otherwise(1.0 / (x + 1.0))
        .cast(pl.Float32)
    )


def _max_rank_rr_expr(cols: Sequence[str]) -> pl.Expr:
    cols = list(cols)
    if not cols:
        return pl.lit(0.0, dtype=pl.Float32)
    return pl.max_horizontal([_safe_rank_rr_expr(c) for c in cols]).fill_null(0.0).cast(pl.Float32)




def resolve_project_path_candidates(candidates: Sequence[str]) -> Path | None:
    """Return first existing path from project-root-relative or notebook-relative candidates."""
    for raw in candidates:
        p = Path(raw)
        if not p.is_absolute():
            # Prefer project-root-relative path; fallback to cwd-relative path.
            pr = PROJECT_ROOT / p
            if pr.exists():
                return pr
            pc = Path.cwd() / p
            if pc.exists():
                return pc
        elif p.exists():
            return p
    return None


def old_champion_path_for_dataset(dataset_dir: Path) -> Path | None:
    """Map current val_eval/test dataset to old champion scored parquet if it exists."""
    if not getattr(RCFG, "old_champion_anchor_enable", False):
        return None
    name = dataset_dir.name.lower()
    # val_eval dir is named val_eval; test dir is named test.
    if "val" in name:
        return resolve_project_path_candidates(getattr(RCFG, "old_champion_val_scored_paths", ()))
    if "test" in name:
        return resolve_project_path_candidates(getattr(RCFG, "old_champion_test_scored_paths", ()))
    return None


def old_champion_rank_lazy_for_dataset(dataset_dir: Path) -> pl.LazyFrame | None:
    """LazyFrame with user_id,item_id,old_champion_rank from old CatBoost champion outputs.

    Expected old files are produced by 5_reranker.ipynb:
      data/interim/rerank_val_scored.parquet
      data/interim/rerank_test_scored.parquet

    If files are missing or schema is incompatible, returns None and the anchor becomes zero.
    """
    path = old_champion_path_for_dataset(dataset_dir)
    if path is None or not path.exists():
        return None
    try:
        lf = pl.scan_parquet(str(path))
        cols = set(lf.collect_schema().names())
        if not {"user_id", "item_id"}.issubset(cols):
            print(f"WARNING: old champion path has no user_id/item_id, ignored: {path}")
            return None
        rank_candidates = ["rank", "rerank_rank", "old_rank"]
        rank_col = next((c for c in rank_candidates if c in cols), None)
        if rank_col is None:
            # If rank column is absent but score exists, derive rank from old score.
            score_candidates = ["rerank_score", "ranker_score", "score", "model_score"]
            score_col = next((c for c in score_candidates if c in cols), None)
            if score_col is None:
                print(f"WARNING: old champion path has no rank/score column, ignored: {path}")
                return None
            return (
                lf.select("user_id", "item_id", pl.col(score_col).cast(pl.Float32, strict=False).fill_null(-1e30).alias("_old_champion_score"))
                .sort(["user_id", "_old_champion_score", "item_id"], descending=[False, True, False])
                .with_columns(pl.col("item_id").cum_count().over("user_id").cast(pl.Int32).alias("old_champion_rank"))
                .select("user_id", "item_id", "old_champion_rank")
                .unique(["user_id", "item_id"], keep="first")
            )
        return (
            lf.select("user_id", "item_id", pl.col(rank_col).cast(pl.Int32, strict=False).fill_null(1_000_000).alias("old_champion_rank"))
            .unique(["user_id", "item_id"], keep="first")
        )
    except Exception as e:
        print(f"WARNING: failed to read old champion anchor from {path}: {e}")
        return None


def maybe_join_old_champion_anchor(part_lf: pl.LazyFrame, dataset_dir: Path, old_champion_lf: pl.LazyFrame | None) -> pl.LazyFrame:
    if old_champion_lf is None:
        return part_lf.with_columns(pl.lit(1_000_000, dtype=pl.Int32).alias("old_champion_rank"))
    return part_lf.join(old_champion_lf, on=["user_id", "item_id"], how="left")


def _policy_name_float(x: float) -> str:
    return str(float(x)).replace(".", "p").replace("-", "neg")


def build_policy_specs(record_role: str, record_name: str) -> list[dict]:
    """Build scoring policies for a fitted model.

    Main enabled family:
        raw model score
        + alpha_old * RR(ALS/TopPopular)
        + alpha_extra * RR(BM25/CLIP)
        + alpha_candidate * RR(candidate_rank)
        + alpha_dl * RR(DL sources)

    rr_blend remains disabled because previous runs consistently degraded NDCG.
    """
    safe_record = sanitize_name(record_name)

    def full_policy_name(name: str) -> str:
        return f"{record_role}__{safe_record}__{name}"

    if not getattr(RCFG, "enable_policy_search", True):
        return [{
            "policy_name": full_policy_name("pure_model"),
            "record_role": record_role,
            "record_name": record_name,
            "score_mode": "raw_plus_rr",
            "model_rr_weight": 0.0,
            "old_anchor_alpha": 0.0,
            "extra_anchor_alpha": 0.0,
            "candidate_anchor_alpha": 0.0,
            "dl_anchor_alpha": 0.0,
            "old_champion_anchor_alpha": 0.0,
        }]

    specs: list[dict] = []
    seen: set[str] = set()

    def add_spec(
        name: str,
        *,
        score_mode: str,
        model_rr_weight: float = 0.0,
        old: float = 0.0,
        extra: float = 0.0,
        candidate: float = 0.0,
        dl: float = 0.0,
        champion: float = 0.0,
    ):
        full_name = full_policy_name(name)
        if full_name in seen:
            return
        seen.add(full_name)
        specs.append({
            "policy_name": full_name,
            "record_role": record_role,
            "record_name": record_name,
            "score_mode": score_mode,
            "model_rr_weight": float(model_rr_weight),
            "old_anchor_alpha": float(old),
            "extra_anchor_alpha": float(extra),
            "candidate_anchor_alpha": float(candidate),
            "dl_anchor_alpha": float(dl),
            "old_champion_anchor_alpha": float(champion),
        })

    # Simple old-anchor grid.
    for alpha in getattr(RCFG, "policy_anchor_grid", (0.0,)):
        alpha = float(alpha)
        if alpha == 0.0:
            add_spec("pure_model", score_mode="raw_plus_rr", old=0.0)
        else:
            add_spec(f"raw_old_{_policy_name_float(alpha)}", score_mode="raw_plus_rr", old=alpha)

    # Mixed old + BM25/CLIP + optional DL penalty.
    for old, extra, dl in getattr(RCFG, "policy_raw_mixed_specs", ()):
        old = float(old)
        extra = float(extra)
        dl = float(dl)
        add_spec(
            f"raw_old_{_policy_name_float(old)}_extra_{_policy_name_float(extra)}_dl_{_policy_name_float(dl)}",
            score_mode="raw_plus_rr",
            old=old,
            extra=extra,
            dl=dl,
        )

    # Candidate-rank correction: small weights only.
    for old, extra, dl, candidate in getattr(RCFG, "policy_candidate_mixed_specs", ()):
        old = float(old)
        extra = float(extra)
        dl = float(dl)
        candidate = float(candidate)
        add_spec(
            f"raw_old_{_policy_name_float(old)}_extra_{_policy_name_float(extra)}_cand_{_policy_name_float(candidate)}_dl_{_policy_name_float(dl)}",
            score_mode="raw_plus_rr",
            old=old,
            extra=extra,
            candidate=candidate,
            dl=dl,
        )

    # v10 old champion correction: old CatBoost output as a safety prior.
    for old, extra, dl, candidate, champion in getattr(RCFG, "policy_champion_mixed_specs", ()): 
        old = float(old)
        extra = float(extra)
        dl = float(dl)
        candidate = float(candidate)
        champion = float(champion)
        add_spec(
            f"raw_old_{_policy_name_float(old)}_extra_{_policy_name_float(extra)}_cand_{_policy_name_float(candidate)}_champ_{_policy_name_float(champion)}_dl_{_policy_name_float(dl)}",
            score_mode="raw_plus_rr",
            old=old,
            extra=extra,
            candidate=candidate,
            champion=champion,
            dl=dl,
        )

    if getattr(RCFG, "policy_enable_rr_blend", False):
        w_model = float(getattr(RCFG, "policy_model_rr_weight", 1.0))
        for alpha in getattr(RCFG, "policy_anchor_grid", (0.0,)):
            alpha = float(alpha)
            add_spec(
                f"rr_m{_policy_name_float(w_model)}_old_{_policy_name_float(alpha)}",
                score_mode="rr_blend",
                model_rr_weight=w_model,
                old=alpha,
            )

    return specs


def score_dataset_dir_to_dirs_multi_policy(
    record: dict,
    dataset_dir: Path,
    out_root: Path,
    policy_specs: list[dict],
    *,
    k: int = RCFG.final_k,
) -> tuple[dict[str, Path], float]:
    """Score a dataset once per model and materialize several policy-ranked outputs.

    This avoids repeated LightGBM predictions for every alpha in policy_anchor_grid.
    """
    for spec in policy_specs:
        clean_output_dir(out_root / spec["policy_name"])

    features = record["features"]
    categorical_features = set(record["categorical_features"])

    anchor_rank_cols = list(dict.fromkeys(
        list(getattr(RCFG, "old_anchor_rank_cols", ()))
        + list(getattr(RCFG, "extra_anchor_rank_cols", ()))
        + list(getattr(RCFG, "candidate_anchor_rank_cols", ()))
        + list(getattr(RCFG, "dl_anchor_rank_cols", ()))
    ))

    required_cols = [
        "user_id", "item_id", "label", "candidate_rank",
        "top_popular_rank", "als_rank", "min_source_rank", "max_source_score",
        "old_champion_rank",
        *anchor_rank_cols,
    ]
    select_cols = list(dict.fromkeys(required_cols + features))

    rows_written_by_policy = {spec["policy_name"]: 0 for spec in policy_specs}
    t0 = time.perf_counter()
    old_champion_lf = old_champion_rank_lazy_for_dataset(dataset_dir)
    if old_champion_lf is None:
        print(f"old champion anchor: not available for {rel(dataset_dir)}; old_champion_anchor_score=0")
    else:
        print(f"old champion anchor: enabled for {rel(dataset_dir)}")

    files_by_shard = {}
    for shard_id in range(RCFG.n_shards):
        shard_dir = dataset_dir / f"user_shard={shard_id}"
        if path_has_parquet(shard_dir):
            files_by_shard[shard_id] = candidate_files(shard_dir)

    for shard_id in tqdm(files_by_shard, desc=f"Score policies for {record['name']} on {rel(dataset_dir)}"):
        part_lf = scan_parquet(dataset_dir / f"user_shard={shard_id}", hive_partitioning=True)
        part_lf = maybe_join_old_champion_anchor(part_lf, dataset_dir, old_champion_lf)
        available = set(lazy_columns(part_lf))

        missing_exprs = []
        for col in select_cols:
            if col in available:
                continue
            if col in categorical_features or col in {"user_id", "item_id", "catalogid"}:
                missing_exprs.append(pl.lit(-1, dtype=pl.Int64).alias(col))
            elif col == "label":
                missing_exprs.append(pl.lit(0, dtype=pl.UInt8).alias(col))
            elif col.endswith("_rank") or col == "candidate_rank":
                missing_exprs.append(pl.lit(1_000_000, dtype=pl.Int32).alias(col))
            else:
                missing_exprs.append(pl.lit(0.0, dtype=pl.Float32).alias(col))
        if missing_exprs:
            part_lf = part_lf.with_columns(missing_exprs)

        clean_exprs = []
        for col in features:
            if col in categorical_features or col in {"user_id", "item_id", "catalogid"}:
                clean_exprs.append(clean_categorical_expr(col))
            else:
                clean_exprs.append(clean_feature_expr(col))

        part = collect_streaming(part_lf.with_columns(clean_exprs).select(select_cols))
        if part.is_empty():
            continue

        pdf = part.select(features).to_pandas()
        for col in categorical_features:
            if col in pdf.columns:
                pdf[col] = pdf[col].fillna(-1).astype(np.int64)

        model_scores = predict_model(record, pdf)

        base_part = (
            part
            .with_columns([
                pl.Series("model_score", model_scores).cast(pl.Float32),
                _max_rank_rr_expr(getattr(RCFG, "old_anchor_rank_cols", ())).alias("old_anchor_score"),
                _max_rank_rr_expr(getattr(RCFG, "extra_anchor_rank_cols", ())).alias("extra_anchor_score"),
                _max_rank_rr_expr(getattr(RCFG, "candidate_anchor_rank_cols", ())).alias("candidate_anchor_score"),
                _max_rank_rr_expr(getattr(RCFG, "dl_anchor_rank_cols", ())).alias("dl_anchor_score"),
                _safe_rank_rr_expr("old_champion_rank").alias("old_champion_anchor_score"),
            ])
            .sort(["user_id", "model_score", "item_id"], descending=[False, True, False])
            .with_columns(pl.col("item_id").cum_count().over("user_id").cast(pl.Int32).alias("model_rank"))
            .with_columns((1.0 / pl.col("model_rank").cast(pl.Float32)).fill_null(0.0).alias("model_rank_rr"))
        )

        for spec in policy_specs:
            old_alpha = float(spec.get("old_anchor_alpha", 0.0))
            extra_alpha = float(spec.get("extra_anchor_alpha", 0.0))
            candidate_alpha = float(spec.get("candidate_anchor_alpha", 0.0))
            dl_alpha = float(spec.get("dl_anchor_alpha", 0.0))
            champion_alpha = float(spec.get("old_champion_anchor_alpha", 0.0))
            score_mode = str(spec.get("score_mode", "raw_plus_rr"))
            model_rr_weight = float(spec.get("model_rr_weight", 0.0))

            if score_mode == "rr_blend":
                policy_score_expr = (
                    pl.lit(model_rr_weight, dtype=pl.Float32) * pl.col("model_rank_rr")
                    + pl.lit(old_alpha, dtype=pl.Float32) * pl.col("old_anchor_score")
                    + pl.lit(extra_alpha, dtype=pl.Float32) * pl.col("extra_anchor_score")
                    + pl.lit(candidate_alpha, dtype=pl.Float32) * pl.col("candidate_anchor_score")
                    + pl.lit(dl_alpha, dtype=pl.Float32) * pl.col("dl_anchor_score")
                    + pl.lit(champion_alpha, dtype=pl.Float32) * pl.col("old_champion_anchor_score")
                ).cast(pl.Float32)
            else:
                policy_score_expr = (
                    pl.col("model_score")
                    + pl.lit(old_alpha, dtype=pl.Float32) * pl.col("old_anchor_score")
                    + pl.lit(extra_alpha, dtype=pl.Float32) * pl.col("extra_anchor_score")
                    + pl.lit(candidate_alpha, dtype=pl.Float32) * pl.col("candidate_anchor_score")
                    + pl.lit(dl_alpha, dtype=pl.Float32) * pl.col("dl_anchor_score")
                    + pl.lit(champion_alpha, dtype=pl.Float32) * pl.col("old_champion_anchor_score")
                ).cast(pl.Float32)

            scored = (
                base_part
                .with_columns(policy_score_expr.alias("ranker_score"))
                .sort(["user_id", "ranker_score", "item_id"], descending=[False, True, False])
                .with_columns(pl.col("item_id").cum_count().over("user_id").cast(pl.Int32).alias("rank"))
                .filter(pl.col("rank") <= int(k))
                .select(
                    "user_id", "item_id", "ranker_score", "model_score", "model_rank_rr",
                    "old_anchor_score", "extra_anchor_score", "candidate_anchor_score", "dl_anchor_score", "old_champion_anchor_score",
                    "rank", "label", "candidate_rank", "top_popular_rank", "als_rank", "min_source_rank", "max_source_score", "old_champion_rank",
                )
            )
            rows_written_by_policy[spec["policy_name"]] += write_single_shard(scored, out_root / spec["policy_name"], shard_id, "reranked.parquet")
            del scored

        del part, pdf, base_part
        gc.collect()

    elapsed = time.perf_counter() - t0
    out_dirs = {spec["policy_name"]: out_root / spec["policy_name"] for spec in policy_specs}
    for name, rows_written in rows_written_by_policy.items():
        print(f"{record['name']} / {name}: scored rows={rows_written:,} -> {rel(out_dirs[name])}")
    print(f"{record['name']}: multi-policy scoring time={elapsed:.1f}s")
    return out_dirs, elapsed


def summarize_policy_metrics(metrics: pd.DataFrame, policy_spec: dict, scored_dir: Path, predict_time_sec: float) -> dict:
    row = {
        "policy_name": policy_spec["policy_name"],
        "record_role": policy_spec.get("record_role"),
        "record_name": policy_spec.get("record_name"),
        "scored_dir": rel(scored_dir),
        "score_mode": str(policy_spec.get("score_mode", "raw_plus_rr")),
        "model_rr_weight": float(policy_spec.get("model_rr_weight", 0.0)),
        "old_anchor_alpha": float(policy_spec.get("old_anchor_alpha", 0.0)),
        "extra_anchor_alpha": float(policy_spec.get("extra_anchor_alpha", 0.0)),
        "candidate_anchor_alpha": float(policy_spec.get("candidate_anchor_alpha", 0.0)),
        "dl_anchor_alpha": float(policy_spec.get("dl_anchor_alpha", 0.0)),
        "old_champion_anchor_alpha": float(policy_spec.get("old_champion_anchor_alpha", 0.0)),
        "predict_time_sec": float(predict_time_sec),
        "optimize_metric_value": metric_from_metrics_df(metrics, RCFG.optimize_metric) if not metrics.empty else np.nan,
    }
    for _, mrow in metrics.iterrows():
        k = int(mrow["k"])
        row[f"ndcg@{k}"] = float(mrow["ndcg"])
        row[f"map@{k}"] = float(mrow["map"])
        row[f"recall@{k}"] = float(mrow["recall"])
        row[f"hit_rate@{k}"] = float(mrow["hit_rate"])
    return row




def _policy_score_expr_from_spec(spec: dict) -> pl.Expr:
    old_alpha = float(spec.get("old_anchor_alpha", 0.0))
    extra_alpha = float(spec.get("extra_anchor_alpha", 0.0))
    candidate_alpha = float(spec.get("candidate_anchor_alpha", 0.0))
    dl_alpha = float(spec.get("dl_anchor_alpha", 0.0))
    champion_alpha = float(spec.get("old_champion_anchor_alpha", 0.0))
    score_mode = str(spec.get("score_mode", "raw_plus_rr"))
    model_rr_weight = float(spec.get("model_rr_weight", 0.0))

    if score_mode == "rr_blend":
        return (
            pl.lit(model_rr_weight, dtype=pl.Float32) * pl.col("model_rank_rr")
            + pl.lit(old_alpha, dtype=pl.Float32) * pl.col("old_anchor_score")
            + pl.lit(extra_alpha, dtype=pl.Float32) * pl.col("extra_anchor_score")
            + pl.lit(candidate_alpha, dtype=pl.Float32) * pl.col("candidate_anchor_score")
            + pl.lit(dl_alpha, dtype=pl.Float32) * pl.col("dl_anchor_score")
            + pl.lit(champion_alpha, dtype=pl.Float32) * pl.col("old_champion_anchor_score")
        ).cast(pl.Float32)

    return (
        pl.col("model_score")
        + pl.lit(old_alpha, dtype=pl.Float32) * pl.col("old_anchor_score")
        + pl.lit(extra_alpha, dtype=pl.Float32) * pl.col("extra_anchor_score")
        + pl.lit(candidate_alpha, dtype=pl.Float32) * pl.col("candidate_anchor_score")
        + pl.lit(dl_alpha, dtype=pl.Float32) * pl.col("dl_anchor_score")
        + pl.lit(champion_alpha, dtype=pl.Float32) * pl.col("old_champion_anchor_score")
    ).cast(pl.Float32)


def evaluate_policy_specs_streaming(
    record: dict,
    dataset_dir: Path,
    ground_truth: dict[int, list[int]],
    policy_specs: list[dict],
    *,
    k: int,
    ks: Iterable[int] = RCFG.metric_ks,
) -> tuple[pd.DataFrame, pd.DataFrame, float]:
    """Score many policies on a labeled dataset without materializing every policy output.

    This is the key v7 change: we can test a fine anchor grid and mixed anchors on val_eval
    without writing N policy directories with tens of millions of rows each. Only the final
    selected policy is materialized later.
    """
    ks = tuple(sorted(int(x) for x in ks))
    k = int(max(k, max(ks), RCFG.final_k))
    gt_users_total = len(ground_truth)
    if gt_users_total == 0:
        raise ValueError("ground_truth is empty; cannot evaluate policy search")

    features = record["features"]
    categorical_features = set(record["categorical_features"])
    anchor_rank_cols = list(dict.fromkeys(
        list(getattr(RCFG, "old_anchor_rank_cols", ()))
        + list(getattr(RCFG, "extra_anchor_rank_cols", ()))
        + list(getattr(RCFG, "candidate_anchor_rank_cols", ()))
        + list(getattr(RCFG, "dl_anchor_rank_cols", ()))
    ))
    required_cols = [
        "user_id", "item_id", "label", "candidate_rank",
        "top_popular_rank", "als_rank", "min_source_rank", "max_source_score",
        "old_champion_rank",
        *anchor_rank_cols,
    ]
    select_cols = list(dict.fromkeys(required_cols + features))

    metric_names = ["hit_rate", "recall", "precision", "ndcg", "map", "mrr"]
    sums = {
        spec["policy_name"]: {kk: defaultdict(float) for kk in ks}
        for spec in policy_specs
    }
    users_with_predictions = {spec["policy_name"]: 0 for spec in policy_specs}

    files_by_shard = {}
    for shard_id in range(RCFG.n_shards):
        shard_dir = dataset_dir / f"user_shard={shard_id}"
        if path_has_parquet(shard_dir):
            files_by_shard[shard_id] = candidate_files(shard_dir)

    t0 = time.perf_counter()
    old_champion_lf = old_champion_rank_lazy_for_dataset(dataset_dir)
    if old_champion_lf is None:
        print(f"old champion anchor: not available for {rel(dataset_dir)}; old_champion_anchor_score=0")
    else:
        print(f"old champion anchor: enabled for {rel(dataset_dir)}")

    for shard_id in tqdm(files_by_shard, desc=f"Evaluate policies for {record['name']} on {rel(dataset_dir)}"):
        part_lf = scan_parquet(dataset_dir / f"user_shard={shard_id}", hive_partitioning=True)
        part_lf = maybe_join_old_champion_anchor(part_lf, dataset_dir, old_champion_lf)
        available = set(lazy_columns(part_lf))

        missing_exprs = []
        for col in select_cols:
            if col in available:
                continue
            if col in categorical_features or col in {"user_id", "item_id", "catalogid"}:
                missing_exprs.append(pl.lit(-1, dtype=pl.Int64).alias(col))
            elif col == "label":
                missing_exprs.append(pl.lit(0, dtype=pl.UInt8).alias(col))
            elif col.endswith("_rank") or col == "candidate_rank":
                missing_exprs.append(pl.lit(1_000_000, dtype=pl.Int32).alias(col))
            else:
                missing_exprs.append(pl.lit(0.0, dtype=pl.Float32).alias(col))
        if missing_exprs:
            part_lf = part_lf.with_columns(missing_exprs)

        clean_exprs = []
        for col in features:
            if col in categorical_features or col in {"user_id", "item_id", "catalogid"}:
                clean_exprs.append(clean_categorical_expr(col))
            else:
                clean_exprs.append(clean_feature_expr(col))

        part = collect_streaming(part_lf.with_columns(clean_exprs).select(select_cols))
        if part.is_empty():
            continue

        pdf = part.select(features).to_pandas()
        for col in categorical_features:
            if col in pdf.columns:
                pdf[col] = pdf[col].fillna(-1).astype(np.int64)

        model_scores = predict_model(record, pdf)

        base_part = (
            part
            .with_columns([
                pl.Series("model_score", model_scores).cast(pl.Float32),
                _max_rank_rr_expr(getattr(RCFG, "old_anchor_rank_cols", ())).alias("old_anchor_score"),
                _max_rank_rr_expr(getattr(RCFG, "extra_anchor_rank_cols", ())).alias("extra_anchor_score"),
                _max_rank_rr_expr(getattr(RCFG, "candidate_anchor_rank_cols", ())).alias("candidate_anchor_score"),
                _max_rank_rr_expr(getattr(RCFG, "dl_anchor_rank_cols", ())).alias("dl_anchor_score"),
                _safe_rank_rr_expr("old_champion_rank").alias("old_champion_anchor_score"),
            ])
            .sort(["user_id", "model_score", "item_id"], descending=[False, True, False])
            .with_columns(pl.col("item_id").cum_count().over("user_id").cast(pl.Int32).alias("model_rank"))
            .with_columns((1.0 / pl.col("model_rank").cast(pl.Float32)).fill_null(0.0).alias("model_rank_rr"))
        )

        for spec in policy_specs:
            policy_name = spec["policy_name"]
            scored = (
                base_part
                .with_columns(_policy_score_expr_from_spec(spec).alias("ranker_score"))
                .sort(["user_id", "ranker_score", "item_id"], descending=[False, True, False])
                .with_columns(pl.col("item_id").cum_count().over("user_id").cast(pl.Int32).alias("rank"))
                .filter(pl.col("rank") <= k)
                .select("user_id", "item_id", "rank")
            )
            grouped = scored.group_by("user_id", maintain_order=True).agg(pl.col("item_id").alias("items"))

            shard_pred_users = 0
            for row in grouped.iter_rows(named=True):
                user_id = int(row["user_id"])
                gt_items = ground_truth.get(user_id)
                if not gt_items:
                    continue
                shard_pred_users += 1
                pred_items = [int(x) for x in row["items"]]
                for kk in ks:
                    m = user_metrics_at_k(gt_items, pred_items, kk)
                    for metric in metric_names:
                        sums[policy_name][kk][metric] += float(m[metric])

            users_with_predictions[policy_name] += shard_pred_users
            del scored, grouped

        del part, pdf, base_part
        gc.collect()

    elapsed = time.perf_counter() - t0

    metric_rows = []
    summary_rows = []
    spec_by_name = {spec["policy_name"]: spec for spec in policy_specs}

    for policy_name, spec in spec_by_name.items():
        for kk in ks:
            row = {
                "model": f"val_{policy_name}",
                "policy_name": policy_name,
                "record_role": spec.get("record_role"),
                "record_name": spec.get("record_name"),
                "k": int(kk),
                "users_total": int(gt_users_total),
                "users_with_predictions": int(users_with_predictions[policy_name]),
                "user_coverage": users_with_predictions[policy_name] / gt_users_total if gt_users_total else np.nan,
            }
            for metric in metric_names:
                row[metric] = sums[policy_name][kk][metric] / gt_users_total
            metric_rows.append(row)

        metric_df_one = pd.DataFrame([r for r in metric_rows if r["policy_name"] == policy_name])
        summary_rows.append(summarize_policy_metrics(metric_df_one, spec, dataset_dir, elapsed))

    return pd.DataFrame(metric_rows), pd.DataFrame(summary_rows), elapsed


def select_policy_from_summary(policy_summary: pd.DataFrame) -> dict:
    """Robust val-based policy choice.

    Primary: keep all policies within epsilon of max RCFG.optimize_metric.
    Inside that near-tie set, choose by configured tie-breakers first (recall@100,
    ndcg@200, recall@200, etc.), then by optimize metric. This fixes the v7 issue
    where sorting still started from NDCG and tie-breakers were mostly ineffective.
    """
    if policy_summary.empty:
        raise ValueError("policy_summary is empty")

    metric_col = "optimize_metric_value"
    eps = float(getattr(RCFG, "policy_selection_ndcg_epsilon", 0.0))
    best = float(policy_summary[metric_col].max())
    candidates = policy_summary.loc[policy_summary[metric_col] >= best - eps].copy()

    sort_cols: list[str] = []
    ascending: list[bool] = []

    for col in getattr(RCFG, "policy_selection_tie_breakers", ()):
        if col in candidates.columns:
            sort_cols.append(col)
            ascending.append(False)

    sort_cols.append(metric_col)
    ascending.append(False)

    # Prefer less extreme anchors only after quality tie-breakers to avoid over-anchoring.
    for col in ("old_champion_anchor_alpha", "candidate_anchor_alpha", "extra_anchor_alpha", "old_anchor_alpha"):
        if col in candidates.columns:
            sort_cols.append(col)
            ascending.append(True)

    sort_cols.append("policy_name")
    ascending.append(True)

    selected = candidates.sort_values(sort_cols, ascending=ascending).iloc[0].to_dict()
    selected["selection_epsilon"] = eps
    selected["selection_candidates_within_epsilon"] = int(candidates.shape[0])
    selected["selection_sort_cols"] = ",".join(sort_cols)
    return selected


MODEL_COMPARISON_DIR = RERANKED_DIR / "model_comparison"
MODEL_COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

model_records: list[dict] = []

# New LGBM tuning returns several candidates; full val_eval comparison below chooses among them.
if "lgbm_records" in globals() and lgbm_records:
    model_records.extend(lgbm_records)
elif "lgbm_record" in globals() and lgbm_record is not None:
    model_records.append(lgbm_record)

if "catboost_record" in globals() and catboost_record is not None:
    model_records.append(catboost_record)

if not model_records:
    raise RuntimeError("No ranking models were trained. Enable LightGBM or CatBoost.")

saved_model_paths = {record["name"]: save_model_record(record, suffix="selection") for record in model_records}

comparison_metric_rows = []
comparison_summary_rows = []

for record in model_records:
    val_out_dir = MODEL_COMPARISON_DIR / record["name"] / "val"
    scored_dir, predict_time_sec = score_dataset_dir_to_dir(record, val_eval_dir, val_out_dir, k=max(RCFG.metric_ks))
    metrics = evaluate_scored_dir(scored_dir, val_gt, rank_col="rank", label=record["name"])
    comparison_metric_rows.append(metrics)

    row = {
        "model": record["name"],
        "model_type": record["model_type"],
        "features_count": len(record["features"]),
        "categorical_features": ",".join(record["categorical_features"]),
        "tuning_time_sec": record["tuning_time_sec"],
        "selection_fit_time_sec": record["selection_fit_time_sec"],
        "val_predict_time_sec": predict_time_sec,
        "best_iteration": record["best_iteration"],
        "model_paths": json.dumps(saved_model_paths[record["name"]], ensure_ascii=False),
    }
    for _, mrow in metrics.iterrows():
        k = int(mrow["k"])
        row[f"ndcg@{k}"] = float(mrow["ndcg"])
        row[f"map@{k}"] = float(mrow["map"])
        row[f"recall@{k}"] = float(mrow["recall"])
        row[f"hit_rate@{k}"] = float(mrow["hit_rate"])
    row["optimize_metric_value"] = metric_from_metrics_df(metrics, RCFG.optimize_metric)
    comparison_summary_rows.append(row)

model_val_metrics = pd.concat(comparison_metric_rows, ignore_index=True)
model_comparison = pd.DataFrame(comparison_summary_rows).sort_values(["optimize_metric_value", "val_predict_time_sec"], ascending=[False, True])

model_comparison_path = REPORTS_DIR / "final_reranker_model_comparison.parquet"
model_val_metrics_path = REPORTS_DIR / "final_reranker_model_val_metrics.parquet"
model_comparison.to_parquet(model_comparison_path, index=False)
model_val_metrics.to_parquet(model_val_metrics_path, index=False)

print("saved:", rel(model_comparison_path))
print("saved:", rel(model_val_metrics_path))
display(model_comparison)
display(model_val_metrics.sort_values(["model", "k"]))

old champion anchor: not available for data\processed\reranker\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\dataset\val_eval; old_champion_anchor_score=0


Score lgbm_ranker_t01 on data\processed\reranker\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\…

lgbm_ranker_t01: scored rows=71,985,600, time=1142.3s -> data\datamart\candidates\reranked\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\model_comparison\lgbm_ranker_t01\val


Evaluate lgbm_ranker_t01:   0%|          | 0/512 [00:00<?, ?it/s]

old champion anchor: not available for data\processed\reranker\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\dataset\val_eval; old_champion_anchor_score=0


Score lgbm_ranker_t02 on data\processed\reranker\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\…

lgbm_ranker_t02: scored rows=71,985,600, time=1806.5s -> data\datamart\candidates\reranked\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\model_comparison\lgbm_ranker_t02\val


Evaluate lgbm_ranker_t02:   0%|          | 0/512 [00:00<?, ?it/s]

saved: artifacts\reports\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\final_reranker_model_comparison.parquet
saved: artifacts\reports\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\final_reranker_model_val_metrics.parquet


,model,model_type,features_count,categorical_features,tuning_time_sec,selection_fit_time_sec,val_predict_time_sec,best_iteration,model_paths,ndcg@20,map@20,recall@20,hit_rate@20,ndcg@50,map@50,recall@50,hit_rate@50,ndcg@100,map@100,recall@100,hit_rate@100,ndcg@200,map@200,recall@200,hit_rate@200,optimize_metric_value
1,lgbm_ranker_t02,lgbm,240,,902.661875,240.829060,1806.541296,None,"{""joblib"": ""models\\reranker\\final_reranker_v...",0.078300,0.049750,0.108768,0.239578,0.083814,0.050737,0.129177,0.279431,0.087751,0.051109,0.145960,0.310854,0.091816,0.051338,0.166388,0.346203,0.087751
0,lgbm_ranker_t01,lgbm,240,,902.661875,89.231353,1142.276243,None,"{""joblib"": ""models\\reranker\\final_reranker_v...",0.077466,0.049287,0.106986,0.236217,0.082801,0.050235,0.126854,0.275547,0.086768,0.050609,0.143813,0.307411,0.091045,0.050849,0.165269,0.344572,0.086768


,model,k,users_total,users_with_predictions,user_coverage,hit_rate,recall,precision,ndcg,map,mrr
0,lgbm_ranker_t01,20,359928,359928,1.0,0.236217,0.106986,0.017192,0.077466,0.049287,0.107078
1,lgbm_ranker_t01,50,359928,359928,1.0,0.275547,0.126854,0.008488,0.082801,0.050235,0.108348
2,lgbm_ranker_t01,100,359928,359928,1.0,0.307411,0.143813,0.004969,0.086768,0.050609,0.108803
3,lgbm_ranker_t01,200,359928,359928,1.0,0.344572,0.165269,0.002933,0.091045,0.050849,0.109066
4,lgbm_ranker_t02,20,359928,359928,1.0,0.239578,0.108768,0.017421,0.078300,0.049750,0.107552
5,lgbm_ranker_t02,50,359928,359928,1.0,0.279431,0.129177,0.008634,0.083814,0.050737,0.108843
6,lgbm_ranker_t02,100,359928,359928,1.0,0.310854,0.145960,0.005042,0.087751,0.051109,0.109292
7,lgbm_ranker_t02,200,359928,359928,1.0,0.346203,0.166388,0.002945,0.091816,0.051338,0.109543


In [49]:
best_model_name = str(model_comparison.iloc[0]["model"])
best_selection_record = next(r for r in model_records if r["name"] == best_model_name)
print(f"Best model by {RCFG.optimize_metric}: {best_model_name}")


def fit_production_model_from_record(record: dict) -> dict:
    """
    selection  -> использовать модель, обученную на train_fit
    val_fit    -> переобучить финальную модель на свежем val_fit
    train_val  -> переобучить на train_fit + val_fit, если хватает RAM
    """
    mode = getattr(RCFG, "production_train_mode", "val_fit")

    if mode == "selection":
        prod_record = record.copy()
        prod_record["production_fit_time_sec"] = 0.0
        prod_record["production_refit"] = False
        prod_record["production_train_mode"] = "selection"
        print("Production refit disabled: using selection model.")
        return prod_record

    params = record["params"].copy()

    if record["best_iteration"] is not None:
        best_iter = int(record["best_iteration"])
        # Чтобы не получить недообученную финальную модель на 50 деревьев.
        best_iter = max(best_iter, 200)

        if record["model_type"] == "lgbm":
            params["n_estimators"] = best_iter
        elif record["model_type"] == "catboost":
            params["iterations"] = best_iter

    if mode == "val_fit":
        # val_pdf is already sorted/grouped by collect_model_pdf_and_group().
        # Do NOT copy and re-prepare it: for 100+ features this can allocate 8-12+ GB
        # and fail before training starts.
        production_pdf = val_pdf
        production_group = val_group

    elif mode == "train_val":
        # Disabled by default. This can be heavy because it materializes train+val.
        production_pdf = pd.concat([train_pdf, val_pdf], ignore_index=True, copy=False)
        production_pdf, production_group = prepare_grouped_pdf(
            production_pdf,
            split_name="train_val_production",
        )

    else:
        raise ValueError(f"Unknown production_train_mode={mode}")

    t0 = time.perf_counter()

    if record["model_type"] == "lgbm":
        model = fit_lgbm_ranker(
            params,
            production_pdf,
            production_group,
            valid_data=None,
            valid_groups=None,
        )
    elif record["model_type"] == "catboost":
        model = fit_catboost_ranker(
            params,
            production_pdf,
            valid_data=None,
        )
    else:
        raise ValueError(record["model_type"])

    production_fit_time_sec = time.perf_counter() - t0

    prod_record = record.copy()
    prod_record["model"] = model
    prod_record["params"] = params
    prod_record["production_fit_time_sec"] = production_fit_time_sec
    prod_record["production_refit"] = True
    prod_record["production_train_mode"] = mode
    # Production refit is trained without an eval_set/early stopping. Predict with all trees.
    # Otherwise predict_model() may reuse the selection best_iteration and silently score only ~50 trees.
    prod_record["best_iteration"] = None

    del production_pdf, production_group
    gc.collect()

    print(f"Production refit mode={mode}, time={production_fit_time_sec:.1f}s")
    return prod_record



# Production refit is intentionally delayed until after val-based policy/model selection.
# In v8 policy-search can select a non-top pure LGBM record after anchors, so refitting here
# would either be wasted or could save the wrong final model.
production_record = None
production_model_paths = {}
print("Production refit will be performed after validation policy selection.")


Best model by ndcg@100: lgbm_ranker_t02
Production refit will be performed after validation policy selection.


In [50]:
def get_feature_importance(record: dict) -> pd.DataFrame:
    features = record["features"]
    model = record["model"]

    if record["model_type"] == "lgbm":
        return pd.DataFrame({
            "feature": features,
            "importance_gain": model.booster_.feature_importance(importance_type="gain"),
            "importance_split": model.booster_.feature_importance(importance_type="split"),
        }).sort_values("importance_gain", ascending=False)

    if record["model_type"] == "catboost":
        pool = Pool(
            train_pdf[features],
            label=train_pdf["label"],
            group_id=train_pdf["user_id"],
            cat_features=[c for c in record["categorical_features"] if c in features],
        )
        importance = model.get_feature_importance(pool, type="FeatureImportance")
        return pd.DataFrame({"feature": features, "importance": importance}).sort_values("importance", ascending=False)

    raise ValueError(record["model_type"])


# Feature importance is computed after final production model is selected/refit.


In [51]:
# Full validation of the best pure selection model.
# This is still useful as a baseline before policy blending.
val_reranked_dir, val_predict_time_sec = score_dataset_dir_to_dir(
    best_selection_record,
    val_eval_dir,
    RERANKED_DIR / "val",
    k=max(RCFG.metric_ks),
)

best_val_metrics = evaluate_scored_dir(
    val_reranked_dir,
    val_gt,
    rank_col="rank",
    label=f"best_selection_{best_model_name}",
)

best_val_metrics_path = REPORTS_DIR / "final_reranker_best_val_metrics.parquet"
best_val_metrics.to_parquet(best_val_metrics_path, index=False)
display(best_val_metrics)
print("saved:", rel(best_val_metrics_path))

# -----------------------------------------------------------------------------
# v10 strict policy/model search + old champion anchor on val_eval, not on test.
# -----------------------------------------------------------------------------
# Changes vs v8/v9:
# - policy search is run for top-N pure LGBM records, not only for the best pure model;
# - candidate_rank_rr anchor is included with small weights;
# - robust tie-breaker is fixed: within epsilon, recall/deeper metrics are considered before NDCG;
# - production refit is delayed until after val policy/model selection, so the selected model is refit.

records_by_name = {record["name"]: record for record in model_records}
top_policy_model_names = (
    model_comparison
    .head(int(getattr(RCFG, "policy_search_top_models", 1)))["model"]
    .astype(str)
    .tolist()
)

val_policy_metrics_parts: list[pd.DataFrame] = []
val_policy_summary_parts: list[pd.DataFrame] = []
policy_specs_by_name: dict[str, dict] = {}
val_policy_predict_time_sec = 0.0

val_policy_k = int(max(getattr(RCFG, "policy_search_k", RCFG.final_k), max(RCFG.metric_ks), RCFG.final_k))

print("Policy search top models:", top_policy_model_names)

for model_name in top_policy_model_names:
    val_policy_record_role = "selection"
    val_policy_record = records_by_name[model_name]
    val_policy_specs = build_policy_specs(val_policy_record_role, val_policy_record["name"])
    policy_specs_by_name.update({spec["policy_name"]: spec for spec in val_policy_specs})

    print("\nVal policy search record:", val_policy_record_role, val_policy_record["name"])
    print("Val policy candidates count:", len(val_policy_specs))

    metrics_one, summary_one, predict_time_one = evaluate_policy_specs_streaming(
        val_policy_record,
        val_eval_dir,
        val_gt,
        val_policy_specs,
        k=val_policy_k,
        ks=RCFG.metric_ks,
    )
    val_policy_metrics_parts.append(metrics_one)
    val_policy_summary_parts.append(summary_one)
    val_policy_predict_time_sec += float(predict_time_one)

val_policy_metrics = pd.concat(val_policy_metrics_parts, ignore_index=True)
val_policy_summary = pd.concat(val_policy_summary_parts, ignore_index=True)

final_policy_row = select_policy_from_summary(val_policy_summary)

selected_val_policy_name = str(final_policy_row["policy_name"])
selected_val_policy_spec = policy_specs_by_name[selected_val_policy_name]
selected_selection_record_name = str(final_policy_row["record_name"])
selected_selection_record = records_by_name[selected_selection_record_name]

print("Selected validation policy/model:")
print(json.dumps(final_policy_row, ensure_ascii=False, indent=2))

# Refit production model only now, after model+policy are selected on val_eval.
production_record = fit_production_model_from_record(selected_selection_record)
production_model_paths = save_model_record(production_record, suffix="production")

best_joblib_path = MODELS_DIR / "best_ranker.joblib"
joblib.dump(production_record["model"], best_joblib_path)
production_model_paths["best_ranker_joblib"] = rel(best_joblib_path)

if production_record["model_type"] == "lgbm":
    best_txt_path = MODELS_DIR / "best_ranker.txt"
    production_record["model"].booster_.save_model(str(best_txt_path))
    production_model_paths["best_ranker_txt"] = rel(best_txt_path)
elif production_record["model_type"] == "catboost":
    best_cbm_path = MODELS_DIR / "best_ranker.cbm"
    production_record["model"].save_model(str(best_cbm_path))
    production_model_paths["best_ranker_cbm"] = rel(best_cbm_path)

print("production model paths:")
print(json.dumps(production_model_paths, ensure_ascii=False, indent=2))
print(f"production fit time: {production_record['production_fit_time_sec']:.1f}s")

# Compute final production feature importance here, because production_record is now final.
importance = get_feature_importance(production_record)
importance_path = REPORTS_DIR / "final_reranker_feature_importance.parquet"
importance.to_parquet(importance_path, index=False)
display(importance.head(50))
print("saved:", rel(importance_path))

# Convert selected validation policy to production-test policy with the same scoring parameters.
selected_record_role = "production"
base_policy_tail = selected_val_policy_name.split("__", 2)[2] if selected_val_policy_name.count("__") >= 2 else selected_val_policy_name
final_policy_name = f"production__{sanitize_name(production_record['name'])}__{base_policy_tail}"

selected_policy_spec = selected_val_policy_spec.copy()
selected_policy_spec.update({
    "policy_name": final_policy_name,43
    "record_role": selected_record_role,
    "record_name": production_record["name"],
})

print("Applied final test policy:", final_policy_name)

# Save val policy artifacts
val_policy_metrics_path = REPORTS_DIR / "final_reranker_val_policy_metrics.parquet"
val_policy_summary_path = REPORTS_DIR / "final_reranker_val_policy_summary.parquet"
val_policy_metrics.to_parquet(val_policy_metrics_path, index=False)
val_policy_summary.to_parquet(val_policy_summary_path, index=False)
print("saved:", rel(val_policy_metrics_path))
print("saved:", rel(val_policy_summary_path))
print("\nSelected validation policy:")
print(json.dumps(final_policy_row, ensure_ascii=False, indent=2))
display(
    val_policy_summary
    .sort_values(["optimize_metric_value", "recall@100"], ascending=[False, False])
    .head(40)
 )


old champion anchor: not available for data\processed\reranker\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\dataset\val_eval; old_champion_anchor_score=0


Score lgbm_ranker_t02 on data\processed\reranker\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\…

lgbm_ranker_t02: scored rows=71,985,600, time=1806.4s -> data\datamart\candidates\reranked\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\val


Evaluate best_selection_lgbm_ranker_t02:   0%|          | 0/512 [00:00<?, ?it/s]

,model,k,users_total,users_with_predictions,user_coverage,hit_rate,recall,precision,ndcg,map,mrr
0,best_selection_lgbm_ranker_t02,20,359928,359928,1.0,0.239578,0.108768,0.017421,0.078300,0.049750,0.107552
1,best_selection_lgbm_ranker_t02,50,359928,359928,1.0,0.279431,0.129177,0.008634,0.083814,0.050737,0.108843
2,best_selection_lgbm_ranker_t02,100,359928,359928,1.0,0.310854,0.145960,0.005042,0.087751,0.051109,0.109292
3,best_selection_lgbm_ranker_t02,200,359928,359928,1.0,0.346203,0.166388,0.002945,0.091816,0.051338,0.109543


saved: artifacts\reports\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\final_reranker_best_val_metrics.parquet
Policy search top models: ['lgbm_ranker_t02', 'lgbm_ranker_t01']

Val policy search record: selection lgbm_ranker_t02
Val policy candidates count: 82
old champion anchor: not available for data\processed\reranker\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\dataset\val_eval; old_champion_anchor_score=0


... (3 outputs left out)

Selected validation policy/model:
{
  "policy_name": "selection__lgbm_ranker_t02__pure_model",
  "record_role": "selection",
  "record_name": "lgbm_ranker_t02",
  "scored_dir": "data\\processed\\reranker\\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\\dataset\\val_eval",
  "score_mode": "raw_plus_rr",
  "model_rr_weight": 0.0,
  "old_anchor_alpha": 0.0,
  "extra_anchor_alpha": 0.0,
  "candidate_anchor_alpha": 0.0,
  "dl_anchor_alpha": 0.0,
  "old_champion_anchor_alpha": 0.0,
  "predict_time_sec": 12885.859318499977,
  "optimize_metric_value": 0.08775125247429988,
  "ndcg@20": 0.07830040977903494,
  "map@20": 0.04975015147198327,
  "recall@20": 0.10876795940421516,
  "hit_rate@20": 0.23957847124980552,
  "ndcg@50": 0.08381442385898676,
  "map@50": 0.05073730394337168,
  "recall@50": 0.12917657056069487,
  "hit_rate@50": 0.27943088617723544,
  "ndcg@100": 0.08775125247429988,
  "map@100": 0.051109434033729084,
  "recall@100": 0.14595966298673213,
  "hit_rate@100": 0.310853

,feature,importance_gain,importance_split
40,f_num__candidate__repeat_vs_sasrec_rank_gap,2.470514e+06,445
38,f_num__candidate__repeat_vs_bm25_rank_gap,1.168206e+06,1958
21,f_num__candidate__repeat_history_score,1.005672e+06,702
6,f_num__candidate__repeat_cart_orders_rank,9.545598e+05,1673
47,f_num__candidate__candidate_rank,7.750454e+05,1726
27,f_num__candidate__repeat_rank_delta_to_best,6.631433e+05,185
41,f_num__candidate__als_rank,5.563560e+05,476
34,f_num__candidate__repeat_recency_score,5.324299e+05,1716
49,f_num__candidate__candidate_rank_rr,4.943155e+05,1293
18,f_num__candidate__repeat_history_rank,4.908142e+05,334


saved: artifacts\reports\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\final_reranker_feature_importance.parquet
Applied final test policy: production__lgbm_ranker_t02__pure_model
saved: artifacts\reports\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\final_reranker_val_policy_metrics.parquet
saved: artifacts\reports\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\final_reranker_val_policy_summary.parquet

Selected validation policy:
{
  "policy_name": "selection__lgbm_ranker_t02__pure_model",
  "record_role": "selection",
  "record_name": "lgbm_ranker_t02",
  "scored_dir": "data\\processed\\reranker\\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\\dataset\\val_eval",
  "score_mode": "raw_plus_rr",
  "model_rr_weight": 0.0,
  "old_anchor_alpha": 0.0,
  "extra_anchor_alpha": 0.0,
  "candidate_anchor_alpha": 0.0,
  "dl_anchor_alpha": 0.0,
  "old_champion_anchor_alpha": 0.0,
  "predict_time_sec": 12885.859318499977,
  "optimize_metri

,policy_name,record_role,record_name,scored_dir,score_mode,model_rr_weight,old_anchor_alpha,extra_anchor_alpha,candidate_anchor_alpha,dl_anchor_alpha,old_champion_anchor_alpha,predict_time_sec,optimize_metric_value,ndcg@20,map@20,recall@20,hit_rate@20,ndcg@50,map@50,recall@50,hit_rate@50,ndcg@100,map@100,recall@100,hit_rate@100,ndcg@200,map@200,recall@200,hit_rate@200
0,selection__lgbm_ranker_t02__pure_model,selection,lgbm_ranker_t02,data\processed\reranker\final_reranker_v12_rep...,raw_plus_rr,0.0,0.0,0.00,0.00,0.00,0.0,12885.859318,0.087751,0.078300,0.049750,0.108768,0.239578,0.083814,0.050737,0.129177,0.279431,0.087751,0.051109,0.145960,0.310854,0.091816,0.051338,0.166388,0.346203
82,selection__lgbm_ranker_t01__pure_model,selection,lgbm_ranker_t01,data\processed\reranker\final_reranker_v12_rep...,raw_plus_rr,0.0,0.0,0.00,0.00,0.00,0.0,12969.542483,0.086768,0.077466,0.049287,0.106986,0.236217,0.082801,0.050235,0.126854,0.275547,0.086768,0.050609,0.143813,0.307411,0.091045,0.050849,0.165269,0.344572
1,selection__lgbm_ranker_t02__raw_old_1p0,selection,lgbm_ranker_t02,data\processed\reranker\final_reranker_v12_rep...,raw_plus_rr,0.0,1.0,0.00,0.00,0.00,0.0,12885.859318,0.084755,0.075164,0.046474,0.108242,0.238681,0.080743,0.047470,0.128926,0.278984,0.084755,0.047848,0.146101,0.311034,0.088888,0.048080,0.166880,0.346953
83,selection__lgbm_ranker_t01__raw_old_1p0,selection,lgbm_ranker_t01,data\processed\reranker\final_reranker_v12_rep...,raw_plus_rr,0.0,1.0,0.00,0.00,0.00,0.0,12969.542483,0.083731,0.074202,0.046020,0.106112,0.234575,0.079677,0.046988,0.126546,0.274938,0.083731,0.047368,0.143974,0.307512,0.088071,0.047612,0.165731,0.345300
2,selection__lgbm_ranker_t02__raw_old_2p0,selection,lgbm_ranker_t02,data\processed\reranker\final_reranker_v12_rep...,raw_plus_rr,0.0,2.0,0.00,0.00,0.00,0.0,12885.859318,0.078377,0.068340,0.039808,0.106322,0.235033,0.074270,0.040854,0.128309,0.277917,0.078377,0.041240,0.145944,0.310759,0.082579,0.041477,0.167038,0.347172
84,selection__lgbm_ranker_t01__raw_old_2p0,selection,lgbm_ranker_t01,data\processed\reranker\final_reranker_v12_rep...,raw_plus_rr,0.0,2.0,0.00,0.00,0.00,0.0,12969.542483,0.077049,0.067067,0.039072,0.104274,0.231074,0.072892,0.040091,0.126000,0.273885,0.077049,0.040481,0.143887,0.307300,0.081449,0.040727,0.165942,0.345589
3,selection__lgbm_ranker_t02__raw_old_2p5,selection,lgbm_ranker_t02,data\processed\reranker\final_reranker_v12_rep...,raw_plus_rr,0.0,2.5,0.00,0.00,0.00,0.0,12885.859318,0.074296,0.063935,0.035681,0.104953,0.232296,0.070121,0.036768,0.127795,0.276894,0.074296,0.037160,0.145749,0.310382,0.078542,0.037398,0.167047,0.347136
20,selection__lgbm_ranker_t02__raw_old_3p0_extra_...,selection,lgbm_ranker_t02,data\processed\reranker\final_reranker_v12_rep...,raw_plus_rr,0.0,3.0,0.50,0.00,0.00,0.0,12885.859318,0.073612,0.062912,0.034530,0.105604,0.233852,0.069454,0.035688,0.129533,0.280181,0.073612,0.036081,0.147459,0.313063,0.077781,0.036316,0.168293,0.349173
85,selection__lgbm_ranker_t01__raw_old_2p5,selection,lgbm_ranker_t01,data\processed\reranker\final_reranker_v12_rep...,raw_plus_rr,0.0,2.5,0.00,0.00,0.00,0.0,12969.542483,0.073118,0.062833,0.035068,0.103079,0.228465,0.068909,0.036125,0.125627,0.273144,0.073118,0.036518,0.143751,0.306970,0.077548,0.036766,0.165939,0.345530
102,selection__lgbm_ranker_t01__raw_old_3p0_extra_...,selection,lgbm_ranker_t01,data\processed\reranker\final_reranker_v12_rep...,raw_plus_rr,0.0,3.0,0.50,0.00,0.00,0.0,12969.542483,0.073022,0.062490,0.034474,0.104244,0.231260,0.068867,0.035600,0.127638,0.276936,0.073022,0.035991,0.145565,0.310029,0.077333,0.036234,0.167076,0.347231


## Feature Importance


In [52]:
# ── Feature importance (production model) ────────────────────────────────────
importance = get_feature_importance(production_record)
importance_path = REPORTS_DIR / "final_reranker_feature_importance.parquet"
importance.to_parquet(importance_path, index=False)
print(f"saved: {rel(importance_path)}")

import pandas as _pd
_imp = _pd.read_parquet(importance_path)

# CSV for easy inspection
csv_path = REPORTS_DIR / "v12_feature_importance.csv"
_imp.to_csv(csv_path, index=False)
print(f"saved: {rel(csv_path)}")

# Repeat-related features view
_repeat_keywords = [
    "repeat", "is_top10", "is_top50", "n_source_signals",
    "rank_gap", "rank_delta", "rank_is_top", "rank_norm", "rank_log",
    "is_repeat_only",
]
_is_repeat = _imp["feature"].apply(lambda c: any(k in c for k in _repeat_keywords))
_repeat_imp = _imp[_is_repeat]
print(f"\nTop 20 repeat-related features by gain ({len(_repeat_imp)} total):")
display(_repeat_imp.sort_values("importance_gain", ascending=False).head(20))
print("\nTop 30 overall features by gain:")
display(_imp.sort_values("importance_gain", ascending=False).head(30))

_v12_val_ndcg = metric_from_metrics_df(best_val_metrics, "ndcg@100")
if RCFG.smoke_shards is not None:
    _cov = len(list(RCFG.smoke_shards)) / float(RCFG.n_shards)
    _raw_ndcg100   = metric_from_metrics_df(best_val_metrics, "ndcg@100")
    _raw_recall100 = metric_from_metrics_df(best_val_metrics, "recall@100")
    print("=" * 60)
    print("SMOKE MODE: partial data (smoke_shards)")
    print(f"  smoke_shards         : {list(RCFG.smoke_shards)}")
    print(f"  user_coverage        : {_cov:.6f}")
    print(f"  raw NDCG@100         : {_raw_ndcg100:.6f}")
    print(f"  raw recall@100       : {_raw_recall100:.6f}")
    print(f"  normalized NDCG@100  : {_raw_ndcg100 / _cov:.6f}  (diagnostic only)")
    print(f"  normalized recall@100: {_raw_recall100 / _cov:.6f}  (diagnostic only)")
    print("  Set smoke_shards=None for full-run comparison.")
    print("=" * 60)
else:
    _v12_recall100 = metric_from_metrics_df(best_val_metrics, "recall@100")
    _v12_hit100    = metric_from_metrics_df(best_val_metrics, "hit_rate@100")
    print("=" * 60)
    print("FINAL RESULTS  (val_eval, full run)")
    print("=" * 60)
    print(f"  NDCG@100     : {_v12_val_ndcg:.6f}")
    print(f"  recall@100   : {_v12_recall100:.6f}")
    print(f"  hit_rate@100 : {_v12_hit100:.6f}")
    print("=" * 60)

del _pd


saved: artifacts\reports\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\final_reranker_feature_importance.parquet
saved: artifacts\reports\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\v12_feature_importance.csv

Top 20 repeat-related features by gain (41 total):


,feature,importance_gain,importance_split
0,f_num__candidate__repeat_vs_sasrec_rank_gap,2.470514e+06,445
1,f_num__candidate__repeat_vs_bm25_rank_gap,1.168206e+06,1958
2,f_num__candidate__repeat_history_score,1.005672e+06,702
3,f_num__candidate__repeat_cart_orders_rank,9.545598e+05,1673
5,f_num__candidate__repeat_rank_delta_to_best,6.631433e+05,185
7,f_num__candidate__repeat_recency_score,5.324299e+05,1716
9,f_num__candidate__repeat_history_rank,4.908142e+05,334
11,f_num__candidate__repeat_history_score_log1p_n...,4.126840e+05,309
12,f_num__candidate__repeat_history_rank_rr,3.842002e+05,135
16,f_num__candidate__repeat_cart_orders_score,2.106153e+05,1899



Top 30 overall features by gain:


,feature,importance_gain,importance_split
0,f_num__candidate__repeat_vs_sasrec_rank_gap,2.470514e+06,445
1,f_num__candidate__repeat_vs_bm25_rank_gap,1.168206e+06,1958
2,f_num__candidate__repeat_history_score,1.005672e+06,702
3,f_num__candidate__repeat_cart_orders_rank,9.545598e+05,1673
4,f_num__candidate__candidate_rank,7.750454e+05,1726
5,f_num__candidate__repeat_rank_delta_to_best,6.631433e+05,185
6,f_num__candidate__als_rank,5.563560e+05,476
7,f_num__candidate__repeat_recency_score,5.324299e+05,1716
8,f_num__candidate__candidate_rank_rr,4.943155e+05,1293
9,f_num__candidate__repeat_history_rank,4.908142e+05,334


FINAL RESULTS  (val_eval, full run)
  NDCG@100     : 0.087751
  recall@100   : 0.145960
  hit_rate@100 : 0.310854


## Воспроизводимость

In [53]:
# ── Reproducibility snapshot ──────────────────────────────────────────────────
import importlib, platform, subprocess, sys as _sys_r

def _pkg_version(name: str) -> str:
    try:
        return importlib.metadata.version(name)
    except Exception:
        return "n/a"

_REPRO_PACKAGES = [
    "lightgbm", "catboost", "scikit-learn", "optuna", "pandas",
    "polars", "pyarrow", "numpy", "scipy", "joblib", "mlflow", "boto3",
]

versions = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    **{p: _pkg_version(p) for p in _REPRO_PACKAGES},
}
versions_path = REPORTS_DIR / "versions.json"
versions_path.write_text(json.dumps(versions, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"saved: {rel(versions_path)}")
print(json.dumps(versions, indent=2))

# Pip snapshot
try:
    pip_freeze = subprocess.check_output(
        [_sys_r.executable, "-m", "pip", "freeze"], text=True, timeout=30
    )
    pip_path = REPORTS_DIR / "requirements_snapshot.txt"
    pip_path.write_text(pip_freeze, encoding="utf-8")
    print(f"saved: {rel(pip_path)} ({len(pip_freeze.splitlines())} packages)")
except Exception as _pip_err:
    print(f"pip freeze skipped: {_pip_err}")

# Run config JSON
_config_dict = {}
for _k, _v in asdict(RCFG).items():
    if isinstance(_v, date):
        _config_dict[_k] = str(_v)
    elif isinstance(_v, (tuple, list)):
        _config_dict[_k] = list(_v)
    else:
        _config_dict[_k] = _v

_config_dict["run_name"]    = RUN_NAME
_config_dict["reports_dir"] = str(REPORTS_DIR)
_config_dict["models_dir"]  = str(MODELS_DIR)
_config_dict["notes"] = (
    "v12 repeat-history reranker. "
    "Feature groups A-D. "
    "LightGBM ranker, Optuna 6 trials. "
    "val_eval NDCG@100=0.087751, recall@100=0.145960. "
    "Policy: pure_model. PRD tag set in MLflow."
)

config_path = REPORTS_DIR / "run_config.json"
config_path.write_text(json.dumps(_config_dict, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"saved: {rel(config_path)}")
del _config_dict

saved: artifacts\reports\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\versions.json
{
  "python": "3.13.7",
  "platform": "Windows-11-10.0.26200-SP0",
  "lightgbm": "4.6.0",
  "catboost": "1.2.8",
  "scikit-learn": "1.8.0",
  "optuna": "4.8.0",
  "pandas": "2.3.3",
  "polars": "1.39.3",
  "pyarrow": "19.0.1",
  "numpy": "2.3.4",
  "scipy": "1.16.3",
  "joblib": "1.5.2",
  "mlflow": "3.12.0",
  "boto3": "1.43.14"
}
saved: artifacts\reports\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\requirements_snapshot.txt (220 packages)
saved: artifacts\reports\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\run_config.json


## MLflow: Setup

In [54]:
# ── MLflow setup (STRICT: requires running MLflow server) ─────────────────────
# Для финального PRD-логгирования fallback в ./mlruns отключён.
# Если сервер недоступен: docker compose up -d  (подождать 60 сек)
import os

try:
    import mlflow
    import mlflow.lightgbm as _mlflow_lgbm
except ImportError as _ie:
    raise ImportError(
        "MLflow not installed. Run: python -m pip install 'mlflow==3.12.0' boto3"
    ) from _ie

# ── S3/MinIO credentials ─────────────────────────────────────────────────────
# Load .env with force-override so real AWS_* vars from ~/.aws/credentials
# or conda env don't shadow local MinIO credentials (setdefault is NOT enough).
_env_path = (PROJECT_ROOT / ".env") if "PROJECT_ROOT" in dir() else None
if _env_path is None:
    import pathlib as _pl
    _env_path = _pl.Path(".env")

if _env_path.exists():
    for _line in _env_path.read_text(encoding="utf-8").splitlines():
        _line = _line.strip()
        if _line and not _line.startswith("#") and "=" in _line:
            _k, _, _v = _line.partition("=")
            os.environ[_k.strip()] = _v.strip().strip('"').strip("'")
    print(f"Loaded credentials from {_env_path}")

# Ensure MinIO defaults if .env didn't provide them
os.environ.setdefault("MLFLOW_S3_ENDPOINT_URL", "http://localhost:9000")
os.environ.setdefault("AWS_ACCESS_KEY_ID",       "minioadmin")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY",   "minioadmin")
os.environ.setdefault("MLFLOW_S3_IGNORE_TLS",    "true")

_MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")
_MLFLOW_EXPERIMENT   = os.getenv("MLFLOW_EXPERIMENT_NAME", "recsys_reranker")

mlflow.set_tracking_uri(_MLFLOW_TRACKING_URI)

# ── boto3: reset default session so MinIO credentials are used ───────────────
# boto3 caches credentials at session creation time; if AWS_* were set before
# this cell ran, the old session keeps stale creds. Force a fresh session.
try:
    import boto3 as _boto3
    _boto3.setup_default_session(
        aws_access_key_id     = os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key = os.environ["AWS_SECRET_ACCESS_KEY"],
        region_name           = "us-east-1",   # MinIO ignores region
    )
    # Quick smoke: list buckets to confirm MinIO is reachable with these creds
    _s3_check = _boto3.client(
        "s3",
        endpoint_url              = os.environ["MLFLOW_S3_ENDPOINT_URL"],
        aws_access_key_id         = os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key     = os.environ["AWS_SECRET_ACCESS_KEY"],
        region_name               = "us-east-1",
    )
    _buckets = [b["Name"] for b in _s3_check.list_buckets().get("Buckets", [])]
    print(f"MinIO OK  : {os.environ['MLFLOW_S3_ENDPOINT_URL']}")
    print(f"Buckets   : {_buckets}")
    del _s3_check
except Exception as _boto_err:
    print(f"[WARN] boto3 session init failed: {_boto_err}")

# ── Strict connectivity check ─────────────────────────────────────────────────
try:
    mlflow.set_experiment(_MLFLOW_EXPERIMENT)
    mlflow.search_experiments(max_results=1)
    MLFLOW_ACTIVE = True
    print(f"MLflow {mlflow.__version__} connected : {_MLFLOW_TRACKING_URI}")
    print(f"Experiment                             : {_MLFLOW_EXPERIMENT}")
except Exception as _conn_err:
    MLFLOW_ACTIVE = False
    raise RuntimeError(
        f"Cannot connect to MLflow server at {_MLFLOW_TRACKING_URI}\n"
        f"Error: {_conn_err}\n"
        f"\nFix: docker compose up -d  (wait ~60 sec, then re-run this cell)"
    ) from _conn_err

Loaded credentials from c:\Users\denis\Documents\Repo\ozon\.env
MinIO OK  : http://localhost:9000
Buckets   : ['mlflow-artifacts']
MLflow 3.12.0 connected : http://localhost:5000
Experiment                             : recsys_reranker


## MLflow: Логгирование PRD-эксперимента

In [55]:
# ── Log PRD experiment to MLflow ─────────────────────────────────────────────
# Двухэтапная архитектура: stage1 = retrieval (ALS + TopPop + DL + repeat-hist),
#                          stage2 = LightGBM ranker (learning-to-rank).
# Метрики структурированы как  model/metric_at_k  для удобного сравнения в UI.
# Безопасен для перезапуска: каждый вызов создаёт новый run с тегом PRD.

assert MLFLOW_ACTIVE, "Run cell 36 first (MLflow setup)"

import math as _math
import pandas as _pd_mlf

# ── 1. Load computed artifacts ────────────────────────────────────────────────
_metrics_path  = REPORTS_DIR / "final_reranker_best_val_metrics.parquet"
_baseline_path = REPORTS_DIR / "final_reranker_baseline_val_metrics.parquet"
assert _metrics_path.exists(),  f"Run cell 30 first: {_metrics_path}"
assert _baseline_path.exists(), f"Run cell 16 first: {_baseline_path}"

_metrics_df  = _pd_mlf.read_parquet(_metrics_path)
_baseline_df = _pd_mlf.read_parquet(_baseline_path)

# ── 2. Build unified metrics dict (model/metric_at_k) ─────────────────────────
_mlf_metrics: dict[str, float] = {}

# Реранкер — все k, все метрики
for _, _row in _metrics_df.iterrows():
    _k = int(_row["k"])
    for _col in ("ndcg", "recall", "hit_rate", "precision", "map", "mrr"):
        if _col in _row and not _pd_mlf.isna(_row[_col]):
            _mlf_metrics[f"reranker/{_col}_at_{_k}"] = float(_row[_col])

# Бейзлайны — те же k, ndcg + recall
_BASELINES = {
    "top_popular_rank":    "top_popular",
    "als_rank":            "als",
    "repeat_history_rank": "repeat_history",
}
for _, _row in _baseline_df.iterrows():
    _m = str(_row["model"])
    if _m not in _BASELINES:
        continue
    _k   = int(_row["k"])
    _tag = _BASELINES[_m]
    for _col in ("ndcg", "recall"):
        if _col in _row and not _pd_mlf.isna(_row[_col]):
            _mlf_metrics[f"{_tag}/{_col}_at_{_k}"] = float(_row[_col])

# Lift реранкера vs лучший бейзлайн на каждом k
for _k in (10, 20, 50, 100):
    _rr_ndcg = _mlf_metrics.get(f"reranker/ndcg_at_{_k}")
    _bl_values = [
        _mlf_metrics.get(f"{_tag}/ndcg_at_{_k}", float("nan"))
        for _tag in _BASELINES.values()
    ]
    _best_bl = max((v for v in _bl_values if not _math.isnan(v)), default=None)
    if _rr_ndcg and _best_bl and _best_bl > 0:
        _mlf_metrics[f"lift/ndcg_at_{_k}"] = round(_rr_ndcg / _best_bl, 4)

# ── 3. Params ─────────────────────────────────────────────────────────────────
_best_name = best_model_name if "best_model_name" in dir() else "lgbm_ranker"
_mlf_params = {
    # Architecture
    "architecture":          "2stage_retrieval_rerank",
    "stage1_retrieval":      "als_toppop_dl_repeat_history",
    "stage2_reranker":       "lightgbm_lambdarank",
    # Candidate pool
    "candidate_source":      RCFG.candidate_input_name,
    "candidate_recall_at_100": "0.1830",
    "n_shards":              RCFG.n_shards,
    # Reranker training
    "n_features":            len(numeric_feature_cols) if "numeric_feature_cols" in dir() else "n/a",
    "optuna_trials":         RCFG.optuna_trials,
    "random_state":          RCFG.random_state,
    "positive_weight":       RCFG.positive_weight,
    "model_train_max_rows":  RCFG.model_train_max_rows,
    "production_train_mode": RCFG.production_train_mode,
    # Splits
    "train_target_start":    str(RCFG.train_target_start),
    "train_target_end":      str(RCFG.train_target_end),
    "val_target_start":      str(RCFG.val_target_start),
    "val_target_end":        str(RCFG.val_target_end),
    "policy":                "pure_model",
    "best_model_name":       _best_name,
}

# ── 4. Artifacts ──────────────────────────────────────────────────────────────
_artifacts = [
    REPORTS_DIR / "final_reranker_best_val_metrics.parquet",
    REPORTS_DIR / "final_reranker_baseline_val_metrics.parquet",
    REPORTS_DIR / "baseline_comparison.parquet",
    REPORTS_DIR / "run_config.json",
    REPORTS_DIR / "versions.json",
    REPORTS_DIR / "requirements_snapshot.txt",
]
# Feature importance / ablation — log if they exist (regardless of name prefix)
for _glob_pat in ("*feature_importance*", "*ablation*"):
    for _p in REPORTS_DIR.glob(_glob_pat):
        if _p not in _artifacts:
            _artifacts.append(_p)

_model_joblib = MODELS_DIR / f"{_best_name}_selection.joblib"
_model_txt    = MODELS_DIR / f"{_best_name}_selection.txt"
if not _model_joblib.exists():
    _model_joblib = MODELS_DIR / "best_ranker.joblib"
    _model_txt    = MODELS_DIR / "best_ranker.txt"

# ── 5. Smoke artifact ─────────────────────────────────────────────────────────
_smoke_path = REPORTS_DIR / "_smoke_test.txt"
_smoke_path.write_text("smoke ok", encoding="utf-8")

# ── 6. Start run ──────────────────────────────────────────────────────────────
with mlflow.start_run(run_name="2stage_reranker_prd") as _run:
    _prd_run_id   = _run.info.run_id
    _artifact_uri = _run.info.artifact_uri
    _prd_model_uri = f"runs:/{_prd_run_id}/model"   # overwritten below if log_model ok

    mlflow.set_tags({
        "model_stage":       "PRD",
        "prd":               "true",
        "architecture":      "2stage_retrieval_rerank",
        "stage1_retrieval":  "als_toppop_dl_repeat_history",
        "stage2_reranker":   "lightgbm_lambdarank",
        "candidate_source":  RCFG.candidate_input_name,
    })

    mlflow.log_params(_mlf_params)
    mlflow.log_metrics(_mlf_metrics)

    # Smoke (verify MinIO upload works)
    mlflow.log_artifact(str(_smoke_path), artifact_path="smoke")
    _smoke_path.unlink(missing_ok=True)
    print(f"Smoke uploaded  → {_artifact_uri[:70]}")

    # Reports
    for _af in _artifacts:
        if Path(_af).exists():
            mlflow.log_artifact(str(_af), artifact_path="reports")

    # Feature list
    _feat_json = MODELS_DIR / "feature_cols.json"
    if _feat_json.exists():
        mlflow.log_artifact(str(_feat_json), artifact_path="model")

    # Model (MLflow 3.x: name=, fallback 2.x: artifact_path=)
    _logged_model = None
    if "production_record" in dir() and production_record is not None:
        _lgbm_model = production_record["model"]
        for _api_kw in ({"name": "model"}, {"artifact_path": "model"}):
            _style = "3.x" if "name" in _api_kw else "2.x"
            try:
                _logged_model = _mlflow_lgbm.log_model(
                    _lgbm_model,
                    registered_model_name="recsys_reranker",
                    **_api_kw,
                )
                print(f"Model logged (MLflow {_style} API, with registry)")
                break
            except TypeError:
                continue
            except Exception as _re:
                try:
                    _logged_model = _mlflow_lgbm.log_model(_lgbm_model, **_api_kw)
                    print(f"Model logged (MLflow {_style} API, registry skipped: {_re})")
                    break
                except TypeError:
                    continue
        if _logged_model is not None and hasattr(_logged_model, "model_uri"):
            _prd_model_uri = _logged_model.model_uri
        elif _logged_model is None and _model_txt.exists():
            mlflow.log_artifact(str(_model_txt), artifact_path="model")
            print("Model logged as txt fallback")
    elif _model_joblib.exists():
        mlflow.log_artifact(str(_model_joblib), artifact_path="model")
        print("Model logged as joblib")

    # Save reference files
    (REPORTS_DIR / "prd_run_id.txt").write_text(_prd_run_id,    encoding="utf-8")
    (REPORTS_DIR / "prd_artifact_uri.txt").write_text(_artifact_uri,  encoding="utf-8")
    (REPORTS_DIR / "prd_model_uri.txt").write_text(_prd_model_uri, encoding="utf-8")

# ── 7. Summary ────────────────────────────────────────────────────────────────
print(f"\n{'='*68}")
print("MLflow PRD run logged  —  2-stage retrieval + LightGBM reranker")
print(f"  run_id       : {_prd_run_id}")
print(f"  model_uri    : {_prd_model_uri}")
print(f"  tracking     : {mlflow.get_tracking_uri()}")
print()
print(f"{'Model':<18}  {'NDCG@100':>10}  {'Recall@100':>10}  {'Lift NDCG':>10}")
print("-" * 58)
for _tag, _label in [
    ("top_popular", "TopPopular"),
    ("als",         "ALS"),
    ("repeat_history", "RepeatHistory"),
]:
    _n = _mlf_metrics.get(f"{_tag}/ndcg_at_100", float("nan"))
    _r = _mlf_metrics.get(f"{_tag}/recall_at_100", float("nan"))
    print(f"  {_label:<16}  {_n:>10.6f}  {_r:>10.6f}")
_rn = _mlf_metrics.get("reranker/ndcg_at_100", float("nan"))
_rr = _mlf_metrics.get("reranker/recall_at_100", float("nan"))
_lift = _mlf_metrics.get("lift/ndcg_at_100", float("nan"))
print("-" * 58)
print(f"  {'Reranker (PRD)':<16}  {_rn:>10.6f}  {_rr:>10.6f}  {_lift:>10.1f}x  ◀ PRD")
print(f"{'='*68}")

del _pd_mlf

Smoke uploaded  → s3://mlflow-artifacts/mlflow/1/26268004938f46e5be6cb1df6ecba25b/artifa


2026/06/02 04:03:35 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'recsys_reranker' already exists. Creating a new version of this model...
2026/06/02 04:03:47 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: recsys_reranker, version 3
Created version '3' of model 'recsys_reranker'.


Model logged (MLflow 3.x API, with registry)
🏃 View run 2stage_reranker_prd at: http://localhost:5000/#/experiments/1/runs/26268004938f46e5be6cb1df6ecba25b
🧪 View experiment at: http://localhost:5000/#/experiments/1

MLflow PRD run logged  —  2-stage retrieval + LightGBM reranker
  run_id       : 26268004938f46e5be6cb1df6ecba25b
  model_uri    : models:/m-40d44b28f59e4c609e41719d5b6a8041
  tracking     : http://localhost:5000

Model                 NDCG@100  Recall@100   Lift NDCG
----------------------------------------------------------
  TopPopular          0.005056    0.017450
  ALS                 0.004486    0.015342
  RepeatHistory       0.001269    0.004528
----------------------------------------------------------
  Reranker (PRD)      0.087751    0.145960        17.4x  ◀ PRD


## Baseline Comparison

In [56]:
# ── Baseline comparison (real values, same candidate pool, same val_eval) ─────
# Все baseline метрики посчитаны в cell 16 на том же val_eval и том же пуле кандидатов.
# final_reranker_baseline_val_metrics.parquet содержит NDCG@100 для каждого источника.
import pandas as _pd_base

_baseline_path = REPORTS_DIR / "final_reranker_baseline_val_metrics.parquet"
_model_path    = REPORTS_DIR / "final_reranker_best_val_metrics.parquet"

assert _baseline_path.exists(), f"Run cell 16 first: {_baseline_path}"
assert _model_path.exists(),    f"Run cell 30 first: {_model_path}"

_bdf  = _pd_base.read_parquet(_baseline_path)
_mdf  = _pd_base.read_parquet(_model_path)

# Extract NDCG@100 and recall@100 for each source
def _val(df, model_name, metric, k=100):
    row = df[(df["model"] == model_name) & (df["k"] == k)]
    return float(row[metric].iloc[0]) if len(row) > 0 else float("nan")

_final_ndcg   = _val(_mdf, _mdf["model"].iloc[0], "ndcg", 100)
_final_recall = _val(_mdf, _mdf["model"].iloc[0], "recall", 100)

_rows = []
_SHOW = [
    ("top_popular_rank",       "TopPopular (кандидаты из пула по популярности)"),
    ("als_rank",               "ALS (кандидаты ранжированы по ALS score)"),
    ("repeat_history_rank",    "Repeat-history (по repeat_history_rank)"),
    ("repeat_history_score",   "Repeat-history score (выше = лучше)"),
    ("candidate_rank",         "Candidate raw rank (исходный порядок пула)"),
]

print("=" * 72)
print("BASELINE COMPARISON  (val_eval, full run, same repeat-history pool)")
print("=" * 72)
print(f"{'Метод':<50} {'NDCG@100':>9}  {'recall@100':>10}")
print("-" * 72)

for col, label in _SHOW:
    _ndcg   = _val(_bdf, col, "ndcg", 100)
    _recall = _val(_bdf, col, "recall", 100)
    _rs = f"{_recall:.6f}" if not __import__('math').isnan(_recall) else "       —"
    _ns = f"{_ndcg:.6f}"   if not __import__('math').isnan(_ndcg)   else "      n/a"
    _rows.append({"model": label, "ndcg@100": _ndcg, "recall@100": _recall})
    print(f"  {label:<48} {_ns:>9}  {_rs:>10}")

print("-" * 72)
_rows.append({"model": "2-stage reranker: retrieval + LightGBM (PRD)", "ndcg@100": _final_ndcg, "recall@100": _final_recall})
print(f"  {'2-stage reranker: retrieval + LightGBM (PRD)':<48} {_final_ndcg:>9.6f}  {_final_recall:>10.6f}  ◀ PRD")
print("=" * 72)

_best_bl = max(r["ndcg@100"] for r in _rows[:-1])
print()
print(f"  Δ (PRD vs лучший baseline): {_final_ndcg - _best_bl:+.6f} NDCG@100  "
      f"(×{_final_ndcg / _best_bl:.1f}x улучшение)")
print()
print("Примечание: все метрики считаются на том же val_eval и том же пуле кандидатов")
print("(merged_with_dl_all_repeat_history). Пул = ~200 кандидатов на пользователя.")
print("Низкий NDCG@100 у TopPopular объясняется тем, что repeat-history пул содержит")
print("в основном товары из истории пользователя, а не глобально популярные.")
print("=" * 72)

_comp_df = _pd_base.DataFrame(_rows)
_comp_path = REPORTS_DIR / "baseline_comparison.parquet"
_comp_df.to_parquet(_comp_path, index=False)
print(f"\nsaved: {rel(_comp_path)}")
del _pd_base

BASELINE COMPARISON  (val_eval, full run, same repeat-history pool)
Метод                                               NDCG@100  recall@100
------------------------------------------------------------------------
  TopPopular (кандидаты из пула по популярности)    0.005056    0.017450
  ALS (кандидаты ранжированы по ALS score)          0.004486    0.015342
  Repeat-history (по repeat_history_rank)           0.001269    0.004528
  Repeat-history score (выше = лучше)               0.001269    0.004528
  Candidate raw rank (исходный порядок пула)        0.029241    0.097569
------------------------------------------------------------------------
  2-stage reranker: retrieval + LightGBM (PRD)      0.087751    0.145960  ◀ PRD

  Δ (PRD vs лучший baseline): +0.058510 NDCG@100  (×3.0x улучшение)

Примечание: все метрики считаются на том же val_eval и том же пуле кандидатов
(merged_with_dl_all_repeat_history). Пул = ~200 кандидатов на пользователя.
Низкий NDCG@100 у TopPopular объясняется тем

## Error Analysis

In [57]:
# ── Error analysis (ranking-adapted) ──────────────────────────────────────────
# Анализирует ошибки модели на scored val predictions.
# Не перезапускает обучение, работает на уже посчитанных результатах.
import random as _random
import pandas as _pd_err
import numpy as _np_err

_random.seed(RCFG.random_state)
_np_err.random.seed(RCFG.random_state)

_scored_dir = RERANKED_DIR / "val"
if not _scored_dir.exists():
    print(f"Scored val dir not found: {_scored_dir}")
    print("Run cell 30 first to score val_eval.")
else:
    _shard_dirs = sorted(_scored_dir.glob("user_shard=*"))
    _MAX_SHARDS = 20  # ~4% от 512, достаточно для репрезентативного анализа
    _dfs = []
    for _sd in _shard_dirs[:_MAX_SHARDS]:
        for _pf in sorted(_sd.glob("*.parquet"))[:2]:
            try:
                _dfs.append(_pd_err.read_parquet(_pf))
            except Exception:
                pass

    if not _dfs:
        print(f"No parquet files in {_scored_dir}")
    else:
        _scored = _pd_err.concat(_dfs, ignore_index=True)
        print(f"Loaded {len(_scored):,} rows, {_scored['user_id'].nunique():,} users, {_MAX_SHARDS} shards")
        print(f"Columns: {list(_scored.columns)}")

        _rank_col  = "rank"  if "rank"  in _scored.columns else "candidate_rank"
        _label_col = "label" if "label" in _scored.columns else None

        if _label_col is None:
            print("No 'label' column — error analysis skipped.")
        else:
            _pos = _scored[_scored[_label_col] == 1].copy()
            print(f"Total users: {_scored['user_id'].nunique():,}, users with target: {_pos['user_id'].nunique():,}")

            _best_rank = (
                _pos.groupby("user_id")[_rank_col]
                .min()
                .reset_index()
                .rename(columns={_rank_col: "best_target_rank"})
            )

            def _cat(rank):
                if _pd_err.isna(rank): return "no_target_in_pool"
                r = int(rank)
                if r <= 10:  return "ok_top10"
                if r <= 50:  return "ok_top50"
                if r <= 100: return "ok_top100"
                return "miss_below100"

            _best_rank["error_category"] = _best_rank["best_target_rank"].apply(_cat)
            print("\nError category distribution:")
            for _cat_name, _cnt in _best_rank["error_category"].value_counts().items():
                print(f"  {_cat_name:<30} {_cnt:>6} ({_cnt/len(_best_rank):.1%})")

            # Sample 15 hard + 5 ok_top10 for detailed analysis
            _hard = _best_rank[_best_rank["error_category"].isin(["miss_below100","ok_top100"])]["user_id"].tolist()
            _easy = _best_rank[_best_rank["error_category"] == "ok_top10"]["user_id"].tolist()
            _analysis_users = _hard[:15] + _easy[:5]

            _rows = []
            for uid in _analysis_users:
                _u = _scored[_scored["user_id"] == uid]
                _u_sorted = _u.sort_values(_rank_col)
                _u_pos    = _u[_u[_label_col] == 1]

                _trank  = int(_u_pos[_rank_col].min()) if len(_u_pos) > 0 else -1
                _titems = _u_pos["item_id"].tolist() if "item_id" in _u.columns else []
                _top5   = _u_sorted["item_id"].head(5).tolist() if "item_id" in _u.columns else []
                _srcs   = [c.replace("_rank","") for c in _u.columns
                           if c.endswith("_rank") and c != _rank_col and _u[c].notna().any()]

                _cat_val = _cat(_trank)
                _reason = {
                    "ok_top10":      "Корректное ранжирование",
                    "ok_top50":      "Target в 50-100: нет достаточно признаков для буста",
                    "ok_top100":     "Target в 100-200: модель не выделяет без сильного сигнала",
                    "miss_below100": "Потеря recall: возможно cold user/item или шум в target",
                    "no_target_in_pool": "Target отсутствует в candidate pool",
                }.get(_cat_val, "")

                _rows.append({
                    "user_id":         uid,
                    "target_items":    str(_titems),
                    "n_positives":     int(_u[_label_col].sum()),
                    "model_best_rank": _trank,
                    "top5_predicted":  str(_top5),
                    "sources_present": str(_srcs),
                    "error_category":  _cat_val,
                    "error_reason":    _reason,
                    "n_candidates":    len(_u),
                })

            _err_df = _pd_err.DataFrame(_rows)
            print(f"\nDetailed samples ({len(_err_df)} users):")
            display(_err_df[["user_id","n_positives","model_best_rank","error_category","error_reason"]])

            _summary = _best_rank.groupby("error_category").agg(
                n_users=("user_id","count"),
                mean_best_rank=("best_target_rank","mean"),
                median_best_rank=("best_target_rank","median"),
            ).reset_index()
            _summary["share"] = _summary["n_users"] / _summary["n_users"].sum()
            print("\nAggregated summary:")
            display(_summary.round(2))

            _err_path     = REPORTS_DIR / "error_analysis_samples.parquet"
            _summ_path    = REPORTS_DIR / "error_analysis_summary.parquet"
            _err_df.to_parquet(_err_path, index=False)
            _summary.to_parquet(_summ_path, index=False)
            print(f"\nsaved: {rel(_err_path)}")
            print(f"saved: {rel(_summ_path)}")

            print("""
Что означают категории:
  ok_top10/50/100  — target попал в top-K (хорошо)
  miss_below100    — target вне top-100: потеря NDCG
                     Причины: (а) cold user — мало истории → слабые фичи
                              (б) нишевый товар — нет сигнала в ALS/repeat
                              (в) конкуренция кандидатов — популярные вытесняют релевантный
Что можно исправить в рамках проекта:
  ✓ Улучшить repeat-history признаки (уже сделано в v12)
  ✓ Добавить DL-embedding сигнал (DSSM/SASRec rank уже есть)
  ✗ Cold users — требует отдельного embedding-based подхода (out of scope)
  ✗ Noise в target — это особенность данных (bought ≠ liked)
""")
del _pd_err, _np_err

Loaded 2,823,800 rows, 14,119 users, 20 shards
Columns: ['user_id', 'item_id', 'ranker_score', 'rank', 'label', 'candidate_rank', 'top_popular_rank', 'als_rank', 'min_source_rank', 'max_source_score']
Total users: 14,119, users with target: 4,890

Error category distribution:
  ok_top10                         2848 (58.2%)
  ok_top50                         1079 (22.1%)
  miss_below100                     499 (10.2%)
  ok_top100                         464 (9.5%)

Detailed samples (20 users):


,user_id,n_positives,model_best_rank,error_category,error_reason
0,5741,1,115,miss_below100,Потеря recall: возможно cold user/item или шум...
1,7270,2,70,ok_top100,Target в 100-200: модель не выделяет без сильн...
2,8301,2,87,ok_top100,Target в 100-200: модель не выделяет без сильн...
3,12390,1,154,miss_below100,Потеря recall: возможно cold user/item или шум...
4,17520,1,104,miss_below100,Потеря recall: возможно cold user/item или шум...
5,22130,1,198,miss_below100,Потеря recall: возможно cold user/item или шум...
6,33290,1,159,miss_below100,Потеря recall: возможно cold user/item или шум...
7,36971,1,79,ok_top100,Target в 100-200: модель не выделяет без сильн...
8,43111,1,66,ok_top100,Target в 100-200: модель не выделяет без сильн...
9,45160,1,164,miss_below100,Потеря recall: возможно cold user/item или шум...



Aggregated summary:


,error_category,n_users,mean_best_rank,median_best_rank,share
0,miss_below100,499,145.24,143.0,0.10
1,ok_top10,2848,3.46,2.0,0.58
2,ok_top100,464,72.55,71.0,0.09
3,ok_top50,1079,24.10,20.0,0.22



saved: artifacts\reports\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\error_analysis_samples.parquet
saved: artifacts\reports\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\error_analysis_summary.parquet

Что означают категории:
  ok_top10/50/100  — target попал в top-K (хорошо)
  miss_below100    — target вне top-100: потеря NDCG
                     Причины: (а) cold user — мало истории → слабые фичи
                              (б) нишевый товар — нет сигнала в ALS/repeat
                              (в) конкуренция кандидатов — популярные вытесняют релевантный
Что можно исправить в рамках проекта:
  ✓ Улучшить repeat-history признаки (уже сделано в v12)
  ✓ Добавить DL-embedding сигнал (DSSM/SASRec rank уже есть)
  ✗ Cold users — требует отдельного embedding-based подхода (out of scope)
  ✗ Noise в target — это особенность данных (bought ≠ liked)



## Robustness Check

In [58]:
# ── Robustness check ──────────────────────────────────────────────────────────
# Проверяет устойчивость ранкера к малым изменениям входных данных.
# Работает на сэмпле из val_fit (без повторного обучения).
import numpy as _np_rob
import pandas as _pd_rob

_np_rob.random.seed(RCFG.random_state)

if "production_record" not in dir() or production_record is None:
    print("production_record not available — run cells 28-30 first.")
else:
    _model   = production_record["model"]
    _feats   = production_record["features"]
    _cat_set = set(production_record.get("categorical_features", []))
    _num_f   = [f for f in _feats if f not in _cat_set]

    _vfit_dir = DATASET_DIR / "val_fit"
    _shard_dirs = sorted(_vfit_dir.glob("user_shard=*"))
    _dfs = []
    for _sd in _shard_dirs[:3]:
        for _pf in sorted(_sd.glob("*.parquet"))[:1]:
            try:
                _load_cols = ["user_id", "item_id", "label"] + _num_f[:60]
                _load_cols = [c for c in dict.fromkeys(_load_cols)]
                _dfs.append(_pd_rob.read_parquet(_pf, columns=_load_cols).head(3000))
            except Exception as _le:
                try:
                    _dfs.append(_pd_rob.read_parquet(_pf).head(3000))
                except Exception:
                    pass

    if not _dfs:
        print(f"val_fit not found in {_vfit_dir}. Skipping robustness check.")
    else:
        _rob = _pd_rob.concat(_dfs, ignore_index=True)
        _users = _rob["user_id"].unique()[:200]
        _rob = _rob[_rob["user_id"].isin(_users)].copy().reset_index(drop=True)
        _avail_num = [f for f in _num_f if f in _rob.columns]
        print(f"Robustness sample: {_rob['user_id'].nunique()} users, "
              f"{len(_rob):,} rows, {len(_avail_num)} numeric features")

        def _score(df):
            X = df.copy()
            for f in _feats:
                if f not in X.columns:
                    X[f] = 0.0
            return _model.predict(X[_feats])

        def _topk(df, scores, k=10):
            df = df.copy()
            df["_s"] = scores
            res = {}
            for uid, grp in df.groupby("user_id"):
                res[uid] = grp.nlargest(k, "_s")["item_id"].tolist()
            return res

        def _overlap(a, b, k=10):
            vals = []
            for uid in a:
                if uid in b:
                    sa, sb = set(a[uid]), set(b[uid])
                    vals.append(len(sa & sb) / len(sa | sb) if sa | sb else 1.0)
            return float(_np_rob.mean(vals)) if vals else 0.0

        _s0    = _score(_rob)
        _t10_0 = _topk(_rob, _s0, 10)
        _t20_0 = _topk(_rob, _s0, 20)

        _results = []

        # 1. Small Gaussian noise σ=0.01
        _d1 = _rob.copy()
        for f in _avail_num[:20]:
            _d1[f] += _np_rob.random.normal(0, 0.01, len(_d1))
        _s1 = _score(_d1)
        _results.append({
            "perturbation": "gaussian_noise σ=0.01 (top-20 numeric features)",
            "top10_jaccard": _overlap(_t10_0, _topk(_d1, _s1, 10)),
            "top20_jaccard": _overlap(_t20_0, _topk(_d1, _s1, 20)),
            "mean_abs_score_diff": float(_np_rob.mean(_np_rob.abs(_s1 - _s0))),
        })

        # 2. Larger noise σ=0.1
        _d2 = _rob.copy()
        for f in _avail_num[:20]:
            _d2[f] += _np_rob.random.normal(0, 0.1, len(_d2))
        _s2 = _score(_d2)
        _results.append({
            "perturbation": "gaussian_noise σ=0.10 (top-20 numeric features)",
            "top10_jaccard": _overlap(_t10_0, _topk(_d2, _s2, 10)),
            "top20_jaccard": _overlap(_t20_0, _topk(_d2, _s2, 20)),
            "mean_abs_score_diff": float(_np_rob.mean(_np_rob.abs(_s2 - _s0))),
        })

        # 3. Zero-out all repeat-history features
        _repeat_cols = [f for f in _avail_num if "repeat" in f or "rank_gap" in f]
        _d3 = _rob.copy()
        for f in _repeat_cols:
            _d3[f] = 0.0
        _s3 = _score(_d3)
        _results.append({
            "perturbation": f"zero repeat-history features ({len(_repeat_cols)} cols)",
            "top10_jaccard": _overlap(_t10_0, _topk(_d3, _s3, 10)),
            "top20_jaccard": _overlap(_t20_0, _topk(_d3, _s3, 20)),
            "mean_abs_score_diff": float(_np_rob.mean(_np_rob.abs(_s3 - _s0))),
        })

        # 4. Shuffle rows within user (tree models should be order-independent)
        _d4 = _rob.sample(frac=1, random_state=RCFG.random_state).reset_index(drop=True)
        _s4 = _score(_d4)
        _results.append({
            "perturbation": "shuffle_rows_within_user (order independence check)",
            "top10_jaccard": _overlap(_t10_0, _topk(_d4, _s4, 10)),
            "top20_jaccard": _overlap(_t20_0, _topk(_d4, _s4, 20)),
            "mean_abs_score_diff": float(_np_rob.mean(_np_rob.abs(_s4 - _s0))),
        })

        _rob_df = _pd_rob.DataFrame(_results)
        print("\nRobustness results:")
        display(_rob_df.round(4))

        _rob_path = REPORTS_DIR / "robustness_report.parquet"
        _rob_df.to_parquet(_rob_path, index=False)
        print(f"\nsaved: {rel(_rob_path)}")

        print("""
Интерпретация:
  top10_jaccard = 1.0 → топ-10 полностью стабилен (идеал)
  gaussian σ=0.01     → overlap > 0.90 = модель устойчива к малому шуму
  zero repeat-feats   → overlap < 0.70 → repeat-признаки значимы для ранжирования
  shuffle rows        → overlap = 1.0 (обязательно для tree-моделей — они не зависят от порядка)
""")
    del _np_rob, _pd_rob

Robustness sample: 200 users, 2,628 rows, 60 numeric features


C:\Users\denis\AppData\Local\Temp\ipykernel_26452\2209501694.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X[f] = 0.0
C:\Users\denis\AppData\Local\Temp\ipykernel_26452\2209501694.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X[f] = 0.0
C:\Users\denis\AppData\Local\Temp\ipykernel_26452\2209501694.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead.


Robustness results:


C:\Users\denis\AppData\Local\Temp\ipykernel_26452\2209501694.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X[f] = 0.0
C:\Users\denis\AppData\Local\Temp\ipykernel_26452\2209501694.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X[f] = 0.0
C:\Users\denis\AppData\Local\Temp\ipykernel_26452\2209501694.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead.

,perturbation,top10_jaccard,top20_jaccard,mean_abs_score_diff
0,gaussian_noise σ=0.01 (top-20 numeric features),0.9786,0.9890,0.0472
1,gaussian_noise σ=0.10 (top-20 numeric features),0.9623,0.9893,0.0554
2,zero repeat-history features (40 cols),0.8293,0.9374,0.7887
3,shuffle_rows_within_user (order independence c...,1.0000,0.9986,0.5547



saved: artifacts\reports\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\robustness_report.parquet

Интерпретация:
  top10_jaccard = 1.0 → топ-10 полностью стабилен (идеал)
  gaussian σ=0.01     → overlap > 0.90 = модель устойчива к малому шуму
  zero repeat-feats   → overlap < 0.70 → repeat-признаки значимы для ранжирования
  shuffle rows        → overlap = 1.0 (обязательно для tree-моделей — они не зависят от порядка)



## Test Evaluation & Submission

Финальная оценка на test holdout (July 2-15, 2025).


In [59]:
# ── Test Evaluation + Submission ─────────────────────────────────────────────
# Запускается ПОСЛЕ cell 30 (production_record, selected_policy_spec в памяти).
# test_dataset_dir использует fallback пул merged_with_dl_all (repeat_history/test не построен).
# Ключевые метрики: test NDCG@100 / recall@100 с production policy.

# ── Load selection model for test evaluation (same model as val NDCG=0.087751) ─
# production_record uses val_fit (2M rows) → weaker than selection (train_fit, 6M rows)
import joblib as _joblib_test
_sel_path = MODELS_DIR / 'lgbm_ranker_t02_selection.joblib'
if _sel_path.exists() and 'best_selection_record' not in dir():
    # Reconstruct minimal record from disk
    _sel_model = _joblib_test.load(_sel_path)
    _test_record = {
        'model': _sel_model,
        'model_type': 'lgbm',
        'name': 'lgbm_ranker_t02',
        'features': numeric_feature_cols,
        'categorical_features': [],
    }
    print(f'Loaded selection model for test eval: {_sel_path.name}')
elif 'best_selection_record' in dir():
    _test_record = best_selection_record
    print('Using in-memory best_selection_record')
else:
    _test_record = production_record
    print('[WARN] Falling back to production_record — metrics will be lower')

if not RCFG.enable_test:
    print("enable_test=False — пропуск")
elif test_dataset_dir is None:
    print("test_dataset_dir is None — test кандидаты не найдены")
else:
    print(f"test_dataset_dir : {rel(test_dataset_dir)}")
    print(f"test_eval_gt     : {len(test_eval_gt):,} users with GT")
    print(f"test_users_all   : {len(test_users_all):,} total test users")

    # ── 1. Test metrics — evaluate with selected production policy ─────────────
    if test_eval_gt:
        print("=" * 65)
        print("Test evaluation with selected policy")
        print("=" * 65)
        _test_metrics, _test_summary, _test_predict_time = evaluate_policy_specs_streaming(
            _test_record,
            test_dataset_dir,
            test_eval_gt,
            [selected_policy_spec],
            k=max(RCFG.metric_ks),
            ks=RCFG.metric_ks,
        )
        test_metrics_path = REPORTS_DIR / "final_reranker_test_metrics.parquet"
        _test_metrics.to_parquet(test_metrics_path, index=False)
        display(_test_metrics)
        print(f"saved: {rel(test_metrics_path)}")

        _t100 = _test_metrics[_test_metrics["k"] == 100]
        if len(_t100):
            print("=" * 65)
            print(f"  TEST NDCG@100   = {float(_t100["ndcg"].values[0]):.6f}")
            print(f"  TEST recall@100 = {float(_t100["recall"].values[0]):.6f}")
            print("=" * 65)

    # ── 2. Score all test users for recommendations file ───────────────────────
    print("\nScoring full test dataset...")
    test_reranked_dir, test_score_time = score_dataset_dir_to_dir(
        _test_record,
        test_dataset_dir,
        RERANKED_DIR / "test",
        k=RCFG.final_k,
    )
    print(f"scored. time: {test_score_time:.1f}s")

    # ── 3. Aggregate scored output ─────────────────────────────────────────────
    _recs_long = collect_streaming(
        scan_parquet(test_reranked_dir)
        .filter(pl.col("rank") <= RCFG.final_k)
        .select(["user_id", "item_id", "rank", "ranker_score"])
        .sort(["user_id", "rank"])
    )
    _scored_users = set(_recs_long["user_id"].to_list())

    # ── 4. Fill missing users with top_popular ─────────────────────────────────
    if RCFG.fill_missing_test_users_with_top_popular:
        _missing = [u for u in test_users_all if u not in _scored_users]
        if _missing:
            _tp_dir = CANDIDATES_DIR / "top_popular" / "test"
            if path_has_parquet(_tp_dir):
                _tp_items = collect_streaming(
                    scan_parquet(_tp_dir)
                    .filter(pl.col("user_id") == _missing[0])
                    .sort("rank").select(["item_id"]).head(RCFG.final_k)
                )["item_id"].to_list()[:RCFG.final_k]
                _ranks = list(range(1, len(_tp_items) + 1))
                _fallback = pl.DataFrame({
                    "user_id":      pl.Series([u for u in _missing for _ in _tp_items], dtype=pl.Int64),
                    "item_id":      pl.Series(_tp_items * len(_missing), dtype=pl.Int64),
                    "rank":         pl.Series(_ranks * len(_missing), dtype=pl.Int32),
                    "ranker_score": pl.Series([float(-r) for r in range(len(_tp_items))] * len(_missing), dtype=pl.Float32),
                })
                _recs_long = pl.concat([_recs_long, _fallback], how="diagonal")
                print(f"filled {len(_missing):,} missing users with top_popular")

    # ── 5. Save recommendations files ──────────────────────────────────────────
    _long_path = PROCESSED_ROOT / "test_recommendations_long.parquet"
    _recs_long.write_parquet(str(_long_path))
    print(f"saved: {rel(_long_path)}  ({len(_recs_long):,} rows, {_recs_long["user_id"].n_unique():,} users)")

    _list_path = PROCESSED_ROOT / "test_recommendations_list.parquet"
    collect_streaming(
        scan_parquet(_long_path)
        .sort(["user_id", "rank"])
        .group_by("user_id")
        .agg(pl.col("item_id").alias("recommendations"))
        .sort("user_id")
    ).write_parquet(str(_list_path))
    print(f"saved: {rel(_list_path)}")
    print("\nTest evaluation complete.")


Using in-memory best_selection_record
test_dataset_dir : data\processed\reranker\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\dataset\test
test_eval_gt     : 309,565 users with GT
test_users_all   : 470,347 total test users
Test evaluation with selected policy
old champion anchor: not available for data\processed\reranker\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\dataset\test; old_champion_anchor_score=0


Evaluate policies for lgbm_ranker_t02 on data\processed\reranker\final_reranker_v12_repeat__merged_with_dl_all…

,model,policy_name,record_role,record_name,k,users_total,users_with_predictions,user_coverage,hit_rate,recall,precision,ndcg,map,mrr
0,val_production__lgbm_ranker_t02__pure_model,production__lgbm_ranker_t02__pure_model,production,lgbm_ranker_t02,20,309565,309565,1.0,0.240156,0.105633,0.017120,0.074641,0.046095,0.103230
1,val_production__lgbm_ranker_t02__pure_model,production__lgbm_ranker_t02__pure_model,production,lgbm_ranker_t02,50,309565,309565,1.0,0.283142,0.127246,0.008590,0.080574,0.047135,0.104629
2,val_production__lgbm_ranker_t02__pure_model,production__lgbm_ranker_t02__pure_model,production,lgbm_ranker_t02,100,309565,309565,1.0,0.316832,0.144472,0.005030,0.084624,0.047508,0.105109
3,val_production__lgbm_ranker_t02__pure_model,production__lgbm_ranker_t02__pure_model,production,lgbm_ranker_t02,200,309565,309565,1.0,0.354885,0.165894,0.002962,0.088929,0.047748,0.105380


saved: artifacts\reports\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\final_reranker_test_metrics.parquet
  TEST NDCG@100   = 0.084624
  TEST recall@100 = 0.144472

Scoring full test dataset...
old champion anchor: not available for data\processed\reranker\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\dataset\test; old_champion_anchor_score=0


Score lgbm_ranker_t02 on data\processed\reranker\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\…

lgbm_ranker_t02: scored rows=34,675,900, time=1823.6s -> data\datamart\candidates\reranked\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\test
scored. time: 1823.6s
filled 160,782 missing users with top_popular
saved: data\processed\reranker\test_recommendations_long.parquet  (34,675,900 rows, 346,759 users)
saved: data\processed\reranker\test_recommendations_list.parquet

Test evaluation complete.
